# COMP30770 GROUP PROJECT - DATA PREPARATION 

## Setup & Configuration
- Import libraries (pandas, numpy, datetime, json, re, etc.)
- Set display options for better readability
- Define file path to dataset
- Log system specs (CPU, RAM, Python version)

In [1]:
import pandas as pd
import numpy as np
import json
import datetime
import re
import time
import os
import sys
import platform
import psutil

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)   
pd.set_option('display.width', 1000)     
pd.set_option('display.float_format', '{:.2f}'.format)  
pd.set_option('display.max_colwidth', 50) 

data_path = "TMDB_all_movies.csv"

if os.path.exists(data_path):
    print("Dataset found at:", data_path)
    file_size_gb = os.path.getsize(data_path) / (1024**3)
    print(f"File size: {file_size_gb:.2f} GB")
else:
    print("Warning: Dataset not found at", data_path)
    print("Please update the data_path variable with the correct path")

# Log System Specifications
print("\n--- SYSTEM SPECIFICATIONS ---")
# Python and Package Versions
print("Python Version:", platform.python_version())
print("Pandas Version:", pd.__version__)
print("Numpy Version:", np.__version__)

# System Information
print("Operating System:", platform.system(), platform.release())
print("Processor:", platform.processor())

# Memory Information
memory = psutil.virtual_memory()
total_ram_gb = memory.total / (1024**3)
available_ram_gb = memory.available / (1024**3)
print(f"Total RAM: {total_ram_gb:.2f} GB")
print(f"Available RAM: {available_ram_gb:.2f} GB")

# CPU Information
print("CPU Cores (Physical):", psutil.cpu_count(logical=False))
print("CPU Cores (Logical):", psutil.cpu_count(logical=True))



Dataset found at: TMDB_all_movies.csv
File size: 0.66 GB

--- SYSTEM SPECIFICATIONS ---
Python Version: 3.12.1
Pandas Version: 2.2.1
Numpy Version: 1.26.4
Operating System: Windows 11
Processor: Intel64 Family 6 Model 154 Stepping 3, GenuineIntel
Total RAM: 15.71 GB
Available RAM: 2.61 GB
CPU Cores (Physical): 12
CPU Cores (Logical): 16


## Step 0: Load Data & Volume Validation
- Load the CSV into a pandas DataFrame
- Display initial shape and memory usage
- Show first/last few rows
- List all columns

In [3]:
# Load Data and Volume Validation
#  start time for performance logging
load_start_time = time.time()

print("Loading dataset...")

try:
    
    df = pd.read_csv(data_path)
    
    load_time = time.time() - load_start_time
    print(f"Dataset loaded successfully in {load_time:.2f} seconds")
    
    # Display initial shape
    print(f"Initial shape: {df.shape[0]} rows, {df.shape[1]} columns")
    
    # Display memory usage
    memory_usage_mb = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"Memory usage: {memory_usage_mb:.2f} MB")
    
    print("\n--- First 5 rows ---")
    display(df.head(5))
    
    print("\n--- Last 5 rows ---")
    display(df.tail(5))
    
    print("\n--- All columns in dataset ---")
    for i, col in enumerate(df.columns, 1):
        print(f"{i:2d}. {col}")
    
except FileNotFoundError:
    print(f"Error: File not found at {data_path}")
    print("Please check the file path and try again")
except Exception as e:
    print(f"Error loading dataset: {str(e)}")

Loading dataset...
Dataset loaded successfully in 22.24 seconds
Initial shape: 1157987 rows, 28 columns
Memory usage: 1624.42 MB

--- First 5 rows ---


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,original_language,original_title,overview,popularity,tagline,genres,production_companies,production_countries,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path
0,2,Ariel,7.10,367.00,Released,1988-10-21,0.00,73.00,0.00,tt0094675,fi,Ariel,A Finnish man goes to the city to find a job a...,4.14,NaN,"Comedy, Drama, Romance, Crime",Villealfa Filmproductions,Finland,suomi,"Tarja Keinänen, Marja Packalén, Kari Helaseppä...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Aki Kaurismäki,NaN,7.40,9607.00,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg
1,3,Shadows in Paradise,7.30,430.00,Released,1986-10-17,0.00,74.00,0.00,tt0092149,fi,Varjoja paratiisissa,"Nikander, a rubbish collector and would-be ent...",5.48,NaN,"Comedy, Drama, Romance",Villealfa Filmproductions,Finland,"svenska, suomi, English","Haije Alanoja, Mari Rantasila, Matti Pellonpää...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Mika Kaurismäki,NaN,7.40,8454.00,/nj01hspawPof0mJmlgfjuLyJuRN.jpg
2,5,Four Rooms,5.90,2780.00,Released,1995-12-09,4257354.00,98.00,4000000.00,tt0113101,en,Four Rooms,It's Ted the Bellhop's first night on the job....,3.02,Twelve outrageous guests. Four scandalous requ...,Comedy,"Miramax, A Band Apart",United States of America,English,"Lili Taylor, Kimberly Blair, Quinn Hellerman, ...","Alexandre Rockwell, Allison Anders, Robert Rod...","Rodrigo García, Phil Parmet, Andrzej Sekula, G...","Allison Anders, Alexandre Rockwell, Quentin Ta...","Alexandre Rockwell, Quentin Tarantino, Lawrenc...",Combustible Edison,6.70,116291.00,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg
3,6,Judgment Night,6.50,360.00,Released,1993-10-15,12136938.00,109.00,21000000.00,tt0107286,en,Judgment Night,"Four young friends, while taking a shortcut en...",4.27,Don't move. Don't whisper. Don't even breathe.,"Action, Crime, Thriller","Largo Entertainment, JVC, Universal Pictures",United States of America,English,"Doug Wert, Angela Alvarado, Everlast, Sean O'G...",Stephen Hopkins,Peter Levy,"Lewis Colick, Jere Cunningham","Marilyn Vance, Gene Levy, Lloyd Segan",Alan Silvestri,6.60,20817.00,/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg
4,8,Life in Loops (A Megacities RMX),7.20,30.00,Released,2006-01-01,0.00,80.00,42000.00,tt0825671,en,Life in Loops (A Megacities RMX),Timo Novotny labels his new project an experim...,2.26,A Megacities remix.,Documentary,inLoops,Austria,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,Wolfgang Thaler,"Timo Novotny, Michael Glawogger","Timo Novotny, Ulrich Gehmacher",NaN,8.10,285.00,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg



--- Last 5 rows ---


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,original_language,original_title,overview,popularity,tagline,genres,production_companies,production_countries,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path
1157982,1629487,Bismarckpolka,0.00,0.00,Released,NaN,0.00,24.00,0.00,tt0112514,en,Bismarckpolka,"Asti and Bruno, two bums, wander through Hambu...",0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1157983,1629488,Ślimak Niecnota,0.00,0.00,Released,NaN,0.00,0.00,0.00,NaN,en,Ślimak Niecnota,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1157984,1629489,我阿爹想旅行,0.00,0.00,Released,NaN,0.00,0.00,0.00,NaN,zh,我阿爹想旅行,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1157985,1629490,MASKMAN,0.00,0.00,Released,NaN,0.00,0.00,0.00,NaN,en,MASKMAN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1157986,1629491,Sorority Secrets,0.00,0.00,Released,NaN,0.00,0.00,25.00,NaN,en,Sorority Secrets,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- All columns in dataset ---
 1. id
 2. title
 3. vote_average
 4. vote_count
 5. status
 6. release_date
 7. revenue
 8. runtime
 9. budget
10. imdb_id
11. original_language
12. original_title
13. overview
14. popularity
15. tagline
16. genres
17. production_companies
18. production_countries
19. spoken_languages
20. cast
21. director
22. director_of_photography
23. writers
24. producers
25. music_composer
26. imdb_rating
27. imdb_votes
28. poster_path


##  Step 1: Exploratory Data Inspection
- Use df.info() to see dtypes and null counts
- Use df.describe() for numeric columns
- Identify columns with poor data quality

In [4]:
print("Starting exploratory data inspection...")
inspect_start_time = time.time()

print("\n--- Dataset Info ---")
print("Basic information about the dataset:")
df.info()

print("\n--- Statistical Summary of Numeric Columns ---")
print("Descriptive statistics for numeric columns:")
numeric_summary = df.describe()
print(numeric_summary)

Starting exploratory data inspection...

--- Dataset Info ---
Basic information about the dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1157987 entries, 0 to 1157986
Data columns (total 28 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   id                       1157987 non-null  int64  
 1   title                    1157973 non-null  object 
 2   vote_average             1157987 non-null  float64
 3   vote_count               1157987 non-null  float64
 4   status                   1157987 non-null  object 
 5   release_date             1037736 non-null  object 
 6   revenue                  1157987 non-null  float64
 7   runtime                  1157987 non-null  float64
 8   budget                   1157987 non-null  float64
 9   imdb_id                  651968 non-null   object 
 10  original_language        1157987 non-null  object 
 11  original_title           1157974 non-null  object 
 12 

In [5]:
print("\n--- Null Value Analysis ---")
print("Columns with highest percentage of null values:")
null_counts = df.isnull().sum()
null_percentage = (null_counts / len(df)) * 100

null_summary = pd.DataFrame({
    'Null Count': null_counts,
    'Null Percentage': null_percentage
})

# Sort by null percentage descending
null_summary_sorted = null_summary.sort_values('Null Percentage', ascending=False)

# Display only columns with null values
columns_with_nulls = null_summary_sorted[null_summary_sorted['Null Count'] > 0]
print(columns_with_nulls.to_string())


--- Null Value Analysis ---
Columns with highest percentage of null values:
                         Null Count  Null Percentage
music_composer              1029678            88.92
tagline                      982912            84.88
director_of_photography      851179            73.51
producers                    765401            66.10
imdb_votes                   697690            60.25
imdb_rating                  697690            60.25
production_companies         601938            51.98
writers                      573691            49.54
imdb_id                      506019            43.70
production_countries         431500            37.26
spoken_languages             424601            36.67
cast                         368857            31.85
genres                       312775            27.01
poster_path                  286020            24.70
director                     187391            16.18
overview                     176554            15.25
release_date          

In [6]:
print("\n--- Data Quality Assessment ---")
print("Identifying columns with potential data quality issues:")

#  thresholds for poor data quality
high_null_threshold = 50
low_variance_threshold = 0.1  

quality_issues = []

high_null_cols = null_summary_sorted[null_summary_sorted['Null Percentage'] > high_null_threshold]
if not high_null_cols.empty:
    print(f"\nColumns with more than {high_null_threshold}% null values:")
    for col in high_null_cols.index:
        print(f"  - {col}: {null_summary_sorted.loc[col, 'Null Percentage']:.1f}% null")
        quality_issues.append(col)

print("\nNumeric columns with potentially low variance (might not be informative):")
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].nunique() > 0: 
        variance = df[col].var()
        unique_ratio = df[col].nunique() / len(df)
        if unique_ratio < low_variance_threshold and variance < 1:
            print(f"  - {col}: {df[col].nunique()} unique values, variance: {variance:.4f}")

print("\n--- Column Data Types Summary ---")
dtype_counts = df.dtypes.value_counts()
print("Data type distribution:")
for dtype, count in dtype_counts.items():
    print(f"  {dtype}: {count} columns")

print("\n--- Memory Usage by Column ---")
memory_by_column = df.memory_usage(deep=True)
print("Top 10 memory-consuming columns:")
print(memory_by_column.sort_values(ascending=False).head(10).to_string())

inspect_time = time.time() - inspect_start_time
print(f"\nExploratory inspection completed in {inspect_time:.2f} seconds")


--- Data Quality Assessment ---
Identifying columns with potential data quality issues:

Columns with more than 50% null values:
  - music_composer: 88.9% null
  - tagline: 84.9% null
  - director_of_photography: 73.5% null
  - producers: 66.1% null
  - imdb_votes: 60.3% null
  - imdb_rating: 60.3% null
  - production_companies: 52.0% null

Numeric columns with potentially low variance (might not be informative):

--- Column Data Types Summary ---
Data type distribution:
  object: 19 columns
  float64: 8 columns
  int64: 1 columns

--- Memory Usage by Column ---
Top 10 memory-consuming columns:
overview                372350797
cast                    178190720
original_title           86155453
title                    82245045
poster_path              79757806
director                 71629237
status                   66073277
release_date             65074456
production_companies     63262841
writers                  63255998

Exploratory inspection completed in 10.87 seconds


## Exclude Low-Quality Columns
- List columns to drop: ['tagline', 'director_of_photography', 'producers', 'music_composer', 'imdb_rating', 'imdb_votes']
- Drop them from the DataFrame
- Confirm new shape

In [7]:
print("Identifying and excluding low-quality columns...")
drop_start_time = time.time()

columns_to_drop = ['tagline', 'director_of_photography', 'producers', 
                   'music_composer', 'imdb_rating', 'imdb_votes']

existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
non_existing_columns = [col for col in columns_to_drop if col not in df.columns]

print(f"Columns to drop due to poor data quality: {existing_columns_to_drop}")

if non_existing_columns:
    print(f"Note: These columns were not found in the dataset: {non_existing_columns}")

shape_before = df.shape
print(f"Shape before dropping columns: {shape_before[0]} rows, {shape_before[1]} columns")

df = df.drop(columns=existing_columns_to_drop, errors='ignore')

shape_after = df.shape
print(f"Shape after dropping columns: {shape_after[0]} rows, {shape_after[1]} columns")

columns_removed = shape_before[1] - shape_after[1]
print(f"Columns removed: {columns_removed}")

print(f"\nRemaining columns: {shape_after[1]}")
print("Remaining columns list:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3d}. {col}")

drop_time = time.time() - drop_start_time
print(f"\nColumn exclusion completed in {drop_time:.2f} seconds")

Identifying and excluding low-quality columns...
Columns to drop due to poor data quality: ['tagline', 'director_of_photography', 'producers', 'music_composer', 'imdb_rating', 'imdb_votes']
Shape before dropping columns: 1157987 rows, 28 columns
Shape after dropping columns: 1157987 rows, 22 columns
Columns removed: 6

Remaining columns: 22
Remaining columns list:
  1. id
  2. title
  3. vote_average
  4. vote_count
  5. status
  6. release_date
  7. revenue
  8. runtime
  9. budget
 10. imdb_id
 11. original_language
 12. original_title
 13. overview
 14. popularity
 15. genres
 16. production_companies
 17. production_countries
 18. spoken_languages
 19. cast
 20. director
 21. writers
 22. poster_path

Column exclusion completed in 0.50 seconds


## Data Cleaning Strategy: Standardized Validation Framework

To ensure data integrity and auditability across the 28 columns of the TMDB dataset, I have implemented a **Standardized Validation Framework**. 

Rather than treating each column with ad-hoc logic, every feature undergoes a consistent 5-step processing pipeline. This deliberate design choice ensures that every data point is accounted for and that the cleaning process itself is measurable and transparent.

### The 5-Step Pipeline:
1. **Initial Audit**: Logging the starting state (data type, null count, and range) to establish a baseline.
2. **Feature Engineering (Quality Flags)**: Creating binary flags (e.g., `missing_`, `low_confidence_`, `is_outlier_`) for every modification. This preserves the original data state for downstream MapReduce tasks or analysis.
3. **Logic-Based Cleaning**: Applying specific domain logic (e.g., regex validation for `imdb_id` or categorical mapping for `spoken_languages`) to normalize values.
4. **Range & Variance Checks**: Identifying outliers or invalid data (e.g., negative revenue or 0-minute runtimes) using statistical thresholds (IQR).
5. **Post-Audit Logging**: Generating a "Column Cleaning Summary" and a visual sample to verify that the target data type and quality standards have been met.


## Step 2: Critical Columns & Null Handling
- Define critical columns list
- Check null counts in critical columns
- Drop rows where any critical column is null
- Log how many rows were removed

In [8]:
print("Starting critical columns null handling...")
null_handling_start_time = time.time()

critical_columns = ['id', 'release_date', 'revenue', 'budget', 
                    'vote_average', 'genres', 'director', 
                    'original_language', 'cast']

existing_critical_columns = [col for col in critical_columns if col in df.columns]
missing_critical_columns = [col for col in critical_columns if col not in df.columns]

print(f"Critical columns identified: {existing_critical_columns}")

if missing_critical_columns:
    print(f"Warning: These critical columns were not found: {missing_critical_columns}")

Starting critical columns null handling...
Critical columns identified: ['id', 'release_date', 'revenue', 'budget', 'vote_average', 'genres', 'director', 'original_language', 'cast']


In [9]:
#  null counts in critical columns 
print("\n--- Null counts in critical columns (before cleaning) ---")
critical_nulls_before = {}
for col in existing_critical_columns:
    null_count = df[col].isnull().sum()
    null_percentage = (null_count / len(df)) * 100
    critical_nulls_before[col] = {'count': null_count, 'percentage': null_percentage}
    print(f"{col:25s}: {null_count:6d} nulls ({null_percentage:6.2f}%)")

rows_before = len(df)
print(f"\nTotal rows before cleaning: {rows_before}")

print("\nDropping rows with null values in critical columns...")
df_clean = df.dropna(subset=existing_critical_columns)

rows_after = len(df_clean)
rows_removed = rows_before - rows_after
removal_percentage = (rows_removed / rows_before) * 100

print(f"Rows after cleaning: {rows_after}")
print(f"Rows removed: {rows_removed} ({removal_percentage:.2f}% of original dataset)")


--- Null counts in critical columns (before cleaning) ---
id                       :      0 nulls (  0.00%)
release_date             : 120251 nulls ( 10.38%)
revenue                  :      0 nulls (  0.00%)
budget                   :      0 nulls (  0.00%)
vote_average             :      0 nulls (  0.00%)
genres                   : 312775 nulls ( 27.01%)
director                 : 187391 nulls ( 16.18%)
original_language        :      0 nulls (  0.00%)
cast                     : 368857 nulls ( 31.85%)

Total rows before cleaning: 1157987

Dropping rows with null values in critical columns...
Rows after cleaning: 590202
Rows removed: 567785 (49.03% of original dataset)


In [10]:
df = df_clean.copy()
print("\n--- Null counts in critical columns (after cleaning) ---")
for col in existing_critical_columns:
    null_count = df[col].isnull().sum()
    print(f"{col:25s}: {null_count:6d} nulls")

print("\n--- Cleaning Summary ---")
print(f"Original dataset size: {rows_before} rows")
print(f"Cleaned dataset size: {rows_after} rows")
print(f"Rows removed due to nulls in critical columns: {rows_removed}")
print(f"Percentage of data retained: {(rows_after/rows_before)*100:.2f}%")

null_handling_time = time.time() - null_handling_start_time
print(f"\nNull handling completed in {null_handling_time:.2f} seconds")


--- Null counts in critical columns (after cleaning) ---
id                       :      0 nulls
release_date             :      0 nulls
revenue                  :      0 nulls
budget                   :      0 nulls
vote_average             :      0 nulls
genres                   :      0 nulls
director                 :      0 nulls
original_language        :      0 nulls
cast                     :      0 nulls

--- Cleaning Summary ---
Original dataset size: 1157987 rows
Cleaned dataset size: 590202 rows
Rows removed due to nulls in critical columns: 567785
Percentage of data retained: 50.97%

Null handling completed in 1.91 seconds


**Data Quality Assessment:**
- Initial dataset: 1,157,987 rows
- Critical columns showed significant null values: 
  cast (31.85%), genres (27.01%), director (16.18%)
- Decision: Remove rows with nulls in critical columns to ensure analysis reliability
- Result: 590,202 high-quality rows retained (49.03% reduction)

**Justification:** While this reduces volume, it increases data veracity, 
ensuring our analysis on genres, directors, and cast is based on complete records.
This trade-off is common in big data projects where data quality varies significantly.

## Step 3: Column-by-Column Cleaning

### 1 Numeric Columns
- revenue, budget, runtime, vote_average, popularity, vote_count
- Convert dtypes, handle nulls (impute median/0), validate ranges

### 2 Date Column
- release_date: parse to datetime, drop invalid dates

### 3 Text/Categorical Columns
- title, original_language, director, status, overview
- Fill nulls with "Unknown" or placeholder text

### 4 List/JSON Columns
- genres, cast, production_companies, production_countries, spoken_languages, writers
- Parse from JSON/string to Python list
- Fill nulls with ["Unknown"]

In [11]:
# Clean Title Column
print("Cleaning title column...")
title_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'title' column: {df['title'].dtype}")
print(f"Sample titles (first 10):")
print(df['title'].head(10).to_string())

# 2. Check for null values
title_nulls_before = df['title'].isnull().sum()
print(f"\\nNull titles before cleaning: {title_nulls_before} ({title_nulls_before/len(df)*100:.2f}%)")

# 3. Create a column to mark missing titles before cleaning
df['missing_title'] = df['title'].isnull()
print(f"\\nCreated 'missing_title' column: {df['missing_title'].sum()} rows marked as True")

# 4. Fill missing titles with "Unknown"
df['title'] = df['title'].fillna("Unknown")
title_nulls_after = df['title'].isnull().sum()
print(f"Null titles after filling: {title_nulls_after}")

# 5. Ensure all titles are strings
df['title'] = df['title'].astype(str)
print(f"Data type after conversion: {df['title'].dtype}")

# 6. Basic English language check (detect non-Latin characters)
def contains_non_latin(text):
    # Check if string contains characters outside basic Latin alphabet
    latin_pattern = re.compile(r'^[a-zA-Z0-9\s\-\'\",.!?:;()\[\]]*$')
    return not bool(latin_pattern.fullmatch(str(text)))

non_latin_titles = df['title'].apply(contains_non_latin).sum()
print(f"\\nTitles that may contain non-Latin characters: {non_latin_titles}")
print("Sample of titles with non-Latin characters:")
non_latin_samples = df[df['title'].apply(contains_non_latin)]['title'].head(10)
for i, title in enumerate(non_latin_samples, 1):
    print(f"  {i:2d}. {title}")

# 7. Check for empty strings after conversion
empty_titles = (df['title'] == "").sum()
print(f"\\nEmpty string titles: {empty_titles}")

# 8. Verify the cleaning
print("\\n--- Title Column Cleaning Summary ---")
print(f"1. Original null count: {title_nulls_before}")
print(f"2. Created 'missing_title' column: {df['missing_title'].sum()} True values")
print(f"3. Current null count: {title_nulls_after}")
print(f"4. Data type: {df['title'].dtype}")
print(f"5. Total unique titles: {df['title'].nunique()}")

print("\\n--- Sample after cleaning ---")
print("First 5 rows with title and missing_title flag:")
sample_df = df[['title', 'missing_title']].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:50] + '...' if len(x) > 50 else x)
print(sample_df[['title_truncated', 'missing_title']].to_string(index=False))

title_clean_time = time.time() - title_clean_start
print(f"\\nTitle column cleaning completed in {title_clean_time:.2f} seconds")

Cleaning title column...
Current data type of 'title' column: object
Sample titles (first 10):
0                   Ariel
1     Shadows in Paradise
2              Four Rooms
3          Judgment Night
5        Sunday in August
6               Star Wars
7            Finding Nemo
8            Forrest Gump
9         American Beauty
10           Citizen Kane
\nNull titles before cleaning: 3 (0.00%)
\nCreated 'missing_title' column: 3 rows marked as True
Null titles after filling: 0
Data type after conversion: object
\nTitles that may contain non-Latin characters: 57678
Sample of titles with non-Latin characters:
   1. Nausicaä of the Valley of the Wind
   2. Léon: The Professional
   3. Amélie
   4. Willy Wonka & the Chocolate Factory
   5. Bollywood/Hollywood
   6. Batman & Robin
   7. 8½
   8. Caché
   9. Romeo + Juliet
  10. Cléo from 5 to 7
\nEmpty string titles: 0
\n--- Title Column Cleaning Summary ---
1. Original null count: 3
2. Created 'missing_title' column: 3 True values
3. Curren

In [12]:
#  vote_average Column
print("Cleaning vote_average column...")
vote_avg_clean_start = time.time()

print(f"Current data type of 'vote_average' column: {df['vote_average'].dtype}")
print(f"\\n--- Basic Statistics Before Cleaning ---")
print(f"Min: {df['vote_average'].min():.2f}")
print(f"Max: {df['vote_average'].max():.2f}")
print(f"Mean: {df['vote_average'].mean():.2f}")
print(f"Median: {df['vote_average'].median():.2f}")
print(f"Null count: {df['vote_average'].isnull().sum()}")

df['missing_vote_average'] = df['vote_average'].isnull()
missing_count = df['missing_vote_average'].sum()
print(f"\\nCreated 'missing_vote_average' column: {missing_count} rows marked as True")

invalid_votes = df[(df['vote_average'] < 0) | (df['vote_average'] > 10)].shape[0]
print(f"\\nVote averages outside valid range (0-10): {invalid_votes}")

if invalid_votes > 0:
    print("Sample of invalid vote averages:")
    invalid_samples = df[(df['vote_average'] < 0) | (df['vote_average'] > 10)][['title', 'vote_average']].head(5)
    print(invalid_samples.to_string(index=False))

# Handle missing values (impute with median)
vote_median = df['vote_average'].median()
print(f"\\nMedian vote average for imputation: {vote_median:.2f}")

df['vote_average'] = df['vote_average'].fillna(vote_median)
print(f"Null vote averages after imputation: {df['vote_average'].isnull().sum()}")

# 5. Ensure proper data type (float for ratings)
df['vote_average'] = df['vote_average'].astype(float)
print(f"Data type after conversion: {df['vote_average'].dtype}")

# 6. Clip values to valid range (0-10) if needed
if invalid_votes > 0:
    df['vote_average'] = df['vote_average'].clip(0, 10)
    print(f"\\nClipped {invalid_votes} values to 0-10 range")

# 7. Check for zero votes (might indicate low reliability)
zero_votes = (df['vote_average'] == 0).sum()
print(f"\\nMovies with 0.0 vote average: {zero_votes} ({zero_votes/len(df)*100:.2f}%)")

# 8. Create a quality flag for potentially unreliable ratings
df['low_confidence_rating'] = (df['vote_average'] == 0) | (df['missing_vote_average'])
low_conf_count = df['low_confidence_rating'].sum()
print(f"\\nCreated 'low_confidence_rating' flag: {low_conf_count} rows marked as True")
print("(Includes original nulls and 0.0 ratings)")

# 9. Verify the cleaning
print("\\n--- vote_average Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count}")
print(f"2. Created 'missing_vote_average' column: {missing_count} True values")
print(f"3. Created 'low_confidence_rating' column: {low_conf_count} True values")
print(f"4. Current null count: {df['vote_average'].isnull().sum()}")
print(f"5. Data type: {df['vote_average'].dtype}")
print(f"6. Valid range check: {df['vote_average'].min():.2f} to {df['vote_average'].max():.2f}")

print("\\n--- Statistics After Cleaning ---")
print(f"Min: {df['vote_average'].min():.2f}")
print(f"Max: {df['vote_average'].max():.2f}")
print(f"Mean: {df['vote_average'].mean():.2f}")
print(f"Median: {df['vote_average'].median():.2f}")
print(f"Standard Deviation: {df['vote_average'].std():.2f}")

print("\\n--- Sample after cleaning ---")
print("First 5 rows with vote_average and quality flags:")
sample_cols = ['title', 'vote_average', 'missing_vote_average', 'low_confidence_rating']
sample_df = df[sample_cols].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:30] + '...' if len(x) > 30 else x)
print(sample_df[['title_truncated', 'vote_average', 'missing_vote_average', 'low_confidence_rating']].to_string(index=False))

# 10. Distribution analysis
print("\\n--- Distribution Analysis ---")
bins = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
labels = ['0-1', '1-2', '2-3', '3-4', '4-5', '5-6', '6-7', '7-8', '8-9', '9-10']
df['vote_range'] = pd.cut(df['vote_average'], bins=bins, labels=labels, include_lowest=True)

distribution = df['vote_range'].value_counts().sort_index()
print("Vote average distribution:")
for range_label, count in distribution.items():
    percentage = (count / len(df)) * 100
    print(f"  {range_label}: {count:6d} movies ({percentage:5.1f}%)")

vote_avg_clean_time = time.time() - vote_avg_clean_start
print(f"\\nvote_average column cleaning completed in {vote_avg_clean_time:.2f} seconds")

Cleaning vote_average column...
Current data type of 'vote_average' column: float64
\n--- Basic Statistics Before Cleaning ---
Min: 0.00
Max: 10.00
Mean: 3.00
Median: 1.00
Null count: 0
\nCreated 'missing_vote_average' column: 0 rows marked as True
\nVote averages outside valid range (0-10): 0
\nMedian vote average for imputation: 1.00
Null vote averages after imputation: 0
Data type after conversion: float64
\nMovies with 0.0 vote average: 289625 (49.07%)
\nCreated 'low_confidence_rating' flag: 289625 rows marked as True
(Includes original nulls and 0.0 ratings)
\n--- vote_average Column Cleaning Summary ---
1. Original null count: 0
2. Created 'missing_vote_average' column: 0 True values
3. Created 'low_confidence_rating' column: 289625 True values
4. Current null count: 0
5. Data type: float64
6. Valid range check: 0.00 to 10.00
\n--- Statistics After Cleaning ---
Min: 0.00
Max: 10.00
Mean: 3.00
Median: 1.00
Standard Deviation: 3.24
\n--- Sample after cleaning ---
First 5 rows with 

In [13]:
# vote count 
print("Cleaning vote_count column...")
vote_count_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'vote_count' column: {df['vote_count'].dtype}")
print(f"\\n--- Basic Statistics Before Cleaning ---")
print(f"Min: {df['vote_count'].min():.0f}")
print(f"Max: {df['vote_count'].max():.0f}")
print(f"Mean: {df['vote_count'].mean():.2f}")
print(f"Median: {df['vote_count'].median():.2f}")
print(f"Null count: {df['vote_count'].isnull().sum()}")

# 2. Create a column to mark missing vote_count before cleaning
df['missing_vote_count'] = df['vote_count'].isnull()
missing_count = df['missing_vote_count'].sum()
print(f"\\nCreated 'missing_vote_count' column: {missing_count} rows marked as True")

# 3. Check for invalid values (negative counts)
negative_votes = df[df['vote_count'] < 0].shape[0]
print(f"\\nNegative vote counts: {negative_votes}")

if negative_votes > 0:
    print("Sample of negative vote counts:")
    negative_samples = df[df['vote_count'] < 0][['title', 'vote_count']].head(5)
    print(negative_samples.to_string(index=False))

# 4. Check for decimal values (vote counts should be integers)
decimal_votes = df[df['vote_count'] != df['vote_count'].astype(int)].shape[0]
print(f"\\nDecimal vote counts: {decimal_votes}")

if decimal_votes > 0:
    print("Sample of decimal vote counts:")
    decimal_samples = df[df['vote_count'] != df['vote_count'].astype(int)][['title', 'vote_count']].head(5)
    print(decimal_samples.to_string(index=False))

# 5. Handle missing values (impute with median)
vote_count_median = df['vote_count'].median()
print(f"\\nMedian vote count for imputation: {vote_count_median:.2f}")

df['vote_count'] = df['vote_count'].fillna(vote_count_median)
print(f"Null vote counts after imputation: {df['vote_count'].isnull().sum()}")

# 6. Ensure proper data type (int for counts)
df['vote_count'] = df['vote_count'].astype(int)
print(f"Data type after conversion: {df['vote_count'].dtype}")

# 7. Clip negative values to 0 if needed
if negative_votes > 0:
    df['vote_count'] = df['vote_count'].clip(lower=0)
    print(f"\\nClipped {negative_votes} negative values to 0")

# 8. Check for zero votes (correlates with 0.0 vote_average)
zero_votes = (df['vote_count'] == 0).sum()
print(f"\\nMovies with 0 vote count: {zero_votes} ({zero_votes/len(df)*100:.2f}%)")

# 9. Investigate relationship with vote_average
print(f"\\n--- Correlation with vote_average ---")
zero_vote_but_rating = df[(df['vote_count'] == 0) & (df['vote_average'] > 0)].shape[0]
print(f"Movies with 0 votes but positive rating: {zero_vote_but_rating}")

positive_vote_zero_rating = df[(df['vote_count'] > 0) & (df['vote_average'] == 0)].shape[0]
print(f"Movies with positive votes but 0.0 rating: {positive_vote_zero_rating}")

both_zero = df[(df['vote_count'] == 0) & (df['vote_average'] == 0)].shape[0]
print(f"Movies with 0 votes AND 0.0 rating: {both_zero} ({both_zero/len(df)*100:.2f}%)")

# 10. Create a quality flag for potentially unreliable vote counts
df['low_vote_count'] = (df['vote_count'] == 0) | (df['missing_vote_count'])
low_vote_count = df['low_vote_count'].sum()
print(f"\\nCreated 'low_vote_count' flag: {low_vote_count} rows marked as True")

# 11. Verify the cleaning
print("\\n--- vote_count Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count}")
print(f"2. Created 'missing_vote_count' column: {missing_count} True values")
print(f"3. Created 'low_vote_count' flag: {low_vote_count} True values")
print(f"4. Current null count: {df['vote_count'].isnull().sum()}")
print(f"5. Data type: {df['vote_count'].dtype}")
print(f"6. Valid range check: {df['vote_count'].min()} to {df['vote_count'].max()}")

print("\\n--- Statistics After Cleaning ---")
print(f"Min: {df['vote_count'].min():.0f}")
print(f"Max: {df['vote_count'].max():.0f}")
print(f"Mean: {df['vote_count'].mean():.2f}")
print(f"Median: {df['vote_count'].median():.2f}")
print(f"Standard Deviation: {df['vote_count'].std():.2f}")

print("\\n--- Distribution Analysis ---")
# Create vote count categories
bins = [0, 1, 10, 100, 1000, 10000, 100000, float('inf')]
labels = ['0', '1-10', '11-100', '101-1K', '1K-10K', '10K-100K', '100K+']
df['vote_count_range'] = pd.cut(df['vote_count'], bins=bins, labels=labels, right=False)

distribution = df['vote_count_range'].value_counts().sort_index()
print("Vote count distribution:")
for range_label, count in distribution.items():
    percentage = (count / len(df)) * 100
    print(f"  {range_label}: {count:6d} movies ({percentage:5.1f}%)")

print("\\n--- Sample after cleaning ---")
print("First 5 rows with vote_count and quality flags:")
sample_cols = ['title', 'vote_count', 'vote_average', 'missing_vote_count', 'low_vote_count']
sample_df = df[sample_cols].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:25] + '...' if len(x) > 25 else x)
print(sample_df[['title_truncated', 'vote_count', 'vote_average', 'missing_vote_count', 'low_vote_count']].to_string(index=False))

# 12. Advanced analysis: Top voted movies
print("\\n--- Top 10 Most Voted Movies ---")
top_voted = df[['title', 'vote_count', 'vote_average']].sort_values('vote_count', ascending=False).head(10)
top_voted['title_truncated'] = top_voted['title'].apply(lambda x: x[:40] + '...' if len(x) > 40 else x)
print(top_voted[['title_truncated', 'vote_count', 'vote_average']].to_string(index=False))

# 13. Correlation analysis
correlation = df['vote_count'].corr(df['vote_average'])
print(f"\\nCorrelation between vote_count and vote_average: {correlation:.3f}")

vote_count_clean_time = time.time() - vote_count_clean_start
print(f"\\nvote_count column cleaning completed in {vote_count_clean_time:.2f} seconds")

Cleaning vote_count column...
Current data type of 'vote_count' column: float64
\n--- Basic Statistics Before Cleaning ---
Min: 0
Max: 38784
Mean: 43.97
Median: 1.00
Null count: 0
\nCreated 'missing_vote_count' column: 0 rows marked as True
\nNegative vote counts: 0
\nDecimal vote counts: 0
\nMedian vote count for imputation: 1.00
Null vote counts after imputation: 0
Data type after conversion: int32
\nMovies with 0 vote count: 289304 (49.02%)
\n--- Correlation with vote_average ---
Movies with 0 votes but positive rating: 500
Movies with positive votes but 0.0 rating: 821
Movies with 0 votes AND 0.0 rating: 288804 (48.93%)
\nCreated 'low_vote_count' flag: 289304 rows marked as True
\n--- vote_count Column Cleaning Summary ---
1. Original null count: 0
2. Created 'missing_vote_count' column: 0 True values
3. Created 'low_vote_count' flag: 289304 True values
4. Current null count: 0
5. Data type: int32
6. Valid range check: 0 to 38784
\n--- Statistics After Cleaning ---
Min: 0
Max: 3878

In [14]:
# Clean status Column (Categorical)
print("Cleaning status column...")
status_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'status' column: {df['status'].dtype}")
print(f"Sample values (first 10):")
print(df['status'].head(10).to_string(index=False))

# 2. Check number of unique values before cleaning
unique_status_before = df['status'].nunique()
print(f"\\nNumber of unique status values before cleaning: {unique_status_before}")
print("\\nUnique status values and their counts:")
status_counts = df['status'].value_counts(dropna=False)
print(status_counts.to_string())

# 3. Check null values
null_status_before = df['status'].isnull().sum()
print(f"\\nNull status values before cleaning: {null_status_before} ({null_status_before/len(df)*100:.2f}%)")

# 4. Create a column to mark missing status before cleaning
df['missing_status'] = df['status'].isnull()
print(f"\\nCreated 'missing_status' column: {df['missing_status'].sum()} rows marked as True")

# 5. Check for inconsistent capitalization or whitespace
print(f"\\n--- Checking for data consistency issues ---")

# Check for leading/trailing whitespace
whitespace_issues = df['status'].apply(lambda x: isinstance(x, str) and x != x.strip()).sum()
print(f"Rows with leading/trailing whitespace: {whitespace_issues}")

# Check for inconsistent capitalization
if null_status_before < len(df):
    # Get all unique non-null values
    unique_values = df['status'].dropna().unique()
    print(f"\\nAll unique status values (case-sensitive):")
    for i, value in enumerate(sorted(unique_values), 1):
        count = (df['status'] == value).sum()
        print(f"  {i:2d}. '{value}' : {count:6d} rows")

# 6. Handle missing values (fill with most common status)
most_common_status = df['status'].mode()[0] if not df['status'].mode().empty else "Unknown"
print(f"\\nMost common status: '{most_common_status}'")

df['status'] = df['status'].fillna(most_common_status)
print(f"Null status after imputation: {df['status'].isnull().sum()}")

# 7. Standardize text (trim whitespace, standardize case)
df['status'] = df['status'].astype(str).str.strip().str.title()
print("\\nStandardized all values to title case and trimmed whitespace")

# 8. Check for any remaining unexpected values
unique_status_after = df['status'].nunique()
print(f"\\nNumber of unique status values after cleaning: {unique_status_after}")

if unique_status_after <= 10:  # Only show if reasonable number of categories
    print("\\nFinal unique status values and counts:")
    final_status_counts = df['status'].value_counts()
    print(final_status_counts.to_string())
else:
    print(f"\\nShowing top 10 most common status values:")
    top_status = df['status'].value_counts().head(10)
    print(top_status.to_string())

# 9. Validate the cleaning
print("\\n--- status Column Cleaning Summary ---")
print(f"1. Original unique values: {unique_status_before}")
print(f"2. Original null count: {null_status_before}")
print(f"3. Created 'missing_status' column: {df['missing_status'].sum()} True values")
print(f"4. Current null count: {df['status'].isnull().sum()}")
print(f"5. Final unique values: {unique_status_after}")
print(f"6. Data type: {df['status'].dtype}")

# 10. Check relationship with release_date
print(f"\\n--- Relationship Analysis ---")
if 'release_date' in df.columns:
    # Check if movies marked as "Released" have release dates
    released_without_date = df[(df['status'] == 'Released') & (df['release_date'].isnull())].shape[0]
    print(f"Movies marked 'Released' but missing release_date: {released_without_date}")
    
    not_released_with_date = df[(df['status'] != 'Released') & (df['release_date'].notnull())].shape[0]
    print(f"Movies not 'Released' but have release_date: {not_released_with_date}")

# 11. Sample after cleaning
print("\\n--- Sample after cleaning ---")
print("First 10 rows with status and missing_status flag:")
sample_df = df[['title', 'status', 'missing_status']].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:25] + '...' if len(x) > 25 else x)
print(sample_df[['title_truncated', 'status', 'missing_status']].to_string(index=False))

status_clean_time = time.time() - status_clean_start
print(f"\\nstatus column cleaning completed in {status_clean_time:.2f} seconds")

Cleaning status column...
Current data type of 'status' column: object
Sample values (first 10):
Released
Released
Released
Released
Released
Released
Released
Released
Released
Released
\nNumber of unique status values before cleaning: 6
\nUnique status values and their counts:
status
Released           588545
Post Production       766
In Production         749
Planned               137
Canceled                3
Rumored                 2
\nNull status values before cleaning: 0 (0.00%)
\nCreated 'missing_status' column: 0 rows marked as True
\n--- Checking for data consistency issues ---
Rows with leading/trailing whitespace: 0
\nAll unique status values (case-sensitive):
   1. 'Canceled' :      3 rows
   2. 'In Production' :    749 rows
   3. 'Planned' :    137 rows
   4. 'Post Production' :    766 rows
   5. 'Released' : 588545 rows
   6. 'Rumored' :      2 rows
\nMost common status: 'Released'
Null status after imputation: 0
\nStandardized all values to title case and trimmed whites

In [15]:
# Clean release_date Column
print("Cleaning release_date column...")
release_date_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'release_date' column: {df['release_date'].dtype}")
print(f"\\nSample values (first 10):")
print(df['release_date'].head(10).to_string(index=False))

# 2. Check null values before cleaning
null_release_before = df['release_date'].isnull().sum()
print(f"\\nNull release_date values before cleaning: {null_release_before} ({null_release_before/len(df)*100:.2f}%)")

# 3. Create a column to mark missing release_date before cleaning
df['missing_release_date'] = df['release_date'].isnull()
print(f"\\nCreated 'missing_release_date' column: {df['missing_release_date'].sum()} rows marked as True")

# 4. Check for invalid date formats or impossible dates
print(f"\\n--- Checking for date format issues ---")

# Count rows that don't match expected date pattern
date_pattern = r'^\\d{4}-\\d{2}-\\d{2}$'
invalid_format_count = df['release_date'].apply(
    lambda x: pd.notna(x) and not re.match(date_pattern, str(x))
).sum()
print(f"Rows with non-standard date format: {invalid_format_count}")

if invalid_format_count > 0:
    print("\\nSample of non-standard date formats:")
    invalid_samples = df[df['release_date'].apply(
        lambda x: pd.notna(x) and not re.match(date_pattern, str(x))
    )]['release_date'].unique()[:10]
    for sample in invalid_samples:
        print(f"  '{sample}'")

# 5. Convert to datetime format
print(f"\\nConverting to datetime format...")
# First, ensure all values are strings
df['release_date'] = df['release_date'].astype(str)

# Replace empty strings and 'nan' with actual NaN
df['release_date'] = df['release_date'].replace(['nan', 'NaN', '', 'None', 'null', 'Null'], pd.NA)

# Convert to datetime with error handling
df['release_date_clean'] = pd.to_datetime(df['release_date'], errors='coerce', format='%Y-%m-%d')

# Check how many failed to parse
failed_parsing = df['release_date_clean'].isnull().sum() - null_release_before
print(f"Rows that failed datetime parsing: {failed_parsing}")

# 6. Check for impossible dates (future dates, very old dates)
if df['release_date_clean'].notna().any():
    current_year = pd.Timestamp.now().year
    future_dates = df[df['release_date_clean'] > pd.Timestamp.now()].shape[0]
    very_old_dates = df[df['release_date_clean'] < pd.Timestamp('1888-01-01')].shape[0]  # First film 1888
    
    print(f"\\n--- Date Validation ---")
    print(f"Future release dates (after {current_year}): {future_dates}")
    print(f"Very old dates (before 1888): {very_old_dates}")
    
    if future_dates > 0:
        print("\\nSample of future release dates:")
        future_samples = df[df['release_date_clean'] > pd.Timestamp.now()][['title', 'release_date_clean']].head(5)
        print(future_samples.to_string(index=False))
    
    if very_old_dates > 0:
        print("\\nSample of very old release dates:")
        old_samples = df[df['release_date_clean'] < pd.Timestamp('1888-01-01')][['title', 'release_date_clean']].head(5)
        print(old_samples.to_string(index=False))

# 7. Extract year, month, day for analysis
print(f"\\n--- Extracting date components ---")
df['release_year'] = df['release_date_clean'].dt.year
df['release_month'] = df['release_date_clean'].dt.month
df['release_day'] = df['release_date_clean'].dt.day
df['release_weekday'] = df['release_date_clean'].dt.day_name()

print(f"Created date component columns: release_year, release_month, release_day, release_weekday")

# 8. Handle missing/impossible dates
print(f"\\n--- Handling missing/invalid dates ---")

# Count current nulls after parsing
current_nulls = df['release_date_clean'].isnull().sum()
print(f"Total null/invalid dates after parsing: {current_nulls} ({current_nulls/len(df)*100:.2f}%)")

# For movies with null dates but have status='Released', we might infer date from other data
if 'status' in df.columns:
    released_without_date = df[(df['status'] == 'Released') & (df['release_date_clean'].isnull())].shape[0]
    print(f"Movies marked 'Released' but missing release date: {released_without_date}")

# 9. Create quality flags
df['invalid_release_date'] = df['release_date_clean'].isnull()
df['future_release_date'] = df['release_date_clean'] > pd.Timestamp.now()

print(f"\\nCreated quality flags:")
print(f"  - 'invalid_release_date': {df['invalid_release_date'].sum()} rows")
print(f"  - 'future_release_date': {df['future_release_date'].sum()} rows")

# 10. Replace original column with cleaned version
df['release_date'] = df['release_date_clean']
df = df.drop(columns=['release_date_clean'])

# 11. Verify the cleaning
print("\\n--- release_date Column Cleaning Summary ---")
print(f"1. Original null count: {null_release_before}")
print(f"2. Created 'missing_release_date' column: {df['missing_release_date'].sum()} True values")
print(f"3. Created 'invalid_release_date' column: {df['invalid_release_date'].sum()} True values")
print(f"4. Created 'future_release_date' column: {df['future_release_date'].sum()} True values")
print(f"5. Current null count: {df['release_date'].isnull().sum()}")
print(f"6. Data type: {df['release_date'].dtype}")

# 12. Date distribution analysis
if df['release_date'].notna().any():
    print(f"\\n--- Date Distribution Analysis ---")
    
    # Year range
    min_year = int(df['release_year'].min())
    max_year = int(df['release_year'].max())
    print(f"Release year range: {min_year} to {max_year}")
    
    # Decade distribution
    df['release_decade'] = (df['release_year'] // 10) * 10
    decade_counts = df['release_decade'].value_counts().sort_index()
    
    print(f"\\nMovies by decade:")
    for decade, count in decade_counts.head(15).items():  # Show top 15 decades
        if not pd.isna(decade):
            print(f"  {int(decade)}s: {count:6d} movies")
    
    # Monthly distribution
    print(f"\\nMovies by month (all years):")
    month_counts = df['release_month'].value_counts().sort_index()
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    for month_num in range(1, 13):
        count = month_counts.get(month_num, 0)
        print(f"  {month_names[month_num-1]}: {count:6d} movies")

# 13. Sample after cleaning
print("\\n--- Sample after cleaning ---")
print("First 10 rows with release_date and components:")
sample_cols = ['title', 'release_date', 'release_year', 'release_month', 'release_weekday']
sample_df = df[sample_cols].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
sample_df['release_date_str'] = sample_df['release_date'].dt.strftime('%Y-%m-%d')
print(sample_df[['title_truncated', 'release_date_str', 'release_year', 'release_month', 'release_weekday']].to_string(index=False))

release_date_clean_time = time.time() - release_date_clean_start
print(f"\\nrelease_date column cleaning completed in {release_date_clean_time:.2f} seconds")

Cleaning release_date column...
Current data type of 'release_date' column: object
\nSample values (first 10):
1988-10-21
1986-10-17
1995-12-09
1993-10-15
2004-09-02
1977-05-25
2003-05-30
1994-06-23
1999-09-15
1941-04-17
\nNull release_date values before cleaning: 0 (0.00%)
\nCreated 'missing_release_date' column: 0 rows marked as True
\n--- Checking for date format issues ---
Rows with non-standard date format: 590202
\nSample of non-standard date formats:
  '1988-10-21'
  '1986-10-17'
  '1995-12-09'
  '1993-10-15'
  '2004-09-02'
  '1977-05-25'
  '2003-05-30'
  '1994-06-23'
  '1999-09-15'
  '1941-04-17'
\nConverting to datetime format...
Rows that failed datetime parsing: 0
\n--- Date Validation ---
Future release dates (after 2026): 1386
Very old dates (before 1888): 0
\nSample of future release dates:
      title release_date_clean
   Avatar 4         2029-12-19
G.O.D. Tech         2026-12-15
 Juega Vivo         2026-05-02
   Avatar 5         2031-12-17
    Shrek 5         2027-06-3

In [16]:
# Clean revenue Column
print("Cleaning revenue column...")
revenue_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'revenue' column: {df['revenue'].dtype}")
print(f"\\n--- Basic Statistics Before Cleaning ---")
print(f"Min: ${df['revenue'].min():,.0f}")
print(f"Max: ${df['revenue'].max():,.0f}")
print(f"Mean: ${df['revenue'].mean():,.0f}")
print(f"Median: ${df['revenue'].median():,.0f}")
print(f"Std Dev: ${df['revenue'].std():,.0f}")
print(f"Null count: {df['revenue'].isnull().sum()}")

# 2. Show distribution percentiles
print(f"\\n--- Revenue Distribution Percentiles ---")
percentiles = [0, 25, 50, 75, 90, 95, 99, 100]
for p in percentiles:
    value = df['revenue'].quantile(p/100)
    print(f"{p:3d}th percentile: ${value:,.0f}")

# 3. Create a column to mark missing revenue before cleaning
df['missing_revenue'] = df['revenue'].isnull()
missing_count = df['missing_revenue'].sum()
print(f"\\nCreated 'missing_revenue' column: {missing_count} rows marked as True")

# 4. Check for invalid values
print(f"\\n--- Checking for Invalid Values ---")

# Check for negative revenue (shouldn't exist for movies)
negative_revenue = df[df['revenue'] < 0].shape[0]
print(f"Negative revenue values: {negative_revenue}")

if negative_revenue > 0:
    print("\\nSample of negative revenue:")
    negative_samples = df[df['revenue'] < 0][['title', 'revenue']].head(5)
    print(negative_samples.to_string(index=False))

# Check for decimal values (revenue should typically be whole dollars)
decimal_revenue = df[df['revenue'] != df['revenue'].astype(int)].shape[0]
print(f"\\nDecimal revenue values: {decimal_revenue}")

if decimal_revenue > 0:
    print("Sample of decimal revenue:")
    decimal_samples = df[df['revenue'] != df['revenue'].astype(int)][['title', 'revenue']].head(5)
    print(decimal_samples.to_string(index=False))

# Check for unreasonably high values (outliers)
revenue_q75 = df['revenue'].quantile(0.75)
revenue_q99 = df['revenue'].quantile(0.99)
iqr = df['revenue'].quantile(0.75) - df['revenue'].quantile(0.25)
outlier_threshold = df['revenue'].quantile(0.75) + 1.5 * iqr

high_outliers = df[df['revenue'] > outlier_threshold].shape[0]
print(f"\\nPotential high outliers (above ${outlier_threshold:,.0f}): {high_outliers}")

# 5. Check for zero revenue
zero_revenue = (df['revenue'] == 0).sum()
print(f"\\nMovies with $0 revenue: {zero_revenue} ({zero_revenue/len(df)*100:.2f}%)")

# Show some samples of high revenue vs zero revenue
if zero_revenue > 0:
    print("\\nSample of movies with $0 revenue:")
    zero_samples = df[df['revenue'] == 0][['title', 'revenue', 'budget']].head(5)
    print(zero_samples.to_string(index=False))

# 6. Handle missing values (impute with median for this dataset)
revenue_median = df['revenue'].median()
print(f"\\nMedian revenue for imputation: ${revenue_median:,.0f}")

df['revenue'] = df['revenue'].fillna(revenue_median)
print(f"Null revenue after imputation: {df['revenue'].isnull().sum()}")

# 7. Ensure proper data type (int64 for currency - FIXED from int32)
# Use int64 to avoid overflow (max value: 9.22 quintillion)
df['revenue'] = df['revenue'].astype('int64')
print(f"Data type after conversion: {df['revenue'].dtype}")
print(f"int64 range: -9.22 quintillion to +9.22 quintillion (safe for any movie revenue)")

# 8. Clip negative values to 0 if needed
if negative_revenue > 0:
    df['revenue'] = df['revenue'].clip(lower=0)
    print(f"\\nClipped {negative_revenue} negative values to 0")

# 9. Create categories for revenue analysis
print(f"\\n--- Creating Revenue Categories ---")

# Define revenue brackets
bins = [-1, 0, 10000, 100000, 1000000, 10000000, 100000000, 1000000000, float('inf')]
labels = ['$0', '$1-10K', '$10K-100K', '$100K-1M', '$1M-10M', '$10M-100M', '$100M-1B', '$1B+']

df['revenue_category'] = pd.cut(df['revenue'], bins=bins, labels=labels)
print(f"Created 'revenue_category' column with {len(labels)} categories")

# 10. Create quality flags
df['zero_revenue'] = (df['revenue'] == 0)
df['low_revenue'] = (df['revenue'] > 0) & (df['revenue'] < 10000)  # Less than $10K
df['high_revenue_outlier'] = df['revenue'] > outlier_threshold

print(f"\\nCreated quality flags:")
print(f"  - 'zero_revenue': {df['zero_revenue'].sum()} rows ({df['zero_revenue'].sum()/len(df)*100:.1f}%)")
print(f"  - 'low_revenue' (<$10K): {df['low_revenue'].sum()} rows ({df['low_revenue'].sum()/len(df)*100:.1f}%)")
print(f"  - 'high_revenue_outlier': {df['high_revenue_outlier'].sum()} rows")

# 11. Verify the cleaning
print("\\n--- revenue Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count}")
print(f"2. Created 'missing_revenue' column: {missing_count} True values")
print(f"3. Created 'zero_revenue' column: {df['zero_revenue'].sum()} True values")
print(f"4. Created 'low_revenue' column: {df['low_revenue'].sum()} True values")
print(f"5. Created 'high_revenue_outlier' column: {df['high_revenue_outlier'].sum()} True values")
print(f"6. Current null count: {df['revenue'].isnull().sum()}")
print(f"7. Data type: {df['revenue'].dtype}")
print(f"8. Valid range check: ${df['revenue'].min():,.0f} to ${df['revenue'].max():,.0f}")

# 12. Post-cleaning statistics
print("\\n--- Statistics After Cleaning ---")
print(f"Min: ${df['revenue'].min():,.0f}")
print(f"Max: ${df['revenue'].max():,.0f}")
print(f"Mean: ${df['revenue'].mean():,.0f}")
print(f"Median: ${df['revenue'].median():,.0f}")
print(f"Std Dev: ${df['revenue'].std():,.0f}")

# 13. Revenue distribution analysis
print("\\n--- Revenue Distribution Analysis ---")
revenue_dist = df['revenue_category'].value_counts().sort_index()
print("Movies by revenue category:")
for category, count in revenue_dist.items():
    percentage = (count / len(df)) * 100
    print(f"  {category:10s}: {count:6d} movies ({percentage:5.1f}%)")

# 14. Top revenue movies
print("\\n--- Top 10 Highest Revenue Movies ---")
top_revenue = df[['title', 'revenue', 'budget', 'release_year']].sort_values('revenue', ascending=False).head(10)
top_revenue['title_truncated'] = top_revenue['title'].apply(lambda x: x[:30] + '...' if len(x) > 30 else x)
top_revenue['revenue_millions'] = top_revenue['revenue'] / 1000000
top_revenue['budget_millions'] = top_revenue['budget'] / 1000000
print(top_revenue[['title_truncated', 'release_year', 'revenue_millions', 'budget_millions']].to_string(index=False, formatters={
    'revenue_millions': '{:,.0f}M'.format,
    'budget_millions': '{:,.0f}M'.format
}))

# 15. Check for any remaining negative values (should be 0 with int64)
remaining_negatives = df[df['revenue'] < 0].shape[0]
print(f"\\nRemaining negative revenue values: {remaining_negatives}")

# 16. Sample after cleaning
print("\\n--- Sample after cleaning ---")
print("First 10 rows with revenue and categories:")
sample_cols = ['title', 'revenue', 'revenue_category', 'zero_revenue', 'missing_revenue']
sample_df = df[sample_cols].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:25] + '...' if len(x) > 25 else x)
sample_df['revenue_formatted'] = sample_df['revenue'].apply(lambda x: f"${x:,.0f}")
print(sample_df[['title_truncated', 'revenue_formatted', 'revenue_category', 'zero_revenue', 'missing_revenue']].to_string(index=False))

revenue_clean_time = time.time() - revenue_clean_start
print(f"\\nrevenue column cleaning completed in {revenue_clean_time:.2f} seconds")

Cleaning revenue column...
Current data type of 'revenue' column: float64
\n--- Basic Statistics Before Cleaning ---
Min: $0
Max: $2,923,706,026
Mean: $1,373,790
Median: $0
Std Dev: $23,594,226
Null count: 0
\n--- Revenue Distribution Percentiles ---
  0th percentile: $0
 25th percentile: $0
 50th percentile: $0
 75th percentile: $0
 90th percentile: $0
 95th percentile: $0
 99th percentile: $18,034,054
100th percentile: $2,923,706,026
\nCreated 'missing_revenue' column: 0 rows marked as True
\n--- Checking for Invalid Values ---
Negative revenue values: 0
\nDecimal revenue values: 5
Sample of decimal revenue:
                   title       revenue
                 Titanic 2264162353.00
                  Avatar 2923706026.00
Avatar: The Way of Water 2353096253.00
       Avengers: Endgame 2799439100.00
                Ne Zha 2 2259822417.00
\nPotential high outliers (above $0): 24048
\nMovies with $0 revenue: 566154 (95.93%)
\nSample of movies with $0 revenue:
                   title  

In [17]:
# Clean budget Column
print("Cleaning budget column...")
budget_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'budget' column: {df['budget'].dtype}")
print(f"\\n--- Basic Statistics Before Cleaning ---")
print(f"Min: ${df['budget'].min():,.0f}")
print(f"Max: ${df['budget'].max():,.0f}")
print(f"Mean: ${df['budget'].mean():,.0f}")
print(f"Median: ${df['budget'].median():,.0f}")
print(f"Std Dev: ${df['budget'].std():,.0f}")
print(f"Null count: {df['budget'].isnull().sum()}")

# 2. Show distribution percentiles
print(f"\\n--- Budget Distribution Percentiles ---")
percentiles = [0, 25, 50, 75, 90, 95, 99, 100]
for p in percentiles:
    value = df['budget'].quantile(p/100)
    print(f"{p:3d}th percentile: ${value:,.0f}")

# 3. Create a column to mark missing budget before cleaning
df['missing_budget'] = df['budget'].isnull()
missing_count = df['missing_budget'].sum()
print(f"\\nCreated 'missing_budget' column: {missing_count} rows marked as True")

# 4. Check for invalid values
print(f"\\n--- Checking for Invalid Values ---")

# Check for negative budget (shouldn't exist)
negative_budget = df[df['budget'] < 0].shape[0]
print(f"Negative budget values: {negative_budget}")

if negative_budget > 0:
    print("\\nSample of negative budget:")
    negative_samples = df[df['budget'] < 0][['title', 'budget']].head(5)
    print(negative_samples.to_string(index=False))

# Check for decimal values
decimal_budget = df[df['budget'] != df['budget'].astype(int)].shape[0]
print(f"\\nDecimal budget values: {decimal_budget}")

if decimal_budget > 0:
    print("Sample of decimal budget:")
    decimal_samples = df[df['budget'] != df['budget'].astype(int)][['title', 'budget']].head(5)
    print(decimal_samples.to_string(index=False))

# Check for unreasonably high values
budget_q75 = df['budget'].quantile(0.75)
budget_q99 = df['budget'].quantile(0.99)
iqr = df['budget'].quantile(0.75) - df['budget'].quantile(0.25)
outlier_threshold = df['budget'].quantile(0.75) + 1.5 * iqr

high_outliers = df[df['budget'] > outlier_threshold].shape[0]
print(f"\\nPotential high outliers (above ${outlier_threshold:,.0f}): {high_outliers}")

# 5. Check for zero budget
zero_budget = (df['budget'] == 0).sum()
print(f"\\nMovies with $0 budget: {zero_budget} ({zero_budget/len(df)*100:.2f}%)")

# Show some samples of high budget vs zero budget
if zero_budget > 0:
    print("\\nSample of movies with $0 budget:")
    zero_samples = df[df['budget'] == 0][['title', 'budget', 'revenue']].head(5)
    print(zero_samples.to_string(index=False))

# 6. Handle missing values (impute with median)
budget_median = df['budget'].median()
print(f"\\nMedian budget for imputation: ${budget_median:,.0f}")

df['budget'] = df['budget'].fillna(budget_median)
print(f"Null budget after imputation: {df['budget'].isnull().sum()}")

# 7. Ensure proper data type (int64 for currency - to avoid overflow)
df['budget'] = df['budget'].astype('int64')
print(f"Data type after conversion: {df['budget'].dtype}")

# 8. Clip negative values to 0 if needed
if negative_budget > 0:
    df['budget'] = df['budget'].clip(lower=0)
    print(f"\\nClipped {negative_budget} negative values to 0")

# 9. Create categories for budget analysis
print(f"\\n--- Creating Budget Categories ---")

# Define budget brackets (similar to revenue but adjusted for typical movie budgets)
bins = [-1, 0, 10000, 100000, 1000000, 10000000, 50000000, 100000000, 200000000, float('inf')]
labels = ['$0', '$1-10K', '$10K-100K', '$100K-1M', '$1M-10M', '$10M-50M', '$50M-100M', '$100M-200M', '$200M+']

df['budget_category'] = pd.cut(df['budget'], bins=bins, labels=labels)
print(f"Created 'budget_category' column with {len(labels)} categories")

# 10. Create quality flags
df['zero_budget'] = (df['budget'] == 0)
df['low_budget'] = (df['budget'] > 0) & (df['budget'] < 10000)  # Less than $10K
df['high_budget_outlier'] = df['budget'] > outlier_threshold

print(f"\\nCreated quality flags:")
print(f"  - 'zero_budget': {df['zero_budget'].sum()} rows ({df['zero_budget'].sum()/len(df)*100:.1f}%)")
print(f"  - 'low_budget' (<$10K): {df['low_budget'].sum()} rows ({df['low_budget'].sum()/len(df)*100:.1f}%)")
print(f"  - 'high_budget_outlier': {df['high_budget_outlier'].sum()} rows")

# 11. Analyze relationship with revenue
print(f"\\n--- Budget-Revenue Relationship ---")

# Only look at movies with both budget and revenue > 0
movies_with_financials = df[(df['budget'] > 0) & (df['revenue'] > 0)]
financial_count = len(movies_with_financials)
print(f"Movies with both budget > 0 and revenue > 0: {financial_count} ({financial_count/len(df)*100:.1f}%)")

if financial_count > 0:
    # Calculate ROI
    roi_values = (movies_with_financials['revenue'] - movies_with_financials['budget']) / movies_with_financials['budget']
    print(f"\\nROI Statistics for movies with financial data:")
    print(f"  Average ROI: {roi_values.mean():.2f}x return")
    print(f"  Median ROI: {roi_values.median():.2f}x return")
    print(f"  Max ROI: {roi_values.max():.2f}x return")
    print(f"  Min ROI: {roi_values.min():.2f}x return")
    
    # Correlation
    correlation = df['budget'].corr(df['revenue'])
    print(f"\\nCorrelation between budget and revenue (all movies): {correlation:.3f}")
    
    if financial_count > 1:
        financial_correlation = movies_with_financials['budget'].corr(movies_with_financials['revenue'])
        print(f"Correlation between budget and revenue (financial movies only): {financial_correlation:.3f}")

# 12. Verify the cleaning
print("\\n--- budget Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count}")
print(f"2. Created 'missing_budget' column: {missing_count} True values")
print(f"3. Created 'zero_budget' column: {df['zero_budget'].sum()} True values")
print(f"4. Created 'low_budget' column: {df['low_budget'].sum()} True values")
print(f"5. Created 'high_budget_outlier' column: {df['high_budget_outlier'].sum()} True values")
print(f"6. Current null count: {df['budget'].isnull().sum()}")
print(f"7. Data type: {df['budget'].dtype}")
print(f"8. Valid range check: ${df['budget'].min():,.0f} to ${df['budget'].max():,.0f}")

# 13. Post-cleaning statistics
print("\\n--- Statistics After Cleaning ---")
print(f"Min: ${df['budget'].min():,.0f}")
print(f"Max: ${df['budget'].max():,.0f}")
print(f"Mean: ${df['budget'].mean():,.0f}")
print(f"Median: ${df['budget'].median():,.0f}")
print(f"Std Dev: ${df['budget'].std():,.0f}")

# 14. Budget distribution analysis
print("\\n--- Budget Distribution Analysis ---")
budget_dist = df['budget_category'].value_counts().sort_index()
print("Movies by budget category:")
for category, count in budget_dist.items():
    percentage = (count / len(df)) * 100
    print(f"  {category:12s}: {count:6d} movies ({percentage:5.1f}%)")

# 15. Top budget movies
print("\\n--- Top 10 Highest Budget Movies ---")
top_budget = df[['title', 'budget', 'revenue', 'release_year']].sort_values('budget', ascending=False).head(10)
top_budget['title_truncated'] = top_budget['title'].apply(lambda x: x[:30] + '...' if len(x) > 30 else x)
top_budget['budget_millions'] = top_budget['budget'] / 1000000
top_budget['revenue_millions'] = top_budget['revenue'] / 1000000
print(top_budget[['title_truncated', 'release_year', 'budget_millions', 'revenue_millions']].to_string(index=False, formatters={
    'budget_millions': '{:,.0f}M'.format,
    'revenue_millions': '{:,.0f}M'.format
}))

# 16. Compare with revenue patterns
print(f"\\n--- Comparison with Revenue Data ---")
print(f"Movies with $0 revenue: {(df['revenue'] == 0).sum()} ({(df['revenue'] == 0).sum()/len(df)*100:.1f}%)")
print(f"Movies with $0 budget: {zero_budget} ({zero_budget/len(df)*100:.1f}%)")

both_zero = df[(df['budget'] == 0) & (df['revenue'] == 0)].shape[0]
print(f"Movies with $0 budget AND $0 revenue: {both_zero} ({both_zero/len(df)*100:.1f}%)")

# 17. Sample after cleaning
print("\\n--- Sample after cleaning ---")
print("First 10 rows with budget and categories:")
sample_cols = ['title', 'budget', 'budget_category', 'revenue', 'revenue_category', 'zero_budget', 'zero_revenue']
sample_df = df[sample_cols].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
sample_df['budget_formatted'] = sample_df['budget'].apply(lambda x: f"${x:,.0f}")
sample_df['revenue_formatted'] = sample_df['revenue'].apply(lambda x: f"${x:,.0f}")
print(sample_df[['title_truncated', 'budget_formatted', 'budget_category', 'revenue_formatted', 'revenue_category', 'zero_budget', 'zero_revenue']].to_string(index=False))

budget_clean_time = time.time() - budget_clean_start
print(f"\\nbudget column cleaning completed in {budget_clean_time:.2f} seconds")

Cleaning budget column...
Current data type of 'budget' column: float64
\n--- Basic Statistics Before Cleaning ---
Min: $0
Max: $583,900,000
Mean: $535,350
Median: $0
Std Dev: $6,434,089
Null count: 0
\n--- Budget Distribution Percentiles ---
  0th percentile: $0
 25th percentile: $0
 50th percentile: $0
 75th percentile: $0
 90th percentile: $0
 95th percentile: $10,000
 99th percentile: $12,000,000
100th percentile: $583,900,000
\nCreated 'missing_budget' column: 0 rows marked as True
\n--- Checking for Invalid Values ---
Negative budget values: 0
\nDecimal budget values: 0
\nPotential high outliers (above $0): 56486
\nMovies with $0 budget: 533716 (90.43%)
\nSample of movies with $0 budget:
              title  budget  revenue
              Ariel    0.00        0
Shadows in Paradise    0.00        0
   Sunday in August    0.00        0
           The Dark    0.00  6593579
 Land Without Bread    0.00        0
\nMedian budget for imputation: $0
Null budget after imputation: 0
Data typ

In [18]:
# Clean runtime Column
print("Cleaning runtime column...")
runtime_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'runtime' column: {df['runtime'].dtype}")
print(f"\\n--- Basic Statistics Before Cleaning ---")
print(f"Min: {df['runtime'].min():.0f} minutes")
print(f"Max: {df['runtime'].max():.0f} minutes")
print(f"Mean: {df['runtime'].mean():.1f} minutes")
print(f"Median: {df['runtime'].median():.1f} minutes")
print(f"Std Dev: {df['runtime'].std():.1f} minutes")
print(f"Null count: {df['runtime'].isnull().sum()}")

# 2. Show distribution percentiles
print(f"\\n--- Runtime Distribution Percentiles ---")
percentiles = [0, 25, 50, 75, 90, 95, 99, 100]
for p in percentiles:
    value = df['runtime'].quantile(p/100)
    print(f"{p:3d}th percentile: {value:.0f} minutes")

# 3. Create a column to mark missing runtime before cleaning
df['missing_runtime'] = df['runtime'].isnull()
missing_count = df['missing_runtime'].sum()
print(f"\\nCreated 'missing_runtime' column: {missing_count} rows marked as True")

# 4. Check for invalid values
print(f"\\n--- Checking for Invalid Values ---")

# Check for negative runtime (impossible)
negative_runtime = df[df['runtime'] < 0].shape[0]
print(f"Negative runtime values: {negative_runtime}")

if negative_runtime > 0:
    print("\\nSample of negative runtime:")
    negative_samples = df[df['runtime'] < 0][['title', 'runtime']].head(5)
    print(negative_samples.to_string(index=False))

# Check for zero runtime (should be rare - maybe short films or data errors)
zero_runtime = (df['runtime'] == 0).sum()
print(f"\\nMovies with 0 minute runtime: {zero_runtime} ({zero_runtime/len(df)*100:.2f}%)")

if zero_runtime > 0:
    print("Sample of movies with 0 minute runtime:")
    zero_samples = df[df['runtime'] == 0][['title', 'runtime', 'genres']].head(10)
    print(zero_samples[['title', 'runtime']].to_string(index=False))

# Check for unreasonably long runtime
# Typical movie: 90-180 minutes, max reasonable: ~4 hours (240 min)
unreasonably_long = df[df['runtime'] > 240].shape[0]
print(f"\\nMovies longer than 4 hours (>240 min): {unreasonably_long}")

if unreasonably_long > 0:
    print("Sample of very long movies (>240 min):")
    long_samples = df[df['runtime'] > 240][['title', 'runtime']].sort_values('runtime', ascending=False).head(10)
    print(long_samples.to_string(index=False))

# Check for very short "movies" (less than 1 minute)
very_short = df[(df['runtime'] > 0) & (df['runtime'] < 1)].shape[0]
print(f"\\nMovies shorter than 1 minute: {very_short}")

# 5. Handle missing values (impute with median)
runtime_median = df['runtime'].median()
print(f"\\nMedian runtime for imputation: {runtime_median:.1f} minutes")

df['runtime'] = df['runtime'].fillna(runtime_median)
print(f"Null runtime after imputation: {df['runtime'].isnull().sum()}")

# 6. Ensure proper data type (int for minutes)
df['runtime'] = df['runtime'].astype(int)
print(f"Data type after conversion: {df['runtime'].dtype}")

# 7. Clip negative values to 0 if needed
if negative_runtime > 0:
    df['runtime'] = df['runtime'].clip(lower=0)
    print(f"\\nClipped {negative_runtime} negative values to 0")

# 8. Create categories for runtime analysis
print(f"\\n--- Creating Runtime Categories ---")

# Define runtime brackets (standard movie lengths)
bins = [-1, 0, 1, 10, 30, 60, 90, 120, 150, 180, 240, float('inf')]
labels = ['0 min', '1 min', '2-10 min', '11-30 min', '31-60 min', '61-90 min', '91-120 min', '121-150 min', '151-180 min', '181-240 min', '240+ min']

df['runtime_category'] = pd.cut(df['runtime'], bins=bins, labels=labels)
print(f"Created 'runtime_category' column with {len(labels)} categories")

# 9. Create quality flags
df['zero_runtime'] = (df['runtime'] == 0)
df['short_film'] = (df['runtime'] > 0) & (df['runtime'] <= 30)  # Short films <= 30 min
df['feature_film'] = (df['runtime'] >= 60) & (df['runtime'] <= 180)  # Standard features
df['extended_film'] = df['runtime'] > 180  # Extended/very long films
df['unusual_runtime'] = (df['runtime'] > 240) | ((df['runtime'] > 0) & (df['runtime'] < 10))

print(f"\\nCreated quality flags:")
print(f"  - 'zero_runtime': {df['zero_runtime'].sum()} rows ({df['zero_runtime'].sum()/len(df)*100:.1f}%)")
print(f"  - 'short_film' (≤30 min): {df['short_film'].sum()} rows ({df['short_film'].sum()/len(df)*100:.1f}%)")
print(f"  - 'feature_film' (60-180 min): {df['feature_film'].sum()} rows ({df['feature_film'].sum()/len(df)*100:.1f}%)")
print(f"  - 'extended_film' (>180 min): {df['extended_film'].sum()} rows ({df['extended_film'].sum()/len(df)*100:.1f}%)")
print(f"  - 'unusual_runtime' (<10 min or >240 min): {df['unusual_runtime'].sum()} rows ({df['unusual_runtime'].sum()/len(df)*100:.1f}%)")

# 10. Analyze relationship with other variables
print(f"\\n--- Runtime Relationships ---")

# With genres (will analyze more when we clean genres)
print(f"Average runtime by feature film status:")
print(f"  Short films: {df[df['short_film']]['runtime'].mean():.1f} min")
print(f"  Feature films: {df[df['feature_film']]['runtime'].mean():.1f} min")
print(f"  Extended films: {df[df['extended_film']]['runtime'].mean():.1f} min")

# With revenue (only for movies with revenue)
if 'revenue' in df.columns and (df['revenue'] > 0).sum() > 0:
    revenue_movies = df[df['revenue'] > 0]
    if len(revenue_movies) > 0:
        correlation = revenue_movies['runtime'].corr(revenue_movies['revenue'])
        print(f"\\nCorrelation between runtime and revenue (movies with revenue): {correlation:.3f}")

# With vote average (only for movies with votes)
if 'vote_average' in df.columns and (df['vote_count'] > 0).sum() > 0:
    voted_movies = df[df['vote_count'] > 0]
    if len(voted_movies) > 0:
        correlation = voted_movies['runtime'].corr(voted_movies['vote_average'])
        print(f"Correlation between runtime and rating (movies with votes): {correlation:.3f}")

# 11. Verify the cleaning
print("\\n--- runtime Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count}")
print(f"2. Created 'missing_runtime' column: {missing_count} True values")
print(f"3. Created 'zero_runtime' column: {df['zero_runtime'].sum()} True values")
print(f"4. Created 'short_film' column: {df['short_film'].sum()} True values")
print(f"5. Created 'feature_film' column: {df['feature_film'].sum()} True values")
print(f"6. Created 'extended_film' column: {df['extended_film'].sum()} True values")
print(f"7. Created 'unusual_runtime' column: {df['unusual_runtime'].sum()} True values")
print(f"8. Current null count: {df['runtime'].isnull().sum()}")
print(f"9. Data type: {df['runtime'].dtype}")
print(f"10. Valid range check: {df['runtime'].min()} to {df['runtime'].max()} minutes")

# 12. Post-cleaning statistics
print("\\n--- Statistics After Cleaning ---")
print(f"Min: {df['runtime'].min()} minutes")
print(f"Max: {df['runtime'].max()} minutes")
print(f"Mean: {df['runtime'].mean():.1f} minutes")
print(f"Median: {df['runtime'].median():.1f} minutes")
print(f"Std Dev: {df['runtime'].std():.1f} minutes")

# 13. Runtime distribution analysis
print("\\n--- Runtime Distribution Analysis ---")
runtime_dist = df['runtime_category'].value_counts().sort_index()
print("Movies by runtime category:")
for category, count in runtime_dist.items():
    percentage = (count / len(df)) * 100
    print(f"  {category:12s}: {count:6d} movies ({percentage:5.1f}%)")

# 14. Longest and shortest movies
print("\\n--- Top 10 Longest Movies ---")
longest = df[['title', 'runtime', 'release_year']].sort_values('runtime', ascending=False).head(10)
longest['title_truncated'] = longest['title'].apply(lambda x: x[:40] + '...' if len(x) > 40 else x)
longest['runtime_hours'] = longest['runtime'] / 60
print(longest[['title_truncated', 'release_year', 'runtime', 'runtime_hours']].to_string(index=False, formatters={
    'runtime_hours': '{:.1f} hours'.format
}))

print("\\n--- Top 10 Shortest Non-Zero Movies ---")
shortest_nonzero = df[df['runtime'] > 0][['title', 'runtime', 'release_year']].sort_values('runtime').head(10)
shortest_nonzero['title_truncated'] = shortest_nonzero['title'].apply(lambda x: x[:40] + '...' if len(x) > 40 else x)
print(shortest_nonzero[['title_truncated', 'release_year', 'runtime']].to_string(index=False))

# 15. Sample after cleaning
print("\\n--- Sample after cleaning ---")
print("First 10 rows with runtime and categories:")
sample_cols = ['title', 'runtime', 'runtime_category', 'feature_film', 'short_film', 'zero_runtime']
sample_df = df[sample_cols].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:25] + '...' if len(x) > 25 else x)
print(sample_df[['title_truncated', 'runtime', 'runtime_category', 'feature_film', 'short_film', 'zero_runtime']].to_string(index=False))

runtime_clean_time = time.time() - runtime_clean_start
print(f"\\nruntime column cleaning completed in {runtime_clean_time:.2f} seconds")

Cleaning runtime column...
Current data type of 'runtime' column: float64
\n--- Basic Statistics Before Cleaning ---
Min: 0 minutes
Max: 13319 minutes
Mean: 59.3 minutes
Median: 69.0 minutes
Std Dev: 53.4 minutes
Null count: 0
\n--- Runtime Distribution Percentiles ---
  0th percentile: 0 minutes
 25th percentile: 11 minutes
 50th percentile: 69 minutes
 75th percentile: 93 minutes
 90th percentile: 110 minutes
 95th percentile: 127 minutes
 99th percentile: 168 minutes
100th percentile: 13319 minutes
\nCreated 'missing_runtime' column: 0 rows marked as True
\n--- Checking for Invalid Values ---
Negative runtime values: 0
\nMovies with 0 minute runtime: 85904 (14.56%)
Sample of movies with 0 minute runtime:
                      title  runtime
               Dos billetes     0.00
        Che notte, ragazzi!     0.00
Das Geheimnis von St. Pauli     0.00
           Deadly Obsession     0.00
                       Evil     0.00
               Nur ein Kuss     0.00
           Mohini Bhasma

In [19]:
# Clean imdb_id Column - CORRECTED VERSION
print("Cleaning imdb_id column...")
imdb_id_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'imdb_id' column: {df['imdb_id'].dtype}")
print(f"\\n--- Basic Statistics ---")
print(f"Total rows: {len(df):,}")
print(f"Non-null imdb_id count: {df['imdb_id'].notnull().sum():,}")
print(f"Null imdb_id count: {df['imdb_id'].isnull().sum():,}")
print(f"Percentage null: {df['imdb_id'].isnull().sum()/len(df)*100:.2f}%")

# 2. Show sample values
print(f"\\n--- Sample Values (first 10 non-null) ---")
non_null_samples = df[df['imdb_id'].notnull()]['imdb_id'].head(10)
for i, imdb_id in enumerate(non_null_samples, 1):
    print(f"{i:2d}. {imdb_id}")

# 3. Create a column to mark missing imdb_id before cleaning
df['missing_imdb_id'] = df['imdb_id'].isnull()
missing_count = df['missing_imdb_id'].sum()
print(f"\\nCreated 'missing_imdb_id' column: {missing_count} rows marked as True ({missing_count/len(df)*100:.1f}%)")

# 4. Check for valid IMDB ID format - CORRECTED PATTERN
print(f"\\n--- Checking IMDB ID Format ---")
# Valid IMDB ID format: starts with 'tt' followed by digits (usually 7-9 digits)
def is_valid_imdb_id_fixed(imdb_id):
    """
    Correctly validates IMDB IDs like 'tt0094675', 'tt0076759', etc.
    IMDB IDs: 'tt' followed by 7-8 digits (but be flexible)
    """
    if pd.isna(imdb_id):
        return False
    
    imdb_str = str(imdb_id).strip()
    
    # Check if it's a placeholder (from previous cleaning)
    if imdb_str.startswith('NO_IMDB_ID'):
        return False
    
    # Correct validation: starts with 'tt' and has only digits after
    # Use a simple check instead of complex regex
    if not imdb_str.startswith('tt'):
        return False
    
    # Check that characters after 'tt' are all digits
    number_part = imdb_str[2:]
    if not number_part.isdigit():
        return False
    
    # Optional: Check for reasonable length (IMDB IDs are usually 7-9 digits total)
    if len(imdb_str) < 9 or len(imdb_str) > 11:  # 'tt' + 7-9 digits
        # But be lenient - some might be different
        pass
    
    return True
valid_count = df['imdb_id'].apply(is_valid_imdb_id_fixed).sum()
invalid_count = df['imdb_id'].notnull().sum() - valid_count
print(f"Valid IMDB ID format (tt + digits): {valid_count:,}")
print(f"Invalid IMDB ID format: {invalid_count:,}")

if invalid_count > 0:
    print("\\nSample of invalid IMDB IDs:")
    invalid_samples = df[~df['imdb_id'].apply(is_valid_imdb_id_fixed) & df['imdb_id'].notnull()]['imdb_id'].unique()[:5]
    for sample in invalid_samples:
        print(f"  '{sample}'")

# 5. Check for duplicates PROPERLY (excluding nulls)
print(f"\\n--- Checking for Real Duplicates (excluding nulls) ---")
# First, get only non-null IMDB IDs
non_null_imdb = df[df['imdb_id'].notnull()]['imdb_id']
real_duplicate_count = non_null_imdb.duplicated().sum()
unique_count = non_null_imdb.nunique()

print(f"Unique IMDB IDs (non-null): {unique_count:,}")
print(f"Duplicate IMDB IDs (non-null): {real_duplicate_count:,}")

if real_duplicate_count > 0:
    print("\\nSample of duplicate IMDB IDs:")
    duplicates = df[df['imdb_id'].notnull() & df['imdb_id'].duplicated(keep=False)]
    duplicate_samples = duplicates['imdb_id'].value_counts().head(5)
    
    for imdb_id, count in duplicate_samples.items():
        print(f"  {imdb_id}: {count} occurrences")
        # Show titles for one duplicate
        titles = duplicates[duplicates['imdb_id'] == imdb_id]['title'].head(2).tolist()
        for title in titles:
            print(f"      - {title[:50]}{'...' if len(title) > 50 else ''}")

# 6. Check relationship with other data completeness
print(f"\\n--- Data Completeness Relationships ---")

# With revenue
if 'revenue' in df.columns:
    imdb_with_revenue = df[df['imdb_id'].notnull() & (df['revenue'] > 0)].shape[0]
    print(f"Movies with IMDB ID AND revenue > 0: {imdb_with_revenue:,} ({imdb_with_revenue/len(df)*100:.1f}%)")

# With votes
if 'vote_count' in df.columns:
    imdb_with_votes = df[df['imdb_id'].notnull() & (df['vote_count'] > 0)].shape[0]
    print(f"Movies with IMDB ID AND votes > 0: {imdb_with_votes:,} ({imdb_with_votes/len(df)*100:.1f}%)")

# 7. Handle missing values WITHOUT creating duplicates
print(f"\\n--- Handling Missing Values (No Duplicate Creation) ---")

# Save original before modification
df['imdb_id_original'] = df['imdb_id'].copy()

# Create cleaned version with unique placeholders to avoid duplicate counts
df['imdb_id_clean'] = df['imdb_id'].copy()

# Fill nulls with unique identifiers (prevents duplicate counting)
null_indices = df[df['imdb_id_clean'].isnull()].index
for i, idx in enumerate(null_indices):
    df.at[idx, 'imdb_id_clean'] = f'NO_IMDB_ID_{i:08d}'

print(f"Filled {len(null_indices):,} null values with unique placeholders")
print(f"Example placeholder: NO_IMDB_ID_00000000")

# 8. Ensure proper data type (string)
df['imdb_id_clean'] = df['imdb_id_clean'].astype(str)
print(f"Data type after conversion: {df['imdb_id_clean'].dtype}")

# 9. Create quality flags using CORRECTED validation
df['valid_imdb_id'] = df['imdb_id_original'].apply(is_valid_imdb_id_fixed)
df['has_imdb_id'] = df['imdb_id_original'].notnull()

print(f"\\nCreated quality flags:")
print(f"  - 'valid_imdb_id': {df['valid_imdb_id'].sum():,} rows ({df['valid_imdb_id'].sum()/len(df)*100:.1f}%)")
print(f"  - 'has_imdb_id': {df['has_imdb_id'].sum():,} rows ({df['has_imdb_id'].sum()/len(df)*100:.1f}%)")

# 10. Extract numeric ID for potential analysis
print(f"\\n--- Extracting IMDB ID Components ---")
def extract_imdb_number(imdb_id):
    if pd.isna(imdb_id) or not is_valid_imdb_id_fixed(imdb_id):
        return None
    try:
        # Extract digits after 'tt'
        return int(imdb_id[2:])
    except:
        return None

df['imdb_id_numeric'] = df['imdb_id_original'].apply(extract_imdb_number)
valid_numeric_count = df['imdb_id_numeric'].notnull().sum()
print(f"Extracted numeric ID from {valid_numeric_count:,} valid IMDB IDs")

# 11. Verify the cleaning
print("\\n--- imdb_id Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count:,} ({missing_count/len(df)*100:.1f}%)")
print(f"2. Created 'missing_imdb_id' column: {missing_count:,} True values")
print(f"3. Created 'valid_imdb_id' column: {df['valid_imdb_id'].sum():,} True values")
print(f"4. Created 'has_imdb_id' column: {df['has_imdb_id'].sum():,} True values")
print(f"5. Cleaned column null count: {df['imdb_id_clean'].isnull().sum()}")
print(f"6. Cleaned column data type: {df['imdb_id_clean'].dtype}")
print(f"7. Valid IMDB IDs: {valid_count:,} ({valid_count/len(df)*100:.1f}%)")
print(f"8. Real duplicate IDs (non-null): {real_duplicate_count:,}")

# 12. Replace original column with cleaned version
df['imdb_id'] = df['imdb_id_clean']
# Drop the temporary columns if desired
# df = df.drop(columns=['imdb_id_original', 'imdb_id_clean'])

# 13. Sample after cleaning
print("\\n--- Sample after cleaning ---")
print("First 15 rows with imdb_id and quality flags:")
sample_cols = ['title', 'imdb_id', 'valid_imdb_id', 'has_imdb_id', 'missing_imdb_id']
sample_df = df[sample_cols].head(15).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:25] + '...' if len(x) > 25 else x)
sample_df['imdb_id_truncated'] = sample_df['imdb_id'].apply(lambda x: x[:15] + '...' if len(x) > 15 else x)

# Reorder columns for better display
display_cols = ['title_truncated', 'imdb_id_truncated', 'valid_imdb_id', 'has_imdb_id', 'missing_imdb_id']
print(sample_df[display_cols].to_string(index=False))

# 14. Distribution of IMDB ID presence
print("\\n--- IMDB ID Distribution Analysis ---")
print(f"Movies with valid IMDB ID: {valid_count:,} ({valid_count/len(df)*100:.1f}%)")
print(f"Movies with any IMDB ID (including invalid): {df['has_imdb_id'].sum():,} ({df['has_imdb_id'].sum()/len(df)*100:.1f}%)")
print(f"Movies without IMDB ID: {missing_count:,} ({missing_count/len(df)*100:.1f}%)")

imdb_id_clean_time = time.time() - imdb_id_clean_start
print(f"\\nimdb_id column cleaning completed in {imdb_id_clean_time:.2f} seconds")

Cleaning imdb_id column...
Current data type of 'imdb_id' column: object
\n--- Basic Statistics ---
Total rows: 590,202
Non-null imdb_id count: 450,448
Null imdb_id count: 139,754
Percentage null: 23.68%
\n--- Sample Values (first 10 non-null) ---
 1. tt0094675
 2. tt0092149
 3. tt0113101
 4. tt0107286
 5. tt0425473
 6. tt0076759
 7. tt0266543
 8. tt0109830
 9. tt0169547
10. tt0033467
\nCreated 'missing_imdb_id' column: 139754 rows marked as True (23.7%)
\n--- Checking IMDB ID Format ---
Valid IMDB ID format (tt + digits): 450,448
Invalid IMDB ID format: 0
\n--- Checking for Real Duplicates (excluding nulls) ---
Unique IMDB IDs (non-null): 450,448
Duplicate IMDB IDs (non-null): 0
\n--- Data Completeness Relationships ---
Movies with IMDB ID AND revenue > 0: 21,915 (3.7%)
Movies with IMDB ID AND votes > 0: 274,243 (46.5%)
\n--- Handling Missing Values (No Duplicate Creation) ---
Filled 139,754 null values with unique placeholders
Example placeholder: NO_IMDB_ID_00000000
Data type after 

In [20]:
# Clean original_language Column
print("Cleaning original_language column...")
language_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'original_language' column: {df['original_language'].dtype}")
print(f"\\n--- Basic Statistics ---")
print(f"Total rows: {len(df):,}")
print(f"Non-null count: {df['original_language'].notnull().sum():,}")
print(f"Null count: {df['original_language'].isnull().sum():,}")
print(f"Percentage null: {df['original_language'].isnull().sum()/len(df)*100:.2f}%")

# 2. Show sample values
print(f"\\n--- Sample Values (first 20) ---")
samples = df['original_language'].head(20).tolist()
for i, lang in enumerate(samples, 1):
    print(f"{i:2d}. '{lang}'")

# 3. Create a column to mark missing language before cleaning
df['missing_original_language'] = df['original_language'].isnull()
missing_count = df['missing_original_language'].sum()
print(f"\\nCreated 'missing_original_language' column: {missing_count} rows marked as True ({missing_count/len(df)*100:.1f}%)")

# 4. Check for unique values and distribution
print(f"\\n--- Language Distribution Analysis ---")
unique_languages = df['original_language'].nunique()
print(f"Unique language codes: {unique_languages}")

# Get top languages
language_counts = df['original_language'].value_counts()
print(f"\\nTop 20 most common languages:")
print(language_counts.head(20).to_string())

# Get languages with very few movies
rare_languages = language_counts[language_counts < 10]
print(f"\\nLanguages with less than 10 movies: {len(rare_languages)} languages")
if len(rare_languages) > 0:
    print("Sample of rare languages:")
    print(rare_languages.head(10).to_string())

# 5. Check for invalid language codes
print(f"\\n--- Checking for Invalid Language Codes ---")
# ISO 639-1 language codes should be 2 letters (most common)
def is_valid_language_code(lang):
    if pd.isna(lang):
        return False
    lang_str = str(lang).strip()
    # Should be 2-3 characters (some might be 'zh-CN' style)
    if len(lang_str) < 2 or len(lang_str) > 10:
        return False
    # Should be alphabetic (though some have hyphens like 'zh-CN')
    if not lang_str.replace('-', '').isalpha():
        return False
    return True

valid_lang_mask = df['original_language'].apply(is_valid_language_code)
valid_count = valid_lang_mask.sum()
invalid_count = len(df) - valid_count - missing_count  # Exclude nulls from invalid count

print(f"Valid language codes: {valid_count:,} ({valid_count/len(df)*100:.1f}%)")
print(f"Invalid language codes: {invalid_count:,} ({invalid_count/len(df)*100:.1f}%)")

if invalid_count > 0:
    print("\\nSample of invalid language codes:")
    invalid_samples = df[~valid_lang_mask & df['original_language'].notnull()]['original_language'].unique()[:10]
    for sample in invalid_samples:
        count = (df['original_language'] == sample).sum()
        print(f"  '{sample}': {count:,} occurrences")

# 6. Handle missing values
print(f"\\n--- Handling Missing Values ---")

# Check most common language for imputation
most_common_lang = df['original_language'].mode()
if not most_common_lang.empty:
    default_lang = most_common_lang.iloc[0]
    print(f"Most common language: '{default_lang}'")
    
    # Fill missing values with most common language
    df['original_language'] = df['original_language'].fillna(default_lang)
    print(f"Filled {missing_count:,} null values with '{default_lang}'")
else:
    print("No common language found for imputation")
    df['original_language'] = df['original_language'].fillna('unknown')

print(f"Null count after imputation: {df['original_language'].isnull().sum()}")

# 7. Standardize language codes (lowercase, strip whitespace)
print(f"\\n--- Standardizing Language Codes ---")
df['original_language'] = df['original_language'].astype(str).str.strip().str.lower()
print("Converted all to lowercase and trimmed whitespace")

# 8. Create categories for analysis
print(f"\\n--- Creating Language Categories ---")

# Identify major languages (appear in at least 1% of movies)
total_movies = len(df)
major_languages = language_counts[language_counts >= total_movies * 0.01].index.tolist()
print(f"Major languages (≥1% of movies): {len(major_languages)}")
print("Major languages:", major_languages[:15])  # Show first 15

# Create category: major language vs other
df['language_category'] = df['original_language'].apply(
    lambda x: x if x in major_languages else 'other'
)

print(f"Created 'language_category' column")
print(f"Movies in major languages: {(df['language_category'] != 'other').sum():,} ({(df['language_category'] != 'other').sum()/len(df)*100:.1f}%)")
print(f"Movies in 'other' category: {(df['language_category'] == 'other').sum():,} ({(df['language_category'] == 'other').sum()/len(df)*100:.1f}%)")

# 9. Create quality flags
df['valid_language_code'] = valid_lang_mask
df['rare_language'] = df['original_language'].isin(rare_languages.index)

print(f"\\nCreated quality flags:")
print(f"  - 'valid_language_code': {df['valid_language_code'].sum():,} rows ({df['valid_language_code'].sum()/len(df)*100:.1f}%)")
print(f"  - 'rare_language': {df['rare_language'].sum():,} rows ({df['rare_language'].sum()/len(df)*100:.1f}%)")

# 10. Analyze language trends
print(f"\\n--- Language Trends Analysis ---")

if 'release_year' in df.columns and df['release_year'].notnull().any():
    # Top languages by decade
    df['decade'] = (df['release_year'] // 10) * 10
    
    recent_decades = [1980, 1990, 2000, 2010, 2020]
    print(f"\\nTop languages by recent decades:")
    
    for decade in recent_decades:
        decade_movies = df[df['decade'] == decade]
        if len(decade_movies) > 0:
            top_langs = decade_movies['original_language'].value_counts().head(5)
            print(f"\\n{decade}s:")
            for lang, count in top_langs.items():
                pct = count / len(decade_movies) * 100
                print(f"  {lang}: {count:,} movies ({pct:.1f}%)")

# 11. Verify the cleaning
print("\\n--- original_language Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count:,} ({missing_count/len(df)*100:.1f}%)")
print(f"2. Created 'missing_original_language' column: {missing_count:,} True values")
print(f"3. Created 'valid_language_code' column: {valid_count:,} True values")
print(f"4. Created 'rare_language' column: {df['rare_language'].sum():,} True values")
print(f"5. Created 'language_category' column with {len(major_languages) + 1} categories")
print(f"6. Current null count: {df['original_language'].isnull().sum()}")
print(f"7. Data type: {df['original_language'].dtype}")
print(f"8. Unique languages: {df['original_language'].nunique()}")

# 12. Post-cleaning distribution
print("\\n--- Post-Cleaning Language Distribution ---")
clean_counts = df['original_language'].value_counts().head(10)
print("Top 10 languages after cleaning:")
for lang, count in clean_counts.items():
    pct = count / len(df) * 100
    print(f"  {lang}: {count:,} movies ({pct:.1f}%)")

# 13. Sample after cleaning
print("\\n--- Sample after cleaning ---")
print("First 15 rows with language data:")
sample_cols = ['title', 'original_language', 'language_category', 'valid_language_code', 'rare_language']
sample_df = df[sample_cols].head(15).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
print(sample_df[['title_truncated', 'original_language', 'language_category', 'valid_language_code', 'rare_language']].to_string(index=False))

# 14. Check relationship with other variables
print(f"\\n--- Language Relationships ---")

# With revenue
if 'revenue' in df.columns and (df['revenue'] > 0).any():
    top_langs_revenue = df[df['revenue'] > 0].groupby('original_language')['revenue'].mean().sort_values(ascending=False).head(5)
    print(f"\\nTop languages by average revenue (movies with revenue > 0):")
    for lang, avg_revenue in top_langs_revenue.items():
        print(f"  {lang}: ${avg_revenue:,.0f}")

# With ratings
if 'vote_average' in df.columns and (df['vote_count'] > 0).any():
    rated_movies = df[df['vote_count'] > 0]
    top_langs_rating = rated_movies.groupby('original_language')['vote_average'].mean().sort_values(ascending=False).head(5)
    print(f"\\nTop languages by average rating (movies with votes > 0):")
    for lang, avg_rating in top_langs_rating.items():
        print(f"  {lang}: {avg_rating:.2f}")

language_clean_time = time.time() - language_clean_start
print(f"\\noriginal_language column cleaning completed in {language_clean_time:.2f} seconds")

Cleaning original_language column...
Current data type of 'original_language' column: object
\n--- Basic Statistics ---
Total rows: 590,202
Non-null count: 590,202
Null count: 0
Percentage null: 0.00%
\n--- Sample Values (first 20) ---
 1. 'fi'
 2. 'fi'
 3. 'en'
 4. 'en'
 5. 'de'
 6. 'en'
 7. 'en'
 8. 'en'
 9. 'en'
10. 'en'
11. 'en'
12. 'en'
13. 'fr'
14. 'de'
15. 'en'
16. 'en'
17. 'en'
18. 'en'
19. 'en'
20. 'he'
\nCreated 'missing_original_language' column: 0 rows marked as True (0.0%)
\n--- Language Distribution Analysis ---
Unique language codes: 169
\nTop 20 most common languages:
original_language
en    263734
fr     36863
es     36458
de     31103
ja     24310
pt     18656
ru     16930
it     16271
zh     14323
ko      8707
cs      7291
hi      6422
tr      6324
sv      6255
tl      6109
ar      5898
id      4925
cn      4732
nl      4617
ta      4272
\nLanguages with less than 10 movies: 50 languages
Sample of rare languages:
original_language
ug    9
ba    9
sm    9
ln    9
eo  

In [21]:
# Create Language Code to Name Mapping
print("Creating language code to name mapping...")
mapping_start = time.time()

# Comprehensive ISO 639-1 and common language code mapping
language_mapping = {
    # Major languages (ISO 639-1)
    'en': 'English',
    'fr': 'French',
    'de': 'German',
    'es': 'Spanish',
    'it': 'Italian',
    'pt': 'Portuguese',
    'ru': 'Russian',
    'ja': 'Japanese',
    'ko': 'Korean',
    'zh': 'Chinese',
    'hi': 'Hindi',
    'ar': 'Arabic',
    'tr': 'Turkish',
    'pl': 'Polish',
    'nl': 'Dutch',
    'sv': 'Swedish',
    'da': 'Danish',
    'no': 'Norwegian',
    'fi': 'Finnish',
    'el': 'Greek',
    'he': 'Hebrew',
    'th': 'Thai',
    'vi': 'Vietnamese',
    
    # European languages
    'uk': 'Ukrainian',
    'be': 'Belarusian',
    'cs': 'Czech',
    'sk': 'Slovak',
    'ro': 'Romanian',
    'hu': 'Hungarian',
    'bg': 'Bulgarian',
    'hr': 'Croatian',
    'sr': 'Serbian',
    'sl': 'Slovenian',
    'mk': 'Macedonian',
    'sq': 'Albanian',
    'bs': 'Bosnian',
    'mt': 'Maltese',
    'ga': 'Irish',
    'cy': 'Welsh',
    'eu': 'Basque',
    'ca': 'Catalan',
    'gl': 'Galician',
    'gd': 'Scottish Gaelic',
    
    # Asian languages
    'id': 'Indonesian',
    'ms': 'Malay',
    'fil': 'Filipino',
    'tl': 'Tagalog',
    'my': 'Burmese',
    'km': 'Khmer',
    'lo': 'Lao',
    'ne': 'Nepali',
    'si': 'Sinhala',
    'ta': 'Tamil',
    'te': 'Telugu',
    'ml': 'Malayalam',
    'kn': 'Kannada',
    'or': 'Oriya',
    'as': 'Assamese',
    'bn': 'Bengali',
    'gu': 'Gujarati',
    'mr': 'Marathi',
    'pa': 'Punjabi',
    'sa': 'Sanskrit',
    'ur': 'Urdu',
    'sd': 'Sindhi',
    'ps': 'Pashto',
    'ku': 'Kurdish',
    'fa': 'Persian',
    
    # African languages
    'sw': 'Swahili',
    'ha': 'Hausa',
    'yo': 'Yoruba',
    'ig': 'Igbo',
    'am': 'Amharic',
    'so': 'Somali',
    'zu': 'Zulu',
    'xh': 'Xhosa',
    'st': 'Sotho',
    'tn': 'Tswana',
    'ss': 'Swati',
    've': 'Venda',
    'ts': 'Tsonga',
    'rw': 'Kinyarwanda',
    'ny': 'Chichewa',
    'mg': 'Malagasy',
    'ff': 'Fulah',
    'wo': 'Wolof',
    
    # Other common codes and variants
    'zh-cn': 'Chinese (Simplified)',
    'zh-tw': 'Chinese (Traditional)',
    'zh-hk': 'Chinese (Hong Kong)',
    'zh-sg': 'Chinese (Singapore)',
    'pt-br': 'Portuguese (Brazil)',
    'pt-pt': 'Portuguese (Portugal)',
    'es-mx': 'Spanish (Mexico)',
    'es-es': 'Spanish (Spain)',
    'en-us': 'English (US)',
    'en-gb': 'English (UK)',
    'en-ca': 'English (Canada)',
    'en-au': 'English (Australia)',
    'fr-fr': 'French (France)',
    'fr-ca': 'French (Canada)',
    'de-de': 'German (Germany)',
    'de-at': 'German (Austria)',
    'de-ch': 'German (Switzerland)',
    'it-it': 'Italian (Italy)',
    'nl-nl': 'Dutch (Netherlands)',
    'nl-be': 'Dutch (Belgium)',
    'ar-sa': 'Arabic (Saudi Arabia)',
    'ar-eg': 'Arabic (Egypt)',
    
    # Three-letter codes (ISO 639-2/3)
    'ara': 'Arabic',
    'ben': 'Bengali',
    'ces': 'Czech',
    'dan': 'Danish',
    'deu': 'German',
    'ell': 'Greek',
    'eng': 'English',
    'epo': 'Esperanto',
    'est': 'Estonian',
    'eus': 'Basque',
    'fin': 'Finnish',
    'fra': 'French',
    'gle': 'Irish',
    'glg': 'Galician',
    'heb': 'Hebrew',
    'hin': 'Hindi',
    'hrv': 'Croatian',
    'hun': 'Hungarian',
    'ind': 'Indonesian',
    'isl': 'Icelandic',
    'ita': 'Italian',
    'jpn': 'Japanese',
    'kat': 'Georgian',
    'kor': 'Korean',
    'lav': 'Latvian',
    'lit': 'Lithuanian',
    'mal': 'Malayalam',
    'nld': 'Dutch',
    'nor': 'Norwegian',
    'pol': 'Polish',
    'por': 'Portuguese',
    'ron': 'Romanian',
    'rus': 'Russian',
    'slk': 'Slovak',
    'slv': 'Slovenian',
    'spa': 'Spanish',
    'swe': 'Swedish',
    'tam': 'Tamil',
    'tel': 'Telugu',
    'tha': 'Thai',
    'tur': 'Turkish',
    'ukr': 'Ukrainian',
    'urd': 'Urdu',
    'vie': 'Vietnamese',
    'zho': 'Chinese',
    
    # Special cases and common variants
    'cn': 'Chinese',
    'in': 'Hindi',  # Often used for Hindi
    'kr': 'Korean',
    'jp': 'Japanese',
    'gb': 'English (UK)',
    'us': 'English (US)',
    'br': 'Portuguese (Brazil)',
    'mx': 'Spanish (Mexico)',
    'ca': 'French (Canada)',  # Could be English or French
    'au': 'English (Australia)',
    
    # Common placeholders and unknown
    'xx': 'Unknown',
    'zz': 'Unknown',
    'mul': 'Multiple Languages',
    'und': 'Undetermined',
    'mis': 'Uncoded Language',
    'zxx': 'No linguistic content',
    'n/a': 'Not Available',
    'nan': 'Not Available',
    'null': 'Not Available',
    'none': 'Not Available',
}

print(f"Created mapping with {len(language_mapping)} language codes")

# Apply mapping to create language_name column
print("\\n--- Applying Language Mapping ---")

def map_language_code(code):
    """
    Map language code to full language name.
    Returns the code itself if not found in mapping.
    """
    if pd.isna(code):
        return 'Unknown'
    
    code_str = str(code).strip().lower()
    
    # Try exact match first
    if code_str in language_mapping:
        return language_mapping[code_str]
    
    # Try case-insensitive match
    for key, value in language_mapping.items():
        if key.lower() == code_str:
            return value
    
    # Try partial match (e.g., 'zh-CN' might be stored as 'zh-cn')
    code_lower = code_str.lower()
    for key, value in language_mapping.items():
        if key.lower() in code_lower or code_lower in key.lower():
            return value
    
    # If no match found, return the code itself
    return code_str.title()  # Title case for readability

# Create language name column
df['original_language_name'] = df['original_language'].apply(map_language_code)

# Count how many were successfully mapped
unique_codes = df['original_language'].unique()
mapped_codes = set()
unmapped_codes = []

for code in unique_codes:
    if pd.isna(code):
        continue
    mapped_name = map_language_code(code)
    if mapped_name.lower() != str(code).lower().title():
        mapped_codes.add(str(code))
    else:
        unmapped_codes.append(str(code))

print(f"Successfully mapped {len(mapped_codes)} unique language codes")
print(f"Could not map {len(unmapped_codes)} unique language codes")

if unmapped_codes:
    print("\\nSample of unmapped language codes:")
    for code in unmapped_codes[:20]:
        count = (df['original_language'] == code).sum()
        print(f"  '{code}': {count:,} occurrences")

# Analyze mapping results
print("\\n--- Mapping Results Analysis ---")
print(f"Total movies: {len(df):,}")
print(f"Unique language codes: {len(unique_codes):,}")
print(f"Successfully mapped: {len(mapped_codes):,} codes")
print(f"Unmapped: {len(unmapped_codes):,} codes")

# Show top languages by name
print("\\n--- Top Languages by Name ---")
language_name_counts = df['original_language_name'].value_counts().head(20)
for name, count in language_name_counts.items():
    pct = count / len(df) * 100
    print(f"{name:30s}: {count:8,} movies ({pct:5.1f}%)")

# Check for movies still showing code instead of name
code_like_names = df[df['original_language_name'].str.len() <= 3]['original_language_name'].unique()
if len(code_like_names) > 0:
    print(f"\\nMovies with code-like names (≤3 chars): {len(code_like_names)} types")
    print("Sample:", code_like_names[:10])

# Create a summary of mapping coverage
print("\\n--- Mapping Coverage Summary ---")
top_20_languages = df['original_language'].value_counts().head(20).index.tolist()

print("\\nMapping of top 20 language codes:")
print("-" * 50)
for code in top_20_languages:
    count = (df['original_language'] == code).sum()
    name = map_language_code(code)
    pct = count / len(df) * 100
    print(f"{code:10s} → {name:25s}: {count:8,} movies ({pct:5.1f}%)")

# Sample after mapping
print("\\n--- Sample After Mapping ---")
sample_cols = ['title', 'original_language', 'original_language_name', 'language_category']
sample_df = df[sample_cols].head(15).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
print(sample_df[['title_truncated', 'original_language', 'original_language_name', 'language_category']].to_string(index=False))

# Save the mapping for reference
mapping_df = pd.DataFrame(list(language_mapping.items()), columns=['language_code', 'language_name'])
mapping_file = 'language_mapping.csv'
mapping_df.to_csv(mapping_file, index=False)
print(f"\\nLanguage mapping saved to: {mapping_file}")

mapping_time = time.time() - mapping_start
print(f"\\nLanguage mapping completed in {mapping_time:.2f} seconds")

Creating language code to name mapping...
Created mapping with 172 language codes
\n--- Applying Language Mapping ---
Successfully mapped 169 unique language codes
Could not map 0 unique language codes
\n--- Mapping Results Analysis ---
Total movies: 590,202
Unique language codes: 169
Successfully mapped: 169 codes
Unmapped: 0 codes
\n--- Top Languages by Name ---
English                       :  263,734 movies ( 44.7%)
French                        :   36,863 movies (  6.2%)
Spanish                       :   36,458 movies (  6.2%)
German                        :   31,103 movies (  5.3%)
Japanese                      :   24,310 movies (  4.1%)
Chinese                       :   19,055 movies (  3.2%)
Portuguese                    :   18,656 movies (  3.2%)
Russian                       :   16,930 movies (  2.9%)
Italian                       :   16,271 movies (  2.8%)
Korean                        :    8,707 movies (  1.5%)
Czech                         :    7,296 movies (  1.2%)
Hindi 

In [22]:
# Clean original_title Column
print("Cleaning original_title column...")
original_title_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'original_title' column: {df['original_title'].dtype}")
print(f"\\n--- Basic Statistics ---")
print(f"Total rows: {len(df):,}")
print(f"Non-null count: {df['original_title'].notnull().sum():,}")
print(f"Null count: {df['original_title'].isnull().sum():,}")
print(f"Percentage null: {df['original_title'].isnull().sum()/len(df)*100:.2f}%")

# 2. Show sample values
print(f"\\n--- Sample Values (first 10) ---")
samples = df['original_title'].head(10).tolist()
for i, title in enumerate(samples, 1):
    print(f"{i:2d}. '{title}'")

# 3. Create a column to mark missing original_title before cleaning
df['missing_original_title'] = df['original_title'].isnull()
missing_count = df['missing_original_title'].sum()
print(f"\\nCreated 'missing_original_title' column: {missing_count} rows marked as True ({missing_count/len(df)*100:.1f}%)")

# 4. Compare with 'title' column
print(f"\\n--- Comparison with 'title' Column ---")
if 'title' in df.columns:
    # Check how many titles are different from original_title
    different_titles = (df['title'] != df['original_title']).sum()
    print(f"Movies where 'title' differs from 'original_title': {different_titles:,} ({different_titles/len(df)*100:.1f}%)")
    
    # Check for nulls in both columns
    title_nulls = df['title'].isnull().sum()
    original_title_nulls = df['original_title'].isnull().sum()
    print(f"Null in 'title': {title_nulls:,} ({title_nulls/len(df)*100:.1f}%)")
    print(f"Null in 'original_title': {original_title_nulls:,} ({original_title_nulls/len(df)*100:.1f}%)")
    
    # Check where one is null but the other isn't
    title_null_original_not = (df['title'].isnull() & df['original_title'].notnull()).sum()
    original_null_title_not = (df['original_title'].isnull() & df['title'].notnull()).sum()
    print(f"Movies with null 'title' but not null 'original_title': {title_null_original_not:,}")
    print(f"Movies with null 'original_title' but not null 'title': {original_null_title_not:,}")
    
    # Show examples where titles differ
    if different_titles > 0:
        print("\\nSample of movies with different 'title' and 'original_title':")
        diff_samples = df[df['title'] != df['original_title']][['title', 'original_title']].head(5)
        for _, row in diff_samples.iterrows():
            print(f"  Title: '{row['title']}'")
            print(f"  Original: '{row['original_title']}'")
            print()

# 5. Check for duplicate original titles
print(f"\\n--- Checking for Duplicate Original Titles ---")
# Only check non-null titles for duplicates
non_null_titles = df[df['original_title'].notnull()]['original_title']
duplicate_titles = non_null_titles.duplicated().sum()
unique_titles = non_null_titles.nunique()

print(f"Unique original titles (non-null): {unique_titles:,}")
print(f"Duplicate original titles (non-null): {duplicate_titles:,}")

if duplicate_titles > 0:
    print("\\nSample of duplicate original titles:")
    duplicates = df[df['original_title'].notnull() & df['original_title'].duplicated(keep=False)]
    duplicate_counts = duplicates['original_title'].value_counts().head(5)
    
    for title, count in duplicate_counts.items():
        print(f"  '{title}': {count} occurrences")
        # Show release years for this duplicate title
        years = duplicates[duplicates['original_title'] == title]['release_year'].unique()[:3]
        if len(years) > 0:
            years_str = ', '.join([str(int(y)) for y in years if not pd.isna(y)])
            print(f"      Release years: {years_str}")

# 6. Check for empty strings or whitespace-only titles
print(f"\\n--- Checking for Empty/Whitespace Titles ---")
def is_empty_or_whitespace(title):
    if pd.isna(title):
        return False  # Already handled as null
    return str(title).strip() == ''

empty_titles = df['original_title'].apply(is_empty_or_whitespace).sum()
print(f"Empty or whitespace-only titles: {empty_titles:,} ({empty_titles/len(df)*100:.2f}%)")

if empty_titles > 0:
    print("Sample of empty/whitespace titles:")
    empty_samples = df[df['original_title'].apply(is_empty_or_whitespace)][['title', 'original_title']].head(5)
    print(empty_samples.to_string(index=False))

# 7. Handle missing values
print(f"\\n--- Handling Missing Values ---")

# Strategy: If original_title is null but title exists, use title
# Otherwise, fill with placeholder
if 'title' in df.columns:
    # Count how many we can fill from title column
    can_fill_from_title = (df['original_title'].isnull() & df['title'].notnull()).sum()
    print(f"Can fill {can_fill_from_title:,} null original_titles from 'title' column")
    
    # Fill missing original_titles with title when available
    mask = df['original_title'].isnull() & df['title'].notnull()
    df.loc[mask, 'original_title'] = df.loc[mask, 'title']
    print(f"Filled {mask.sum():,} null original_titles from 'title' column")

# For any remaining nulls, fill with placeholder
remaining_nulls = df['original_title'].isnull().sum()
if remaining_nulls > 0:
    placeholder = "Title Not Available"
    df['original_title'] = df['original_title'].fillna(placeholder)
    print(f"Filled remaining {remaining_nulls:,} nulls with '{placeholder}'")

print(f"Null count after imputation: {df['original_title'].isnull().sum()}")

# 8. Clean and standardize titles
print(f"\\n--- Cleaning and Standardizing Titles ---")

# Remove extra whitespace
df['original_title'] = df['original_title'].astype(str).str.strip()
print("Trimmed leading/trailing whitespace")

# Replace empty strings with placeholder
empty_mask = df['original_title'] == ''
df.loc[empty_mask, 'original_title'] = 'Title Not Available'
print(f"Replaced {empty_mask.sum():,} empty strings with 'Title Not Available'")

# Create cleaned version (optional: remove special characters, standardize case)
df['original_title_clean'] = df['original_title'].str.strip()

# 9. Create quality flags
df['filled_from_title'] = mask if 'mask' in locals() else pd.Series(False, index=df.index)
df['is_placeholder_title'] = df['original_title'].isin(['Title Not Available', 'Unknown', 'NO_TITLE'])
df['has_original_title'] = ~df['is_placeholder_title']

print(f"\\nCreated quality flags:")
print(f"  - 'filled_from_title': {df['filled_from_title'].sum():,} rows ({df['filled_from_title'].sum()/len(df)*100:.1f}%)")
print(f"  - 'is_placeholder_title': {df['is_placeholder_title'].sum():,} rows ({df['is_placeholder_title'].sum()/len(df)*100:.1f}%)")
print(f"  - 'has_original_title': {df['has_original_title'].sum():,} rows ({df['has_original_title'].sum()/len(df)*100:.1f}%)")

# 10. Analyze title characteristics
print(f"\\n--- Title Characteristics Analysis ---")

# Title length analysis
df['original_title_length'] = df['original_title'].str.len()
print(f"\\nTitle length statistics (characters):")
print(f"  Min: {df['original_title_length'].min()}")
print(f"  Max: {df['original_title_length'].max()}")
print(f"  Mean: {df['original_title_length'].mean():.1f}")
print(f"  Median: {df['original_title_length'].median():.1f}")

# Create length categories
bins = [0, 10, 20, 30, 40, 50, 100, float('inf')]
labels = ['≤10', '11-20', '21-30', '31-40', '41-50', '51-100', '100+']
df['original_title_length_category'] = pd.cut(df['original_title_length'], bins=bins, labels=labels)

print(f"\\nTitle length distribution:")
length_dist = df['original_title_length_category'].value_counts().sort_index()
for category, count in length_dist.items():
    pct = count / len(df) * 100
    print(f"  {category:8s}: {count:6,} movies ({pct:5.1f}%)")

# 11. Language analysis of titles
print(f"\\n--- Language Analysis of Titles ---")
if 'original_language' in df.columns:
    # Check if titles match their language (basic check for non-Latin characters)
    def contains_non_latin(text):
        if pd.isna(text):
            return False
        # Simple check for non-ASCII characters
        try:
            text.encode('ascii')
            return False
        except UnicodeEncodeError:
            return True
    
    df['title_has_non_latin'] = df['original_title'].apply(contains_non_latin)
    non_latin_titles = df['title_has_non_latin'].sum()
    print(f"Titles containing non-Latin characters: {non_latin_titles:,} ({non_latin_titles/len(df)*100:.1f}%)")
    
    # Compare with original_language
    if 'original_language_name' in df.columns:
        non_latin_by_language = df[df['title_has_non_latin']].groupby('original_language_name').size().sort_values(ascending=False).head(10)
        print(f"\\nTop languages with non-Latin titles:")
        for lang, count in non_latin_by_language.items():
            pct = count / non_latin_titles * 100
            print(f"  {lang:20s}: {count:6,} titles ({pct:5.1f}%)")

# 12. Verify the cleaning
print("\\n--- original_title Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count:,} ({missing_count/len(df)*100:.1f}%)")
print(f"2. Created 'missing_original_title' column: {missing_count:,} True values")
print(f"3. Created 'filled_from_title' column: {df['filled_from_title'].sum():,} True values")
print(f"4. Created 'is_placeholder_title' column: {df['is_placeholder_title'].sum():,} True values")
print(f"5. Created 'has_original_title' column: {df['has_original_title'].sum():,} True values")
print(f"6. Created 'title_has_non_latin' column: {df['title_has_non_latin'].sum() if 'title_has_non_latin' in df.columns else 'N/A':,} True values")
print(f"7. Current null count: {df['original_title'].isnull().sum()}")
print(f"8. Data type: {df['original_title'].dtype}")
print(f"9. Unique titles: {df['original_title'].nunique():,}")

# 13. Sample after cleaning
print("\\n--- Sample After Cleaning ---")
print("First 15 rows with original_title data:")
sample_cols = ['title', 'original_title', 'original_language_name', 'has_original_title', 'filled_from_title']
if 'original_language_name' not in df.columns:
    sample_cols = ['title', 'original_title', 'has_original_title', 'filled_from_title']

sample_df = df[sample_cols].head(15).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
sample_df['original_title_truncated'] = sample_df['original_title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)

print(sample_df[['title_truncated', 'original_title_truncated', 'has_original_title', 'filled_from_title'] + 
               (['original_language_name'] if 'original_language_name' in df.columns else [])].to_string(index=False))

# 14. Show examples of different title scenarios
print("\\n--- Examples of Different Title Scenarios ---")

# Example 1: Same title in both columns
same_titles = df[(df['title'] == df['original_title']) & df['has_original_title']].head(3)
if len(same_titles) > 0:
    print("\\nMovies with identical 'title' and 'original_title':")
    for _, row in same_titles.iterrows():
        print(f"  Both: '{row['title'][:40]}{'...' if len(row['title']) > 40 else ''}'")

# Example 2: Different titles
if 'title' in df.columns:
    diff_titles = df[(df['title'] != df['original_title']) & df['has_original_title']].head(3)
    if len(diff_titles) > 0:
        print("\\nMovies with different 'title' and 'original_title':")
        for _, row in diff_titles.iterrows():
            print(f"  Title: '{row['title'][:30]}{'...' if len(row['title']) > 30 else ''}'")
            print(f"  Original: '{row['original_title'][:30]}{'...' if len(row['original_title']) > 30 else ''}'")
            if 'original_language_name' in df.columns:
                print(f"  Language: {row.get('original_language_name', 'N/A')}")
            print()

# Example 3: Placeholder titles
placeholder_titles = df[df['is_placeholder_title']].head(3)
if len(placeholder_titles) > 0:
    print("\\nMovies with placeholder titles:")
    for _, row in placeholder_titles.iterrows():
        print(f"  Original Title: '{row['original_title']}'")
        print(f"  Regular Title: '{row.get('title', 'N/A')[:30]}{'...' if len(row.get('title', '')) > 30 else ''}'")

original_title_clean_time = time.time() - original_title_clean_start
print(f"\\noriginal_title column cleaning completed in {original_title_clean_time:.2f} seconds")

Cleaning original_title column...
Current data type of 'original_title' column: object
\n--- Basic Statistics ---
Total rows: 590,202
Non-null count: 590,199
Null count: 3
Percentage null: 0.00%
\n--- Sample Values (first 10) ---
 1. 'Ariel'
 2. 'Varjoja paratiisissa'
 3. 'Four Rooms'
 4. 'Judgment Night'
 5. 'Sonntag, im August'
 6. 'Star Wars'
 7. 'Finding Nemo'
 8. 'Forrest Gump'
 9. 'American Beauty'
10. 'Citizen Kane'
\nCreated 'missing_original_title' column: 3 rows marked as True (0.0%)
\n--- Comparison with 'title' Column ---
Movies where 'title' differs from 'original_title': 184,100 (31.2%)
Null in 'title': 0 (0.0%)
Null in 'original_title': 3 (0.0%)
Movies with null 'title' but not null 'original_title': 0
Movies with null 'original_title' but not null 'title': 3
\nSample of movies with different 'title' and 'original_title':
  Title: 'Shadows in Paradise'
  Original: 'Varjoja paratiisissa'

  Title: 'Sunday in August'
  Original: 'Sonntag, im August'

  Title: 'The Fifth El

In [23]:
# Clean overview Column
print("Cleaning overview column...")
overview_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'overview' column: {df['overview'].dtype}")
print(f"\\n--- Basic Statistics ---")
print(f"Total rows: {len(df):,}")
print(f"Non-null count: {df['overview'].notnull().sum():,}")
print(f"Null count: {df['overview'].isnull().sum():,}")
print(f"Percentage null: {df['overview'].isnull().sum()/len(df)*100:.2f}%")

# 2. Show sample values
print(f"\\n--- Sample Overviews (first 5) ---")
samples = df['overview'].head(5).tolist()
for i, overview in enumerate(samples, 1):
    overview_preview = str(overview)[:150] + '...' if len(str(overview)) > 150 else str(overview)
    print(f"{i:2d}. {overview_preview}")
    print()

# 3. Create a column to mark missing overview before cleaning
df['missing_overview'] = df['overview'].isnull()
missing_count = df['missing_overview'].sum()
print(f"\\nCreated 'missing_overview' column: {missing_count} rows marked as True ({missing_count/len(df)*100:.1f}%)")

# 4. Check for empty strings or whitespace-only overviews
print(f"\\n--- Checking for Empty/Whitespace Overviews ---")
def is_empty_or_whitespace(text):
    if pd.isna(text):
        return False  # Already handled as null
    return str(text).strip() == ''

empty_overviews = df['overview'].apply(is_empty_or_whitespace).sum()
print(f"Empty or whitespace-only overviews: {empty_overviews:,} ({empty_overviews/len(df)*100:.2f}%)")

if empty_overviews > 0:
    print("Sample of empty/whitespace overviews:")
    empty_samples = df[df['overview'].apply(is_empty_or_whitespace)][['title', 'overview']].head(5)
    print(empty_samples[['title']].to_string(index=False))

# 5. Analyze overview length
print(f"\\n--- Overview Length Analysis ---")
# Calculate length for non-null overviews
overview_lengths = df[df['overview'].notnull()]['overview'].apply(lambda x: len(str(x)))
print(f"Overview length statistics (characters, non-null only):")
print(f"  Min: {overview_lengths.min()}")
print(f"  Max: {overview_lengths.max()}")
print(f"  Mean: {overview_lengths.mean():.1f}")
print(f"  Median: {overview_lengths.median():.1f}")
print(f"  Std Dev: {overview_lengths.std():.1f}")

# Create length categories
df['overview_length'] = df['overview'].apply(lambda x: len(str(x)) if not pd.isna(x) else 0)
bins = [0, 1, 10, 50, 100, 200, 500, 1000, float('inf')]
labels = ['0 (null/empty)', '1-10', '11-50', '51-100', '101-200', '201-500', '501-1000', '1000+']
df['overview_length_category'] = pd.cut(df['overview_length'], bins=bins, labels=labels)

print(f"\\nOverview length distribution:")
length_dist = df['overview_length_category'].value_counts().sort_index()
for category, count in length_dist.items():
    pct = count / len(df) * 100
    print(f"  {category:15s}: {count:6,} movies ({pct:5.1f}%)")

# 6. Check for duplicate overviews (plagiarism or template use)
print(f"\\n--- Checking for Duplicate Overviews ---")
# Only check non-null, non-empty overviews
non_empty_overviews = df[~df['overview'].apply(is_empty_or_whitespace) & df['overview'].notnull()]['overview']
duplicate_overviews = non_empty_overviews.duplicated().sum()
unique_overviews = non_empty_overviews.nunique()

print(f"Unique overviews (non-empty): {unique_overviews:,}")
print(f"Duplicate overviews (non-empty): {duplicate_overviews:,}")

if duplicate_overviews > 0:
    print("\\nSample of duplicate overviews:")
    duplicates = df[~df['overview'].apply(is_empty_or_whitespace) & df['overview'].notnull() & df['overview'].duplicated(keep=False)]
    duplicate_counts = duplicates['overview'].value_counts().head(3)
    
    for overview, count in duplicate_counts.items():
        overview_preview = str(overview)[:100] + '...' if len(str(overview)) > 100 else str(overview)
        print(f"  Overview: '{overview_preview}'")
        print(f"  Occurrences: {count}")
        # Show titles for this duplicate overview
        titles = duplicates[duplicates['overview'] == overview]['title'].head(3).tolist()
        for title in titles:
            print(f"      - {title[:50]}{'...' if len(title) > 50 else ''}")
        print()

# 7. Check for common patterns or templates
print(f"\\n--- Checking for Common Patterns ---")
# Look for overviews that start with common phrases
common_patterns = {
    'This movie is about': 'Template-like start',
    'The story follows': 'Narrative template',
    'In a world where': 'Common opening',
    'After': 'Time-based opening',
    'When': 'Conditional opening',
    'The film tells the story of': 'Standard description',
    'NaN': 'Placeholder NaN string',
    'None': 'Placeholder None',
    'N/A': 'Placeholder N/A',
    'No overview': 'Explicit missing indicator',
    'No description': 'Explicit missing indicator',
}

pattern_counts = {}
for pattern, description in common_patterns.items():
    count = df['overview'].apply(lambda x: str(x).startswith(pattern) if not pd.isna(x) else False).sum()
    if count > 0:
        pattern_counts[pattern] = count

if pattern_counts:
    print(f"Found {sum(pattern_counts.values()):,} overviews with common patterns:")
    for pattern, count in sorted(pattern_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
        pct = count / len(df) * 100
        print(f"  Starts with '{pattern}': {count:,} movies ({pct:.1f}%)")

# 8. Handle missing values
print(f"\\n--- Handling Missing Values ---")

# Strategy: Fill nulls and empties with placeholder
placeholder = "No overview available"
null_and_empty_mask = df['overview'].isnull() | df['overview'].apply(is_empty_or_whitespace)
fill_count = null_and_empty_mask.sum()

df.loc[null_and_empty_mask, 'overview'] = placeholder
print(f"Filled {fill_count:,} null/empty overviews with '{placeholder}'")

# Also replace common placeholder strings
placeholder_strings = ['nan', 'NaN', 'None', 'none', 'N/A', 'n/a', 'no overview', 'no description', 'no overview available']
for placeholder_str in placeholder_strings:
    mask = df['overview'].str.lower() == placeholder_str.lower()
    if mask.any():
        df.loc[mask, 'overview'] = placeholder
        print(f"Replaced '{placeholder_str}' with '{placeholder}'")

print(f"Null count after imputation: {df['overview'].isnull().sum()}")

# 9. Clean and standardize overviews
print(f"\\n--- Cleaning and Standardizing Overviews ---")

# Ensure all are strings
df['overview'] = df['overview'].astype(str)

# Trim whitespace
df['overview'] = df['overview'].str.strip()
print("Trimmed leading/trailing whitespace")

# Remove extra internal whitespace (multiple spaces/tabs/newlines)
df['overview'] = df['overview'].str.replace(r'\\s+', ' ', regex=True)
print("Normalized internal whitespace")

# Create cleaned version
df['overview_clean'] = df['overview']

# 10. Create quality flags
df['has_overview'] = df['overview'] != placeholder
df['short_overview'] = (df['overview_length'] > 0) & (df['overview_length'] <= 50)
df['detailed_overview'] = df['overview_length'] > 200
df['template_overview'] = df['overview'].apply(
    lambda x: any(x.lower().startswith(pattern.lower()) for pattern in ['this movie is about', 'the story follows', 'in a world where'])
)

print(f"\\nCreated quality flags:")
print(f"  - 'has_overview': {df['has_overview'].sum():,} rows ({df['has_overview'].sum()/len(df)*100:.1f}%)")
print(f"  - 'short_overview' (≤50 chars): {df['short_overview'].sum():,} rows ({df['short_overview'].sum()/len(df)*100:.1f}%)")
print(f"  - 'detailed_overview' (>200 chars): {df['detailed_overview'].sum():,} rows ({df['detailed_overview'].sum()/len(df)*100:.1f}%)")
print(f"  - 'template_overview': {df['template_overview'].sum():,} rows ({df['template_overview'].sum()/len(df)*100:.1f}%)")

# 11. Analyze overview quality by other factors
print(f"\\n--- Overview Quality Analysis ---")

# By release decade
if 'release_year' in df.columns and df['release_year'].notnull().any():
    df['decade'] = (df['release_year'] // 10) * 10
    print(f"\\nOverview availability by decade:")
    for decade in sorted(df['decade'].dropna().unique()):
        if not pd.isna(decade):
            decade_movies = df[df['decade'] == decade]
            if len(decade_movies) > 0:
                has_overview_pct = (decade_movies['has_overview'].sum() / len(decade_movies)) * 100
                avg_length = decade_movies[decade_movies['has_overview']]['overview_length'].mean()
                print(f"  {int(decade)}s: {has_overview_pct:5.1f}% have overviews, avg length: {avg_length:.0f} chars")

# By language
if 'original_language_name' in df.columns:
    print(f"\\nOverview availability by top languages:")
    top_languages = df['original_language_name'].value_counts().head(10).index
    for lang in top_languages:
        lang_movies = df[df['original_language_name'] == lang]
        if len(lang_movies) > 0:
            has_overview_pct = (lang_movies['has_overview'].sum() / len(lang_movies)) * 100
            avg_length = lang_movies[lang_movies['has_overview']]['overview_length'].mean()
            print(f"  {lang:20s}: {has_overview_pct:5.1f}% have overviews, avg length: {avg_length:.0f} chars")

# 12. Text analysis (basic)
print(f"\\n--- Basic Text Analysis ---")

# Word count analysis (for non-placeholder overviews)
def word_count(text):
    if text == placeholder or pd.isna(text):
        return 0
    return len(str(text).split())

df['overview_word_count'] = df['overview'].apply(word_count)

print(f"\\nWord count statistics (non-placeholder only):")
non_placeholder = df[df['has_overview']]
if len(non_placeholder) > 0:
    word_counts = non_placeholder['overview_word_count']
    print(f"  Min: {word_counts.min()}")
    print(f"  Max: {word_counts.max()}")
    print(f"  Mean: {word_counts.mean():.1f}")
    print(f"  Median: {word_counts.median():.1f}")

# Most common words (basic)
print(f"\\nChecking for most common overview words...")
if len(non_placeholder) > 0:
    from collections import Counter
    all_words = []
    for overview in non_placeholder['overview'].head(1000):  # Sample first 1000 for speed
        words = str(overview).lower().split()
        all_words.extend(words)
    
    # Remove common stop words
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'this', 'that', 'these', 'those', 'his', 'her', 'its', 'their'}
    filtered_words = [word for word in all_words if word not in stop_words and len(word) > 2]
    
    word_freq = Counter(filtered_words)
    print(f"Top 10 most common words in overviews (sample):")
    for word, count in word_freq.most_common(10):
        print(f"  '{word}': {count} occurrences")

# 13. Verify the cleaning
print("\\n--- overview Column Cleaning Summary ---")
print(f"1. Original null/empty count: {missing_count + empty_overviews:,}")
print(f"2. Created 'missing_overview' column: {missing_count:,} True values")
print(f"3. Created 'has_overview' column: {df['has_overview'].sum():,} True values")
print(f"4. Created 'short_overview' column: {df['short_overview'].sum():,} True values")
print(f"5. Created 'detailed_overview' column: {df['detailed_overview'].sum():,} True values")
print(f"6. Created 'template_overview' column: {df['template_overview'].sum():,} True values")
print(f"7. Current null count: {df['overview'].isnull().sum()}")
print(f"8. Data type: {df['overview'].dtype}")
print(f"9. Placeholder overviews: {(df['overview'] == placeholder).sum():,} ({(df['overview'] == placeholder).sum()/len(df)*100:.1f}%)")

# 14. Sample after cleaning
print("\\n--- Sample After Cleaning ---")
print("First 10 rows with overview data:")
sample_cols = ['title', 'overview_length', 'has_overview', 'short_overview', 'detailed_overview']
sample_df = df[sample_cols].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
sample_df['overview_preview'] = df['overview'].head(10).apply(lambda x: x[:50] + '...' if len(x) > 50 else x)

print(sample_df[['title_truncated', 'overview_preview', 'overview_length', 'has_overview', 'short_overview', 'detailed_overview']].to_string(index=False))

# 15. Show examples of different overview scenarios
print("\\n--- Examples of Different Overview Scenarios ---")

# Example 1: Detailed overview
detailed = df[df['detailed_overview']].head(2)
if len(detailed) > 0:
    print("\\nDetailed overviews (>200 chars):")
    for _, row in detailed.iterrows():
        print(f"  Title: {row['title'][:40]}{'...' if len(row['title']) > 40 else ''}")
        print(f"  Length: {row['overview_length']} chars")
        preview = row['overview'][:100] + '...' if len(row['overview']) > 100 else row['overview']
        print(f"  Preview: {preview}")
        print()

# Example 2: Short overview
short = df[df['short_overview'] & df['has_overview']].head(2)
if len(short) > 0:
    print("\\nShort overviews (≤50 chars):")
    for _, row in short.iterrows():
        print(f"  Title: {row['title'][:40]}{'...' if len(row['title']) > 40 else ''}")
        print(f"  Overview: '{row['overview']}'")
        print()

# Example 3: No overview
no_overview = df[~df['has_overview']].head(2)
if len(no_overview) > 0:
    print("\\nMovies without overviews:")
    for _, row in no_overview.iterrows():
        print(f"  Title: {row['title'][:40]}{'...' if len(row['title']) > 40 else ''}")
        print(f"  Overview: '{row['overview']}'")
        print()

overview_clean_time = time.time() - overview_clean_start
print(f"\\noverview column cleaning completed in {overview_clean_time:.2f} seconds")

Cleaning overview column...
Current data type of 'overview' column: object
\n--- Basic Statistics ---
Total rows: 590,202
Non-null count: 527,567
Null count: 62,635
Percentage null: 10.61%
\n--- Sample Overviews (first 5) ---
 1. A Finnish man goes to the city to find a job after the mine where he worked is closed and his father commits suicide.

 2. Nikander, a rubbish collector and would-be entrepreneur, finds his plans for success dashed when his business associate dies. One evening, he meets Il...

 3. It's Ted the Bellhop's first night on the job...and the hotel's very unusual guests are about to place him in some outrageous predicaments. It seems t...

 4. Four young friends, while taking a shortcut en route to a local boxing match, witness a brutal murder which leaves them running for their lives.

 5. A couple on a boat. Their love is burnt out. But how to let go when souls are entangled?

\nCreated 'missing_overview' column: 62635 rows marked as True (10.6%)
\n--- Checking for

In [24]:
# Clean popularity Column
print("Cleaning popularity column...")
popularity_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'popularity' column: {df['popularity'].dtype}")
print(f"\\n--- Basic Statistics ---")
print(f"Total rows: {len(df):,}")
print(f"Non-null count: {df['popularity'].notnull().sum():,}")
print(f"Null count: {df['popularity'].isnull().sum():,}")
print(f"Percentage null: {df['popularity'].isnull().sum()/len(df)*100:.2f}%")

# 2. Show sample values
print(f"\\n--- Sample Values (first 10) ---")
samples = df['popularity'].head(10).tolist()
for i, popularity in enumerate(samples, 1):
    print(f"{i:2d}. {popularity}")

# 3. Show basic statistics before cleaning
print(f"\\n--- Statistical Summary Before Cleaning ---")
print(f"Min: {df['popularity'].min():.2f}")
print(f"Max: {df['popularity'].max():.2f}")
print(f"Mean: {df['popularity'].mean():.2f}")
print(f"Median: {df['popularity'].median():.2f}")
print(f"Std Dev: {df['popularity'].std():.2f}")

# 4. Create a column to mark missing popularity before cleaning
df['missing_popularity'] = df['popularity'].isnull()
missing_count = df['missing_popularity'].sum()
print(f"\\nCreated 'missing_popularity' column: {missing_count} rows marked as True ({missing_count/len(df)*100:.1f}%)")

# 5. Check for invalid values
print(f"\\n--- Checking for Invalid Values ---")

# Check for negative values (popularity should be >= 0)
negative_popularity = df[df['popularity'] < 0].shape[0]
print(f"Negative popularity values: {negative_popularity}")

if negative_popularity > 0:
    print("\\nSample of negative popularity:")
    negative_samples = df[df['popularity'] < 0][['title', 'popularity']].head(5)
    print(negative_samples.to_string(index=False))

# Check for zero popularity
zero_popularity = (df['popularity'] == 0).sum()
print(f"\\nMovies with 0 popularity: {zero_popularity} ({zero_popularity/len(df)*100:.2f}%)")

if zero_popularity > 0:
    print("Sample of movies with 0 popularity:")
    zero_samples = df[df['popularity'] == 0][['title', 'popularity']].head(5)
    print(zero_samples.to_string(index=False))

# Check for unreasonably high values (outliers)
# First check percentiles to understand distribution
print(f"\\n--- Popularity Distribution Percentiles ---")
percentiles = [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100]
for p in percentiles:
    value = df['popularity'].quantile(p/100)
    print(f"{p:3d}th percentile: {value:.2f}")

# Calculate IQR for outlier detection
Q1 = df['popularity'].quantile(0.25)
Q3 = df['popularity'].quantile(0.75)
IQR = Q3 - Q1
outlier_threshold = Q3 + 1.5 * IQR

high_outliers = df[df['popularity'] > outlier_threshold].shape[0]
print(f"\\nPotential high outliers (above {outlier_threshold:.2f}): {high_outliers} ({high_outliers/len(df)*100:.2f}%)")

if high_outliers > 0:
    print("Sample of high outlier popularity values:")
    outlier_samples = df[df['popularity'] > outlier_threshold][['title', 'popularity']].sort_values('popularity', ascending=False).head(5)
    print(outlier_samples.to_string(index=False))

# 6. Handle missing values
print(f"\\n--- Handling Missing Values ---")

# Strategy: Impute with median (less affected by outliers)
popularity_median = df['popularity'].median()
print(f"Median popularity for imputation: {popularity_median:.2f}")

df['popularity'] = df['popularity'].fillna(popularity_median)
print(f"Null popularity after imputation: {df['popularity'].isnull().sum()}")

# 7. Ensure proper data type
df['popularity'] = df['popularity'].astype(float)
print(f"Data type after conversion: {df['popularity'].dtype}")

# 8. Clip negative values to 0 if needed
if negative_popularity > 0:
    df['popularity'] = df['popularity'].clip(lower=0)
    print(f"\\nClipped {negative_popularity} negative values to 0")

# 9. Create categories for analysis (but don't analyze yet)
print(f"\\n--- Creating Popularity Categories ---")

# Create bins based on percentiles for even distribution
bins = [
    df['popularity'].min() - 0.01,  # Slightly below min
    df['popularity'].quantile(0.20),
    df['popularity'].quantile(0.40),
    df['popularity'].quantile(0.60),
    df['popularity'].quantile(0.80),
    df['popularity'].max() + 0.01   # Slightly above max
]

labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
df['popularity_category'] = pd.cut(df['popularity'], bins=bins, labels=labels)
print(f"Created 'popularity_category' column with {len(labels)} categories")

# 10. Create quality flags
df['zero_popularity'] = (df['popularity'] == 0)
df['low_popularity'] = (df['popularity'] > 0) & (df['popularity'] <= df['popularity'].quantile(0.25))
df['high_popularity_outlier'] = df['popularity'] > outlier_threshold

print(f"\\nCreated quality flags:")
print(f"  - 'zero_popularity': {df['zero_popularity'].sum():,} rows ({df['zero_popularity'].sum()/len(df)*100:.1f}%)")
print(f"  - 'low_popularity' (bottom 25%): {df['low_popularity'].sum():,} rows ({df['low_popularity'].sum()/len(df)*100:.1f}%)")
print(f"  - 'high_popularity_outlier': {df['high_popularity_outlier'].sum():,} rows ({df['high_popularity_outlier'].sum()/len(df)*100:.1f}%)")

# 11. Verify the cleaning
print("\\n--- popularity Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count:,} ({missing_count/len(df)*100:.1f}%)")
print(f"2. Created 'missing_popularity' column: {missing_count:,} True values")
print(f"3. Created 'zero_popularity' column: {df['zero_popularity'].sum():,} True values")
print(f"4. Created 'low_popularity' column: {df['low_popularity'].sum():,} True values")
print(f"5. Created 'high_popularity_outlier' column: {df['high_popularity_outlier'].sum():,} True values")
print(f"6. Current null count: {df['popularity'].isnull().sum()}")
print(f"7. Data type: {df['popularity'].dtype}")
print(f"8. Valid range check: {df['popularity'].min():.2f} to {df['popularity'].max():.2f}")

# 12. Post-cleaning statistics
print("\\n--- Statistics After Cleaning ---")
print(f"Min: {df['popularity'].min():.2f}")
print(f"Max: {df['popularity'].max():.2f}")
print(f"Mean: {df['popularity'].mean():.2f}")
print(f"Median: {df['popularity'].median():.2f}")
print(f"Std Dev: {df['popularity'].std():.2f}")

# 13. Sample after cleaning
print("\\n--- Sample After Cleaning ---")
print("First 10 rows with popularity data:")
sample_cols = ['title', 'popularity', 'popularity_category', 'zero_popularity', 'low_popularity']
sample_df = df[sample_cols].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
print(sample_df[['title_truncated', 'popularity', 'popularity_category', 'zero_popularity', 'low_popularity']].to_string(index=False))

# 14. Quick distribution check (just counts, not analysis)
print(f"\\n--- Quick Distribution Check ---")
print("Movies by popularity category:")
category_dist = df['popularity_category'].value_counts().sort_index()
for category, count in category_dist.items():
    print(f"  {category:10s}: {count:6,} movies")

popularity_clean_time = time.time() - popularity_clean_start
print(f"\\npopularity column cleaning completed in {popularity_clean_time:.2f} seconds")

Cleaning popularity column...
Current data type of 'popularity' column: float64
\n--- Basic Statistics ---
Total rows: 590,202
Non-null count: 590,202
Null count: 0
Percentage null: 0.00%
\n--- Sample Values (first 10) ---
 1. 4.1418
 2. 5.4837
 3. 3.0156
 4. 4.273
 5. 0.6529
 6. 17.4429
 7. 16.6558
 8. 18.1375
 9. 7.0234
10. 4.6511
\n--- Statistical Summary Before Cleaning ---
Min: 0.00
Max: 673.33
Mean: 1.13
Median: 0.63
Std Dev: 2.52
\nCreated 'missing_popularity' column: 0 rows marked as True (0.0%)
\n--- Checking for Invalid Values ---
Negative popularity values: 0
\nMovies with 0 popularity: 862 (0.15%)
Sample of movies with 0 popularity:
                title  popularity
            Lenteveld        0.00
    Personal Sergeant        0.00
From 5 p.m. to 5 a.m.        0.00
         Kernel Panic        0.00
  Nic & Jerry Get OFF        0.00
\n--- Popularity Distribution Percentiles ---
  0th percentile: 0.00
  1th percentile: 0.01
  5th percentile: 0.02
 10th percentile: 0.05
 25th

In [25]:
# Clean genres Column
print("Cleaning genres column...")
genres_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'genres' column: {df['genres'].dtype}")
print(f"\\n--- Basic Statistics ---")
print(f"Total rows: {len(df):,}")
print(f"Non-null count: {df['genres'].notnull().sum():,}")
print(f"Null count: {df['genres'].isnull().sum():,}")
print(f"Percentage null: {df['genres'].isnull().sum()/len(df)*100:.2f}%")

# 2. Show sample values
print(f"\\n--- Sample Values (first 10) ---")
samples = df['genres'].head(10).tolist()
for i, genres in enumerate(samples, 1):
    print(f"{i:2d}. {genres}")

# 3. Create a column to mark missing genres before cleaning
df['missing_genres'] = df['genres'].isnull()
missing_count = df['missing_genres'].sum()
print(f"\\nCreated 'missing_genres' column: {missing_count} rows marked as True ({missing_count/len(df)*100:.1f}%)")

# 4. Check for empty strings or whitespace-only genres
print(f"\\n--- Checking for Empty/Whitespace Genres ---")
def is_empty_or_whitespace(text):
    if pd.isna(text):
        return False  # Already handled as null
    return str(text).strip() == ''

empty_genres = df['genres'].apply(is_empty_or_whitespace).sum()
print(f"Empty or whitespace-only genres: {empty_genres:,} ({empty_genres/len(df)*100:.2f}%)")

if empty_genres > 0:
    print("Sample of empty/whitespace genres:")
    empty_samples = df[df['genres'].apply(is_empty_or_whitespace)][['title', 'genres']].head(5)
    print(empty_samples[['title']].to_string(index=False))

# 5. Analyze the format of genre data
print(f"\\n--- Analyzing Genre Format ---")
# Check common formats: comma-separated, JSON-like, or other
def analyze_genre_format(genre_str):
    if pd.isna(genre_str):
        return 'null'
    if is_empty_or_whitespace(genre_str):
        return 'empty'
    
    s = str(genre_str).strip()
    
    # Check for comma-separated list
    if ',' in s and '[' not in s and ']' not in s:
        return 'comma_separated'
    
    # Check for JSON/list format
    if ('[' in s and ']' in s) or ('{' in s and '}' in s):
        return 'json_like'
    
    # Check if it's a single genre
    if len(s.split()) <= 3:  # Short string, likely single genre
        return 'single_genre'
    
    return 'other'

df['genre_format'] = df['genres'].apply(analyze_genre_format)
format_counts = df['genre_format'].value_counts()

print(f"\\nGenre format distribution:")
for format_type, count in format_counts.items():
    pct = count / len(df) * 100
    print(f"  {format_type:20s}: {count:6,} movies ({pct:5.1f}%)")

# Show examples of each format
print(f"\\nExamples of each format:")
for format_type in format_counts.index:
    if format_type != 'null' and format_type != 'empty':
        examples = df[df['genre_format'] == format_type]['genres'].head(2).tolist()
        print(f"\\n{format_type}:")
        for ex in examples:
            print(f"  '{ex[:80]}{'...' if len(str(ex)) > 80 else ''}'")

# 6. Parse genres into lists
print(f"\\n--- Parsing Genres into Lists ---")

def parse_genres(genre_str):
    """
    Parse genre string into list of individual genres.
    Handles multiple formats.
    """
    if pd.isna(genre_str) or is_empty_or_whitespace(genre_str):
        return []
    
    s = str(genre_str).strip()
    
    # Handle comma-separated lists (most common)
    if ',' in s:
        # Split by comma, strip whitespace, filter empties
        genres = [g.strip() for g in s.split(',') if g.strip()]
        return genres
    
    # Handle JSON-like formats (might have brackets, quotes)
    # Simple approach: remove brackets and quotes, then split by comma
    s_clean = s.replace('[', '').replace(']', '').replace('{', '').replace('}', '').replace('"', '').replace("'", "")
    if ',' in s_clean:
        genres = [g.strip() for g in s_clean.split(',') if g.strip()]
        return genres
    
    # Handle single genre
    return [s]

# Parse genres
print("Parsing genre strings into lists...")
df['genres_list'] = df['genres'].apply(parse_genres)

# Count number of genres per movie
df['genre_count'] = df['genres_list'].apply(len)

print(f"\\nNumber of genres per movie:")
genre_count_dist = df['genre_count'].value_counts().sort_index()
for count, num_movies in genre_count_dist.head(10).items():
    pct = num_movies / len(df) * 100
    print(f"  {count} genres: {num_movies:6,} movies ({pct:5.1f}%)")

if len(genre_count_dist) > 10:
    print(f"  ... and {len(genre_count_dist) - 10} more categories")

# Check movies with no genres after parsing
no_genres_after_parse = (df['genre_count'] == 0).sum()
print(f"Movies with no genres after parsing: {no_genres_after_parse:,} ({no_genres_after_parse/len(df)*100:.1f}%)")

# 7. Extract all unique genres
print(f"\\n--- Extracting Unique Genres ---")
all_genres = []
for genres_list in df['genres_list']:
    all_genres.extend(genres_list)

unique_genres = set(all_genres)
print(f"Total unique genre labels found: {len(unique_genres)}")

# Show top 20 most common genres
from collections import Counter
genre_counter = Counter(all_genres)
print(f"\\nTop 20 most common genres:")
top_genres = genre_counter.most_common(20)
for genre, count in top_genres:
    pct = count / len(df) * 100
    print(f"  {genre:25s}: {count:6,} movies ({pct:5.1f}%)")

# Show some rare genres
rare_genres = [(g, c) for g, c in genre_counter.items() if c <= 5]
if rare_genres:
    print(f"\\nSample of rare genres (≤5 movies):")
    for genre, count in rare_genres[:10]:
        print(f"  '{genre}': {count} movies")

# 8. Handle missing/null genres
print(f"\\n--- Handling Missing/Null Genres ---")

# For movies with no genres, fill with placeholder
placeholder = "Unknown"
no_genres_mask = (df['genre_count'] == 0)
fill_count = no_genres_mask.sum()

if fill_count > 0:
    df.loc[no_genres_mask, 'genres_list'] = df.loc[no_genres_mask, 'genres_list'].apply(lambda x: [placeholder])
    print(f"Filled {fill_count:,} movies with no genres with ['{placeholder}']")

# Update genre count
df.loc[no_genres_mask, 'genre_count'] = 1

# 9. Clean and standardize genre names
print(f"\\n--- Standardizing Genre Names ---")

# Common genre standardization mappings
genre_standardization = {
    # Remove extra whitespace and title case
    'sci-fi': 'Science Fiction',
    'sci fi': 'Science Fiction',
    'scifi': 'Science Fiction',
    's-f': 'Science Fiction',
    'sf': 'Science Fiction',
    
    'tv movie': 'TV Movie',
    'tvmovie': 'TV Movie',
    
    'kids': 'Family',
    'children': 'Family',
    'children\'s': 'Family',
    
    'action & adventure': 'Action & Adventure',
    'action/adventure': 'Action & Adventure',
    
    'sci-fi & fantasy': 'Science Fiction & Fantasy',
    'sci fi & fantasy': 'Science Fiction & Fantasy',
    'fantasy/sci-fi': 'Science Fiction & Fantasy',
    
    'soap': 'Soap Opera',
    
    'talk': 'Talk Show',
    
    'news': 'News',
    
    'reality': 'Reality TV',
    
    # Common misspellings or variations
    'dram': 'Drama',
    'dra': 'Drama',
    'dramma': 'Drama',
    
    'come': 'Comedy',
    'com': 'Comedy',
    
    'act': 'Action',
    'acción': 'Action',  # Spanish
    
    'aventura': 'Adventure',  # Spanish
    'aventure': 'Adventure',  # French
    
    'terror': 'Horror',  # Spanish
    'horreur': 'Horror',  # French
    
    'romantic': 'Romance',
    'romantique': 'Romance',  # French
    
    'crim': 'Crime',
    'krimi': 'Crime',  # German
    
    'thril': 'Thriller',
    'suspense': 'Thriller',
    
    'doc': 'Documentary',
    'documentaire': 'Documentary',  # French
    'documental': 'Documentary',  # Spanish
    
    'anim': 'Animation',
    'animación': 'Animation',  # Spanish
    
    'hist': 'History',
    'histoire': 'History',  # French
    
    'mus': 'Music',
    'música': 'Music',  # Spanish
    
    'mystery': 'Mystery',
    'misterio': 'Mystery',  # Spanish
    
    'war': 'War',
    'guerra': 'War',  # Spanish
    
    'western': 'Western',
    'wéstern': 'Western',  # Spanish
    
    # Title case common genres
    'drama': 'Drama',
    'comedy': 'Comedy',
    'action': 'Action',
    'adventure': 'Adventure',
    'horror': 'Horror',
    'romance': 'Romance',
    'crime': 'Crime',
    'thriller': 'Thriller',
    'documentary': 'Documentary',
    'animation': 'Animation',
    'family': 'Family',
    'fantasy': 'Fantasy',
    'history': 'History',
    'music': 'Music',
    'mystery': 'Mystery',
    'war': 'War',
    'western': 'Western',
}

def standardize_genre(genre):
    """Standardize a single genre name."""
    if pd.isna(genre):
        return placeholder
    
    genre_lower = str(genre).strip().lower()
    
    # Check for exact match in standardization dictionary
    if genre_lower in genre_standardization:
        return genre_standardization[genre_lower]
    
    # Check for partial matches
    for key, value in genre_standardization.items():
        if key in genre_lower or genre_lower in key:
            return value
    
    # If no match found, title case and return
    return str(genre).strip().title()

def standardize_genre_list(genre_list):
    """Standardize a list of genres."""
    if not genre_list:
        return [placeholder]
    
    standardized = []
    for genre in genre_list:
        std_genre = standardize_genre(genre)
        if std_genre not in standardized:  # Avoid duplicates
            standardized.append(std_genre)
    
    return standardized if standardized else [placeholder]

print("Standardizing genre names...")
df['genres_clean'] = df['genres_list'].apply(standardize_genre_list)

# Update unique genres after standardization
all_clean_genres = []
for genres_list in df['genres_clean']:
    all_clean_genres.extend(genres_list)

unique_clean_genres = set(all_clean_genres)
print(f"Unique genres after standardization: {len(unique_clean_genres)}")

# Show top genres after standardization
clean_genre_counter = Counter(all_clean_genres)
print(f"\\nTop genres after standardization:")
top_clean_genres = clean_genre_counter.most_common(15)
for genre, count in top_clean_genres:
    pct = count / len(df) * 100
    print(f"  {genre:25s}: {count:6,} movies ({pct:5.1f}%)")

# 10. Create binary columns for top genres (optional)
print(f"\\n--- Creating Binary Genre Columns ---")
# Create binary columns for top N genres
top_n = 15
top_genres_list = [genre for genre, _ in clean_genre_counter.most_common(top_n)]

for genre in top_genres_list:
    col_name = f'genre_{genre.lower().replace(" ", "_").replace("&", "and")}'
    df[col_name] = df['genres_clean'].apply(lambda x: genre in x)
    print(f"  Created '{col_name}' for genre '{genre}'")

print(f"Created binary columns for top {top_n} genres")

# 11. Create quality flags
df['has_genres'] = df['genres_clean'].apply(lambda x: x != [placeholder] and len(x) > 0)
df['multiple_genres'] = df['genres_clean'].apply(lambda x: len(x) > 1)

print(f"\\nCreated quality flags:")
print(f"  - 'has_genres': {df['has_genres'].sum():,} rows ({df['has_genres'].sum()/len(df)*100:.1f}%)")
print(f"  - 'multiple_genres': {df['multiple_genres'].sum():,} rows ({df['multiple_genres'].sum()/len(df)*100:.1f}%)")

# 12. Verify the cleaning
print("\\n--- genres Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count:,} ({missing_count/len(df)*100:.1f}%)")
print(f"2. Created 'missing_genres' column: {missing_count:,} True values")
print(f"3. Created 'has_genres' column: {df['has_genres'].sum():,} True values")
print(f"4. Created 'multiple_genres' column: {df['multiple_genres'].sum():,} True values")
print(f"5. Current genres per movie: {df['genre_count'].mean():.1f} average")
print(f"6. Unique genres found: {len(unique_clean_genres):,}")
print(f"7. Movies with standardized genres: {len(df):,}")

# 13. Sample after cleaning
print("\\n--- Sample After Cleaning ---")
print("First 10 rows with genre data:")
sample_cols = ['title', 'genres_clean', 'genre_count', 'has_genres', 'multiple_genres']
sample_df = df[sample_cols].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
sample_df['genres_str'] = sample_df['genres_clean'].apply(lambda x: ', '.join(x) if x else 'None')

print(sample_df[['title_truncated', 'genres_str', 'genre_count', 'has_genres', 'multiple_genres']].to_string(index=False))

# 14. Show examples of different genre scenarios
print("\\n--- Examples of Different Genre Scenarios ---")

# Example 1: Multiple genres
multiple = df[df['multiple_genres']].head(2)
if len(multiple) > 0:
    print("\\nMovies with multiple genres:")
    for _, row in multiple.iterrows():
        print(f"  Title: {row['title'][:40]}{'...' if len(row['title']) > 40 else ''}")
        print(f"  Genres: {', '.join(row['genres_clean'])}")

# Example 2: Single genre
single = df[(df['has_genres']) & (~df['multiple_genres'])].head(2)
if len(single) > 0:
    print("\\nMovies with single genre:")
    for _, row in single.iterrows():
        print(f"  Title: {row['title'][:40]}{'...' if len(row['title']) > 40 else ''}")
        print(f"  Genre: {row['genres_clean'][0]}")

# Example 3: No genres
no_genres = df[~df['has_genres']].head(2)
if len(no_genres) > 0:
    print("\\nMovies with no genres (placeholder):")
    for _, row in no_genres.iterrows():
        print(f"  Title: {row['title'][:40]}{'...' if len(row['title']) > 40 else ''}")
        print(f"  Genres: {row['genres_clean']}")

genres_clean_time = time.time() - genres_clean_start
print(f"\\ngenres column cleaning completed in {genres_clean_time:.2f} seconds")

Cleaning genres column...
Current data type of 'genres' column: object
\n--- Basic Statistics ---
Total rows: 590,202
Non-null count: 590,202
Null count: 0
Percentage null: 0.00%
\n--- Sample Values (first 10) ---
 1. Comedy, Drama, Romance, Crime
 2. Comedy, Drama, Romance
 3. Comedy
 4. Action, Crime, Thriller
 5. Drama
 6. Adventure, Action, Science Fiction
 7. Animation, Family
 8. Comedy, Drama, Romance
 9. Drama
10. Mystery, Drama
\nCreated 'missing_genres' column: 0 rows marked as True (0.0%)
\n--- Checking for Empty/Whitespace Genres ---
Empty or whitespace-only genres: 0 (0.00%)
\n--- Analyzing Genre Format ---
\nGenre format distribution:
  single_genre        : 321,831 movies ( 54.5%)
  comma_separated     : 268,371 movies ( 45.5%)
\nExamples of each format:
\nsingle_genre:
  'Comedy'
  'Drama'
\ncomma_separated:
  'Comedy, Drama, Romance, Crime'
  'Comedy, Drama, Romance'
\n--- Parsing Genres into Lists ---
Parsing genre strings into lists...
\nNumber of genres per movie:
 

In [26]:
# Clean production_companies Column
print("Cleaning production_companies column...")
production_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'production_companies' column: {df['production_companies'].dtype}")
print(f"\\n--- Basic Statistics ---")
print(f"Total rows: {len(df):,}")
print(f"Non-null count: {df['production_companies'].notnull().sum():,}")
print(f"Null count: {df['production_companies'].isnull().sum():,}")
print(f"Percentage null: {df['production_companies'].isnull().sum()/len(df)*100:.2f}%")

# 2. Show sample values
print(f"\\n--- Sample Values (first 10) ---")
samples = df['production_companies'].head(10).tolist()
for i, companies in enumerate(samples, 1):
    preview = str(companies)[:100] + '...' if len(str(companies)) > 100 else str(companies)
    print(f"{i:2d}. {preview}")

# 3. Create a column to mark missing production_companies before cleaning
df['missing_production_companies'] = df['production_companies'].isnull()
missing_count = df['missing_production_companies'].sum()
print(f"\\nCreated 'missing_production_companies' column: {missing_count} rows marked as True ({missing_count/len(df)*100:.1f}%)")

# 4. Check for empty strings or whitespace-only values
print(f"\\n--- Checking for Empty/Whitespace Values ---")
def is_empty_or_whitespace(text):
    if pd.isna(text):
        return False  # Already handled as null
    return str(text).strip() == ''

empty_companies = df['production_companies'].apply(is_empty_or_whitespace).sum()
print(f"Empty or whitespace-only values: {empty_companies:,} ({empty_companies/len(df)*100:.2f}%)")

if empty_companies > 0:
    print("Sample of empty/whitespace values:")
    empty_samples = df[df['production_companies'].apply(is_empty_or_whitespace)][['title', 'production_companies']].head(5)
    print(empty_samples[['title']].to_string(index=False))

# 5. Analyze the format of production company data
print(f"\\n--- Analyzing Production Companies Format ---")
# Check common formats: comma-separated, JSON-like, or other
def analyze_company_format(company_str):
    if pd.isna(company_str) or is_empty_or_whitespace(company_str):
        return 'null_or_empty'
    
    s = str(company_str).strip()
    
    # Check for comma-separated list
    if ',' in s and '[' not in s and ']' not in s:
        return 'comma_separated'
    
    # Check for JSON/list format
    if ('[' in s and ']' in s) or ('{' in s and '}' in s):
        return 'json_like'
    
    # Check if it's a single company
    return 'single_company'

df['production_companies_format'] = df['production_companies'].apply(analyze_company_format)
format_counts = df['production_companies_format'].value_counts()

print(f"\\nProduction companies format distribution:")
for format_type, count in format_counts.items():
    pct = count / len(df) * 100
    print(f"  {format_type:20s}: {count:6,} movies ({pct:5.1f}%)")

# Show examples of each format
if format_counts.get('comma_separated', 0) > 0:
    examples = df[df['production_companies_format'] == 'comma_separated']['production_companies'].head(2).tolist()
    print(f"\\nExamples of comma_separated format:")
    for ex in examples:
        print(f"  '{ex[:80]}{'...' if len(str(ex)) > 80 else ''}'")

if format_counts.get('json_like', 0) > 0:
    examples = df[df['production_companies_format'] == 'json_like']['production_companies'].head(2).tolist()
    print(f"\\nExamples of json_like format:")
    for ex in examples:
        print(f"  '{ex[:80]}{'...' if len(str(ex)) > 80 else ''}'")

if format_counts.get('single_company', 0) > 0:
    examples = df[df['production_companies_format'] == 'single_company']['production_companies'].head(2).tolist()
    print(f"\\nExamples of single_company format:")
    for ex in examples:
        print(f"  '{ex[:80]}{'...' if len(str(ex)) > 80 else ''}'")

# 6. Parse production companies into lists
print(f"\\n--- Parsing Production Companies into Lists ---")

def parse_production_companies(company_str):
    """
    Parse production company string into list of individual companies.
    Handles multiple formats.
    """
    if pd.isna(company_str) or is_empty_or_whitespace(company_str):
        return []
    
    s = str(company_str).strip()
    
    # Handle comma-separated lists
    if ',' in s:
        # Split by comma, strip whitespace, filter empties
        companies = [c.strip() for c in s.split(',') if c.strip()]
        return companies
    
    # Handle JSON-like formats
    # Simple approach: remove brackets and quotes, then split by comma
    s_clean = s.replace('[', '').replace(']', '').replace('{', '').replace('}', '').replace('"', '').replace("'", "")
    if ',' in s_clean:
        companies = [c.strip() for c in s_clean.split(',') if c.strip()]
        return companies
    
    # Handle single company
    return [s]

# Parse production companies
print("Parsing production company strings into lists...")
df['production_companies_list'] = df['production_companies'].apply(parse_production_companies)

# Count number of production companies per movie
df['production_company_count'] = df['production_companies_list'].apply(len)

print(f"\\nNumber of production companies per movie:")
company_count_dist = df['production_company_count'].value_counts().sort_index()
for count, num_movies in company_count_dist.head(15).items():
    pct = num_movies / len(df) * 100
    if count == 0:
        print(f"  {count} companies: {num_movies:6,} movies ({pct:5.1f}%)")
    else:
        print(f"  {count} companies: {num_movies:6,} movies ({pct:5.1f}%)")

if len(company_count_dist) > 15:
    print(f"  ... and {len(company_count_dist) - 15} more categories")

# Check movies with no production companies after parsing
no_companies_after_parse = (df['production_company_count'] == 0).sum()
print(f"Movies with no production companies after parsing: {no_companies_after_parse:,} ({no_companies_after_parse/len(df)*100:.1f}%)")

# 7. Extract all unique production companies
print(f"\\n--- Extracting Unique Production Companies ---")
all_companies = []
for companies_list in df['production_companies_list']:
    all_companies.extend(companies_list)

unique_companies = set(all_companies)
print(f"Total unique production company labels found: {len(unique_companies)}")

# Show top 20 most common production companies
from collections import Counter
company_counter = Counter(all_companies)
print(f"\\nTop 20 most common production companies:")
top_companies = company_counter.most_common(20)
for company, count in top_companies:
    pct = count / len(df) * 100
    print(f"  {company:40s}: {count:6,} movies ({pct:5.1f}%)")

# Show some rare production companies
rare_companies = [(c, cnt) for c, cnt in company_counter.items() if cnt <= 5]
if rare_companies:
    print(f"\\nSample of rare production companies (≤5 movies):")
    for company, count in rare_companies[:10]:
        print(f"  '{company}': {count} movies")

# 8. Handle missing/null production companies
print(f"\\n--- Handling Missing/Null Production Companies ---")

# For movies with no production companies, fill with placeholder
placeholder = "Unknown"
no_companies_mask = (df['production_company_count'] == 0)
fill_count = no_companies_mask.sum()

if fill_count > 0:
    df.loc[no_companies_mask, 'production_companies_list'] = df.loc[no_companies_mask, 'production_companies_list'].apply(lambda x: [placeholder])
    print(f"Filled {fill_count:,} movies with no production companies with ['{placeholder}']")

# Update company count
df.loc[no_companies_mask, 'production_company_count'] = 1

# 9. Clean and standardize production company names
print(f"\\n--- Standardizing Production Company Names ---")

# Common production company standardization mappings
company_standardization = {
    # Major studios - standardize variations
    'Warner Bros.': 'Warner Bros.',
    'Warner Bros': 'Warner Bros.',
    'Warner Brothers': 'Warner Bros.',
    'Warner Bros. Pictures': 'Warner Bros.',
    
    'Walt Disney Pictures': 'Walt Disney Pictures',
    'Disney': 'Walt Disney Pictures',
    'The Walt Disney Company': 'Walt Disney Pictures',
    
    'Universal Pictures': 'Universal Pictures',
    'Universal Studios': 'Universal Pictures',
    'Universal': 'Universal Pictures',
    
    'Paramount Pictures': 'Paramount Pictures',
    'Paramount': 'Paramount Pictures',
    
    'Columbia Pictures': 'Columbia Pictures',
    'Columbia': 'Columbia Pictures',
    'Columbia Pictures Corporation': 'Columbia Pictures',
    
    '20th Century Fox': '20th Century Fox',
    'Twentieth Century Fox': '20th Century Fox',
    '20th Century Fox Film Corporation': '20th Century Fox',
    
    'Metro-Goldwyn-Mayer': 'Metro-Goldwyn-Mayer',
    'MGM': 'Metro-Goldwyn-Mayer',
    'Metro Goldwyn Mayer': 'Metro-Goldwyn-Mayer',
    
    'New Line Cinema': 'New Line Cinema',
    'New Line': 'New Line Cinema',
    
    'Miramax': 'Miramax',
    'Miramax Films': 'Miramax',
    
    'DreamWorks': 'DreamWorks',
    'DreamWorks Pictures': 'DreamWorks',
    'DreamWorks SKG': 'DreamWorks',
    
    'Sony Pictures': 'Sony Pictures',
    'Sony Pictures Entertainment': 'Sony Pictures',
    
    # TV networks
    'HBO': 'HBO',
    'Home Box Office': 'HBO',
    
    'BBC': 'BBC',
    'British Broadcasting Corporation': 'BBC',
    
    # Common abbreviations
    'Ltd.': 'Ltd',
    'Limited': 'Ltd',
    'Inc.': 'Inc',
    'Incorporated': 'Inc',
    'Co.': 'Co',
    'Company': 'Co',
}

def standardize_company(company):
    """Standardize a single production company name."""
    if pd.isna(company):
        return placeholder
    
    company_str = str(company).strip()
    if company_str == '':
        return placeholder
    
    # Check for exact match in standardization dictionary
    if company_str in company_standardization:
        return company_standardization[company_str]
    
    # Check for partial matches (company contains key)
    for key, value in company_standardization.items():
        if key.lower() in company_str.lower() or company_str.lower() in key.lower():
            return value
    
    # Remove common suffixes for standardization
    suffixes = [' Ltd.', ' Limited', ' Inc.', ' Incorporated', ' Co.', ' Company', ' Studios', ' Pictures', ' Film', ' Productions']
    for suffix in suffixes:
        if company_str.endswith(suffix):
            base_name = company_str[:-len(suffix)]
            # Check if base name is in standardization
            if base_name in company_standardization:
                return company_standardization[base_name]
            return base_name.strip()
    
    # If no match found, return the original (cleaned)
    return company_str

def standardize_company_list(company_list):
    """Standardize a list of production companies."""
    if not company_list:
        return [placeholder]
    
    standardized = []
    for company in company_list:
        std_company = standardize_company(company)
        if std_company not in standardized:  # Avoid duplicates
            standardized.append(std_company)
    
    return standardized if standardized else [placeholder]

print("Standardizing production company names...")
df['production_companies_clean'] = df['production_companies_list'].apply(standardize_company_list)

# Update unique companies after standardization
all_clean_companies = []
for companies_list in df['production_companies_clean']:
    all_clean_companies.extend(companies_list)

unique_clean_companies = set(all_clean_companies)
print(f"Unique production companies after standardization: {len(unique_clean_companies)}")

# Show top companies after standardization
clean_company_counter = Counter(all_clean_companies)
print(f"\\nTop production companies after standardization:")
top_clean_companies = clean_company_counter.most_common(15)
for company, count in top_clean_companies:
    pct = count / len(df) * 100
    print(f"  {company:40s}: {count:6,} movies ({pct:5.1f}%)")

# 10. Create binary columns for top production companies
print(f"\\n--- Creating Binary Production Company Columns ---")
# Create binary columns for top N companies
top_n = 10
top_companies_list = [company for company, _ in clean_company_counter.most_common(top_n)]

for company in top_companies_list:
    if company != placeholder:
        col_name = f'production_company_{company.lower().replace(" ", "_").replace(".", "").replace(",", "").replace("&", "and")}'
        df[col_name] = df['production_companies_clean'].apply(lambda x: company in x)
        print(f"  Created '{col_name}' for company '{company}'")

print(f"Created binary columns for top {top_n} production companies")

# 11. Create quality flags
df['has_production_companies'] = df['production_companies_clean'].apply(lambda x: x != [placeholder] and len(x) > 0)
df['multiple_production_companies'] = df['production_companies_clean'].apply(lambda x: len(x) > 1)
df['major_studio'] = df['production_companies_clean'].apply(
    lambda x: any(company in x for company in [
        'Warner Bros.', 'Walt Disney Pictures', 'Universal Pictures', 
        'Paramount Pictures', 'Columbia Pictures', '20th Century Fox',
        'Metro-Goldwyn-Mayer', 'Sony Pictures'
    ])
)

print(f"\\nCreated quality flags:")
print(f"  - 'has_production_companies': {df['has_production_companies'].sum():,} rows ({df['has_production_companies'].sum()/len(df)*100:.1f}%)")
print(f"  - 'multiple_production_companies': {df['multiple_production_companies'].sum():,} rows ({df['multiple_production_companies'].sum()/len(df)*100:.1f}%)")
print(f"  - 'major_studio': {df['major_studio'].sum():,} rows ({df['major_studio'].sum()/len(df)*100:.1f}%)")

# 12. Verify the cleaning
print("\\n--- production_companies Column Cleaning Summary ---")
print(f"1. Original null/empty count: {missing_count + empty_companies:,} ({missing_count + empty_companies/len(df)*100:.1f}%)")
print(f"2. Created 'missing_production_companies' column: {missing_count:,} True values")
print(f"3. Created 'has_production_companies' column: {df['has_production_companies'].sum():,} True values")
print(f"4. Created 'multiple_production_companies' column: {df['multiple_production_companies'].sum():,} True values")
print(f"5. Created 'major_studio' column: {df['major_studio'].sum():,} True values")
print(f"6. Current companies per movie: {df['production_company_count'].mean():.1f} average")
print(f"7. Unique production companies found: {len(unique_clean_companies):,}")
print(f"8. Movies with standardized companies: {len(df):,}")

# 13. Sample after cleaning
print("\\n--- Sample After Cleaning ---")
print("First 10 rows with production company data:")
sample_cols = ['title', 'production_companies_clean', 'production_company_count', 'has_production_companies', 'major_studio']
sample_df = df[sample_cols].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
sample_df['companies_str'] = sample_df['production_companies_clean'].apply(lambda x: ', '.join(x) if x else 'None')

print(sample_df[['title_truncated', 'companies_str', 'production_company_count', 'has_production_companies', 'major_studio']].to_string(index=False))

# 14. Show examples of different production company scenarios
print("\\n--- Examples of Different Production Company Scenarios ---")

# Example 1: Major studio films
major_studio_films = df[df['major_studio']].head(2)
if len(major_studio_films) > 0:
    print("\\nMajor studio films:")
    for _, row in major_studio_films.iterrows():
        print(f"  Title: {row['title'][:40]}{'...' if len(row['title']) > 40 else ''}")
        print(f"  Companies: {', '.join(row['production_companies_clean'])}")

# Example 2: Independent/unknown companies
independent_films = df[(df['has_production_companies']) & (~df['major_studio'])].head(2)
if len(independent_films) > 0:
    print("\\nIndependent/unknown production companies:")
    for _, row in independent_films.iterrows():
        print(f"  Title: {row['title'][:40]}{'...' if len(row['title']) > 40 else ''}")
        print(f"  Companies: {', '.join(row['production_companies_clean'])}")

# Example 3: No production companies
no_companies = df[~df['has_production_companies']].head(2)
if len(no_companies) > 0:
    print("\\nMovies with no production companies (placeholder):")
    for _, row in no_companies.iterrows():
        print(f"  Title: {row['title'][:40]}{'...' if len(row['title']) > 40 else ''}")
        print(f"  Companies: {row['production_companies_clean']}")

# 15. Check relationship with budget/revenue
print(f"\\n--- Quick Check: Production Companies vs Budget/Revenue ---")
if 'budget' in df.columns and 'revenue' in df.columns:
    print(f"\\nAverage budget by production company type:")
    
    # Major studio films
    if df['major_studio'].sum() > 0:
        major_budget = df[df['major_studio']]['budget'].mean()
        print(f"  Major studio films: ${major_budget:,.0f}")
    
    # Independent films
    independent_mask = df['has_production_companies'] & (~df['major_studio']) & (df['budget'] > 0)
    if independent_mask.sum() > 0:
        indie_budget = df[independent_mask]['budget'].mean()
        print(f"  Independent films: ${indie_budget:,.0f}")
    
    # Unknown/placeholder
    unknown_mask = ~df['has_production_companies'] & (df['budget'] > 0)
    if unknown_mask.sum() > 0:
        unknown_budget = df[unknown_mask]['budget'].mean()
        print(f"  Unknown companies: ${unknown_budget:,.0f}")

production_clean_time = time.time() - production_clean_start
print(f"\\nproduction_companies column cleaning completed in {production_clean_time:.2f} seconds")

Cleaning production_companies column...
Current data type of 'production_companies' column: object
\n--- Basic Statistics ---
Total rows: 590,202
Non-null count: 406,610
Null count: 183,592
Percentage null: 31.11%
\n--- Sample Values (first 10) ---
 1. Villealfa Filmproductions
 2. Villealfa Filmproductions
 3. Miramax, A Band Apart
 4. Largo Entertainment, JVC, Universal Pictures
 5. nan
 6. Lucasfilm Ltd., 20th Century Fox
 7. Pixar
 8. Paramount Pictures, The Steve Tisch Company, Wendy Finerman Productions
 9. DreamWorks Pictures, Jinks/Cohen Company
10. Mercury Productions, RKO Radio Pictures
\nCreated 'missing_production_companies' column: 183592 rows marked as True (31.1%)
\n--- Checking for Empty/Whitespace Values ---
Empty or whitespace-only values: 0 (0.00%)
\n--- Analyzing Production Companies Format ---
\nProduction companies format distribution:
  single_company      : 267,027 movies ( 45.2%)
  null_or_empty       : 183,592 movies ( 31.1%)
  comma_separated     : 139,373 mo

C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2825418719.py:287: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['production_companies_clean'] = df['production_companies_list'].apply(standardize_company_list)


Unique production companies after standardization: 158865
\nTop production companies after standardization:
  Unknown                                 : 183,593 movies ( 31.1%)
  Co                                      : 16,390 movies (  2.8%)
  Warner Bros.                            :  5,020 movies (  0.9%)
  Universal Pictures                      :  4,817 movies (  0.8%)
  Ltd                                     :  4,750 movies (  0.8%)
  BBC                                     :  4,294 movies (  0.7%)
  Inc                                     :  3,622 movies (  0.6%)
  Columbia Pictures                       :  3,343 movies (  0.6%)
  Paramount Pictures                      :  3,142 movies (  0.5%)
  Metro-Goldwyn-Mayer                     :  3,050 movies (  0.5%)
  20th Century Fox                        :  2,916 movies (  0.5%)
  ARTE                                    :  2,276 movies (  0.4%)
  ZDF                                     :  2,221 movies (  0.4%)
  Walt Disney Pictur

C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2825418719.py:314: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = df['production_companies_clean'].apply(lambda x: company in x)
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2825418719.py:314: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = df['production_companies_clean'].apply(lambda x: company in x)


  Created 'production_company_co' for company 'Co'
  Created 'production_company_warner_bros' for company 'Warner Bros.'


...

  Created 'production_company_columbia_pictures' for company 'Columbia Pictures'
  Created 'production_company_paramount_pictures' for company 'Paramount Pictures'


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2825418719.py:314: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = df['production_companies_clean'].apply(lambda x: company in x)


  Created 'production_company_metro-goldwyn-mayer' for company 'Metro-Goldwyn-Mayer'
Created binary columns for top 10 production companies


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2825418719.py:320: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_production_companies'] = df['production_companies_clean'].apply(lambda x: x != [placeholder] and len(x) > 0)
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2825418719.py:321: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['multiple_production_companies'] = df['production_companies_clean'].apply(lambda x: len(x) > 1)
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2825418719.py:322: PerformanceWarning: Da

\nCreated quality flags:
  - 'has_production_companies': 406,609 rows (68.9%)
  - 'multiple_production_companies': 138,249 rows (23.4%)
  - 'major_studio': 24,612 rows (4.2%)
\n--- production_companies Column Cleaning Summary ---
1. Original null/empty count: 183,592 (183592.0%)
2. Created 'missing_production_companies' column: 183,592 True values
3. Created 'has_production_companies' column: 406,609 True values
4. Created 'multiple_production_companies' column: 138,249 True values
5. Created 'major_studio' column: 24,612 True values
6. Current companies per movie: 1.4 average
7. Unique production companies found: 158,865
8. Movies with standardized companies: 590,202
\n--- Sample After Cleaning ---
First 10 rows with production company data:
    title_truncated                                companies_str  production_company_count  has_production_companies  major_studio
              Ariel                    Villealfa Filmproductions                         1                      True

In [27]:
# Clean production_countries Column
print("Cleaning production_countries column...")
production_countries_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'production_countries' column: {df['production_countries'].dtype}")
print(f"\\n--- Basic Statistics ---")
print(f"Total rows: {len(df):,}")
print(f"Non-null count: {df['production_countries'].notnull().sum():,}")
print(f"Null count: {df['production_countries'].isnull().sum():,}")
print(f"Percentage null: {df['production_countries'].isnull().sum()/len(df)*100:.2f}%")

# 2. Show sample values
print(f"\\n--- Sample Values (first 20) ---")
non_null_samples = df[df['production_countries'].notnull()]['production_countries'].head(20)
for i, countries in enumerate(non_null_samples, 1):
    countries_str = str(countries)[:100] + '...' if len(str(countries)) > 100 else str(countries)
    print(f"{i:2d}. {countries_str}")

# 3. Create a column to mark missing data before cleaning
df['missing_production_countries'] = df['production_countries'].isnull()
missing_count = df['missing_production_countries'].sum()
print(f"\\nCreated 'missing_production_countries' column: {missing_count} rows marked as True ({missing_count/len(df)*100:.1f}%)")

# 4. Analyze the format and structure
print(f"\\n--- Data Structure Analysis ---")

# Check if it's a string representation of a list/dictionary
def analyze_countries_format(countries_str):
    if pd.isna(countries_str):
        return 'null'
    
    countries_str = str(countries_str)
    
    # Check if it looks like JSON/list format
    if countries_str.startswith('[') or countries_str.startswith('{'):
        return 'json_like'
    elif ',' in countries_str and any(char.isalpha() for char in countries_str):
        return 'comma_separated'
    elif '|' in countries_str:
        return 'pipe_separated'
    else:
        return 'single_country'

format_counts = df['production_countries'].apply(analyze_countries_format).value_counts()
print(f"Format distribution:")
for fmt, count in format_counts.items():
    pct = count / len(df) * 100
    print(f"  {fmt:20s}: {count:8,d} rows ({pct:5.1f}%)")

# 5. Function to parse production countries
def parse_production_countries(countries_str):
    """
    Parse production countries from various string formats.
    Returns a list of country names.
    """
    if pd.isna(countries_str):
        return ["Unknown"]
    
    try:
        countries_str = str(countries_str).strip()
        
        # Handle empty string
        if countries_str == '':
            return ["Unknown"]
        
        # Try to parse as JSON if it looks like JSON
        if (countries_str.startswith('[') and countries_str.endswith(']')) or \
           (countries_str.startswith('{') and countries_str.endswith('}')):
            try:
                import json
                data = json.loads(countries_str)
                
                if isinstance(data, list):
                    # Extract country names from list of dictionaries
                    countries = []
                    for item in data:
                        if isinstance(item, dict):
                            if 'name' in item:
                                countries.append(item['name'])
                            elif 'iso_3166_1' in item:
                                countries.append(item['iso_3166_1'])
                        elif isinstance(item, str):
                            countries.append(item)
                    return countries if countries else ["Unknown"]
                
                elif isinstance(data, dict):
                    # Single dictionary
                    if 'name' in data:
                        return [data['name']]
                    elif 'iso_3166_1' in data:
                        return [data['iso_3166_1']]
                    else:
                        return ["Unknown"]
                
            except json.JSONDecodeError:
                pass
        
        # Try comma-separated
        if ',' in countries_str:
            # Check if it's a list of country names
            countries = [c.strip() for c in countries_str.split(',')]
            # Clean each country name
            cleaned_countries = []
            for country in countries:
                # Remove any quotes or brackets
                country = country.strip('[]{}"\' ')
                if country and country.lower() not in ['', 'nan', 'null', 'none']:
                    cleaned_countries.append(country)
            return cleaned_countries if cleaned_countries else ["Unknown"]
        
        # Try pipe-separated
        if '|' in countries_str:
            countries = [c.strip() for c in countries_str.split('|')]
            cleaned_countries = []
            for country in countries:
                country = country.strip('[]{}"\' ')
                if country and country.lower() not in ['', 'nan', 'null', 'none']:
                    cleaned_countries.append(country)
            return cleaned_countries if cleaned_countries else ["Unknown"]
        
        # Single country
        country = countries_str.strip('[]{}"\' ')
        if country and country.lower() not in ['', 'nan', 'null', 'none']:
            return [country]
        else:
            return ["Unknown"]
            
    except Exception as e:
        print(f"Error parsing countries string: {countries_str[:50]}... - {e}")
        return ["Unknown"]

# 6. Parse and clean the countries
print(f"\\n--- Parsing Production Countries ---")
df['production_countries_parsed'] = df['production_countries'].apply(parse_production_countries)

# Count unique countries
all_countries = []
for countries_list in df['production_countries_parsed']:
    all_countries.extend(countries_list)

unique_countries = set(all_countries)
print(f"Total unique country names found: {len(unique_countries)}")

# Show most common countries
from collections import Counter
country_counts = Counter(all_countries)
print(f"\\nTop 20 most common production countries:")
top_countries = country_counts.most_common(20)
for country, count in top_countries:
    pct = count / len(df) * 100
    print(f"  {country:40s}: {count:8,d} movies ({pct:5.1f}%)")

# 7. Create useful columns from parsed data
print(f"\\n--- Creating Derived Columns ---")

# Number of production countries
df['num_production_countries'] = df['production_countries_parsed'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Main production country (first one)
df['main_production_country'] = df['production_countries_parsed'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else "Unknown"
)

# Check for international co-productions
df['is_co_production'] = df['num_production_countries'] > 1
co_production_count = df['is_co_production'].sum()
print(f"Movies with co-productions (multiple countries): {co_production_count:,} ({co_production_count/len(df)*100:.1f}%)")

# Count by number of countries
print(f"\\nNumber of production countries per movie:")
num_countries_dist = df['num_production_countries'].value_counts().sort_index()
for num, count in num_countries_dist.head(10).items():
    pct = count / len(df) * 100
    print(f"  {num} country{'s' if num != 1 else ''}: {count:8,d} movies ({pct:5.1f}%)")

if len(num_countries_dist) > 10:
    others = num_countries_dist.iloc[10:].sum()
    print(f"  More than 10 countries: {others:8,d} movies ({others/len(df)*100:.1f}%)")

# 8. Standardize country names
print(f"\\n--- Standardizing Country Names ---")

# Common country name variations mapping
country_name_mapping = {
    # United States variations
    'United States of America': 'United States',
    'USA': 'United States',
    'US': 'United States',
    'United States': 'United States',
    'U.S.A.': 'United States',
    'U.S.': 'United States',
    
    # United Kingdom variations
    'United Kingdom': 'United Kingdom',
    'UK': 'United Kingdom',
    'U.K.': 'United Kingdom',
    'Great Britain': 'United Kingdom',
    'Britain': 'United Kingdom',
    'England': 'United Kingdom',
    
    # Other common variations
    'South Korea': 'South Korea',
    'Korea': 'South Korea',
    'Republic of Korea': 'South Korea',
    
    'North Korea': 'North Korea',
    "Democratic People's Republic of Korea": 'North Korea',
    
    'Russia': 'Russia',
    'Russian Federation': 'Russia',
    
    'China': 'China',
    "People's Republic of China": 'China',
    'PR China': 'China',
    
    'Taiwan': 'Taiwan',
    'Republic of China': 'Taiwan',
    'Taiwan, Province of China': 'Taiwan',
    
    'Hong Kong': 'Hong Kong',
    'Hong Kong SAR': 'Hong Kong',
    
    'Iran': 'Iran',
    'Islamic Republic of Iran': 'Iran',
    
    'Syria': 'Syria',
    'Syrian Arab Republic': 'Syria',
    
    'Vietnam': 'Vietnam',
    'Viet Nam': 'Vietnam',
    
    'Czech Republic': 'Czech Republic',
    'Czechia': 'Czech Republic',
    
    'Macedonia': 'North Macedonia',
    'North Macedonia': 'North Macedonia',
    
    'Burma': 'Myanmar',
    'Myanmar': 'Myanmar',
    
    'Swaziland': 'Eswatini',
    'Eswatini': 'Eswatini',
    
    # Common abbreviations to full names
    'FRG': 'Germany',
    'GDR': 'Germany',
    'West Germany': 'Germany',
    'East Germany': 'Germany',
    'BRD': 'Germany',
    'DDR': 'Germany',
    
    'USSR': 'Soviet Union',
    'Soviet Union': 'Soviet Union',
    'U.S.S.R.': 'Soviet Union',
    
    'Yugoslavia': 'Yugoslavia',
    'SFR Yugoslavia': 'Yugoslavia',
}

def standardize_country_name(country_name):
    """Standardize country name using mapping"""
    if pd.isna(country_name):
        return 'Unknown'
    
    country_name = str(country_name).strip()
    
    # Check exact match
    if country_name in country_name_mapping:
        return country_name_mapping[country_name]
    
    # Check case-insensitive match
    country_lower = country_name.lower()
    for key, value in country_name_mapping.items():
        if key.lower() == country_lower:
            return value
    
    # Try partial match for common patterns
    if 'united states' in country_lower:
        return 'United States'
    elif 'united kingdom' in country_lower:
        return 'United Kingdom'
    elif 'south korea' in country_lower or 'republic of korea' in country_lower:
        return 'South Korea'
    elif 'north korea' in country_lower:
        return 'North Korea'
    elif 'soviet' in country_lower or 'ussr' in country_lower:
        return 'Soviet Union'
    elif 'china' in country_lower and 'taiwan' not in country_lower:
        return 'China'
    elif 'taiwan' in country_lower:
        return 'Taiwan'
    
    # Return title case for consistency
    return country_name.title()

# Apply standardization
df['production_countries_standardized'] = df['production_countries_parsed'].apply(
    lambda countries: [standardize_country_name(c) for c in countries] if isinstance(countries, list) else ["Unknown"]
)

# Update derived columns with standardized names
df['num_production_countries_clean'] = df['production_countries_standardized'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

df['main_production_country_clean'] = df['production_countries_standardized'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else "Unknown"
)

# 9. Create quality flags
print(f"\\n--- Creating Quality Flags ---")

df['has_production_country'] = df['production_countries'].notnull()
df['valid_production_country'] = df['main_production_country_clean'] != "Unknown"
df['multiple_production_countries'] = df['num_production_countries_clean'] > 1

print(f"Created quality flags:")
print(f"  - 'has_production_country': {df['has_production_country'].sum():,} rows ({df['has_production_country'].sum()/len(df)*100:.1f}%)")
print(f"  - 'valid_production_country': {df['valid_production_country'].sum():,} rows ({df['valid_production_country'].sum()/len(df)*100:.1f}%)")
print(f"  - 'multiple_production_countries': {df['multiple_production_countries'].sum():,} rows ({df['multiple_production_countries'].sum()/len(df)*100:.1f}%)")

# 10. Analyze standardized country distribution
print(f"\\n--- Standardized Country Distribution ---")

# Get all standardized country names
all_std_countries = []
for countries_list in df['production_countries_standardized']:
    all_std_countries.extend(countries_list)

std_country_counts = Counter(all_std_countries)
print(f"\\nTop 20 standardized production countries:")
top_std_countries = std_country_counts.most_common(20)
for country, count in top_std_countries:
    pct = count / len(df) * 100
    print(f"  {country:40s}: {count:8,d} movies ({pct:5.1f}%)")

# 11. Replace original column with cleaned data (as string)
print(f"\\n--- Updating Original Column ---")

# Convert cleaned list to readable string
df['production_countries'] = df['production_countries_standardized'].apply(
    lambda countries: ', '.join(countries) if isinstance(countries, list) else "Unknown"
)

print(f"Updated 'production_countries' column with cleaned data")

# 12. Verify the cleaning
print("\\n--- production_countries Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count:,} ({missing_count/len(df)*100:.1f}%)")
print(f"2. Created 'missing_production_countries' column: {missing_count:,} True values")
print(f"3. Created 'has_production_country' column: {df['has_production_country'].sum():,} True values")
print(f"4. Created 'valid_production_country' column: {df['valid_production_country'].sum():,} True values")
print(f"5. Created 'multiple_production_countries' column: {df['multiple_production_countries'].sum():,} True values")
print(f"6. Current null count: {df['production_countries'].isnull().sum()}")
print(f"7. Current data type: {df['production_countries'].dtype}")
print(f"8. Unique standardized countries: {len(set(all_std_countries)):,}")
print(f"9. Movies with co-productions: {co_production_count:,} ({co_production_count/len(df)*100:.1f}%)")

# 13. Sample after cleaning
print("\\n--- Sample After Cleaning ---")
print("First 15 rows with production countries data:")

sample_cols = [
    'title', 
    'production_countries', 
    'num_production_countries_clean',
    'main_production_country_clean',
    'multiple_production_countries'
]

sample_df = df[sample_cols].head(15).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
sample_df['countries_truncated'] = sample_df['production_countries'].apply(lambda x: x[:30] + '...' if len(x) > 30 else x)

display_cols = [
    'title_truncated', 
    'countries_truncated',
    'num_production_countries_clean', 
    'main_production_country_clean',
    'multiple_production_countries'
]

print(sample_df[display_cols].to_string(index=False))

# 14. Additional analysis
print(f"\\n--- Additional Analysis ---")

# Country relationships with other variables
if 'revenue' in df.columns:
    # Top countries by average revenue
    print(f"\\nTop 10 countries by average revenue (movies with revenue > 0):")
    revenue_by_country = df[df['revenue'] > 0].groupby('main_production_country_clean')['revenue'].mean().sort_values(ascending=False).head(10)
    for country, avg_revenue in revenue_by_country.items():
        print(f"  {country:30s}: ${avg_revenue:,.0f}")

if 'vote_average' in df.columns and (df['vote_count'] > 0).any():
    # Top countries by average rating
    print(f"\\nTop 10 countries by average rating (movies with votes > 0):")
    rated_movies = df[df['vote_count'] > 0]
    rating_by_country = rated_movies.groupby('main_production_country_clean')['vote_average'].mean().sort_values(ascending=False).head(10)
    for country, avg_rating in rating_by_country.items():
        print(f"  {country:30s}: {avg_rating:.2f}")

# Temporal trends
if 'release_year' in df.columns:
    print(f"\\n--- Temporal Trends by Decade ---")
    
    # Calculate decade
    df['decade'] = (df['release_year'] // 10) * 10
    
    recent_decades = [1980, 1990, 2000, 2010, 2020]
    for decade in recent_decades:
        decade_movies = df[df['decade'] == decade]
        if len(decade_movies) > 0:
            top_countries_decade = Counter()
            for countries in decade_movies['production_countries_standardized']:
                if isinstance(countries, list):
                    top_countries_decade.update(countries)
            
            print(f"\\n{decade}s - Top 5 production countries:")
            top_5 = top_countries_decade.most_common(5)
            for country, count in top_5:
                pct = count / len(decade_movies) * 100
                print(f"  {country:30s}: {count:6,d} movies ({pct:5.1f}%)")

# Clean up temporary columns (optional)
# Uncomment the following line to remove temporary columns:
# df = df.drop(columns=['production_countries_parsed', 'production_countries_standardized', 'num_production_countries'])

production_countries_clean_time = time.time() - production_countries_clean_start
print(f"\\nproduction_countries column cleaning completed in {production_countries_clean_time:.2f} seconds")

Cleaning production_countries column...
Current data type of 'production_countries' column: object
\n--- Basic Statistics ---
Total rows: 590,202
Non-null count: 486,076
Null count: 104,126
Percentage null: 17.64%
\n--- Sample Values (first 20) ---
 1. Finland
 2. Finland
 3. United States of America
 4. United States of America
 5. Germany
 6. United States of America
 7. United States of America
 8. United States of America
 9. United States of America
10. United States of America
11. Denmark, Finland, France, Germany, Iceland, Netherlands, Norway, Sweden
12. Germany, United Kingdom
13. France
14. Germany
15. Canada, Spain
16. United States of America
17. United States of America
18. United States of America
19. United Kingdom, United States of America
20. Israel, Sweden
\nCreated 'missing_production_countries' column: 104126 rows marked as True (17.6%)
\n--- Data Structure Analysis ---


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\4117517964.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['missing_production_countries'] = df['production_countries'].isnull()


Format distribution:
  single_country      :  441,644 rows ( 74.8%)
  null                :  104,126 rows ( 17.6%)
  comma_separated     :   44,432 rows (  7.5%)
\n--- Parsing Production Countries ---


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\4117517964.py:135: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['production_countries_parsed'] = df['production_countries'].apply(parse_production_countries)


Total unique country names found: 245
\nTop 20 most common production countries:
  United States of America                :  135,917 movies ( 23.0%)
  Unknown                                 :  104,126 movies ( 17.6%)
  France                                  :   35,896 movies (  6.1%)
  United Kingdom                          :   34,265 movies (  5.8%)
  Germany                                 :   31,953 movies (  5.4%)
  Japan                                   :   24,432 movies (  4.1%)
  India                                   :   20,689 movies (  3.5%)
  Canada                                  :   17,941 movies (  3.0%)
  Italy                                   :   17,170 movies (  2.9%)
  Spain                                   :   13,852 movies (  2.3%)
  Brazil                                  :   13,329 movies (  2.3%)
  Soviet Union                            :    9,959 movies (  1.7%)
  Mexico                                  :    9,342 movies (  1.6%)
  China               

...

Movies with co-productions (multiple countries): 44,432 (7.5%)
\nNumber of production countries per movie:
  1 country:  545,770 movies ( 92.5%)
  2 countrys:   33,998 movies (  5.8%)
  3 countrys:    7,228 movies (  1.2%)
  4 countrys:    2,090 movies (  0.4%)
  5 countrys:      701 movies (  0.1%)
  6 countrys:      233 movies (  0.0%)
  7 countrys:       96 movies (  0.0%)
  8 countrys:       48 movies (  0.0%)
  9 countrys:       12 movies (  0.0%)
  10 countrys:        7 movies (  0.0%)
  More than 10 countries:       19 movies (0.0%)
\n--- Standardizing Country Names ---


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\4117517964.py:298: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['production_countries_standardized'] = df['production_countries_parsed'].apply(
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\4117517964.py:303: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['num_production_countries_clean'] = df['production_countries_standardized'].apply(
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\4117517964.py:307: PerformanceWarning: DataFrame is highly fragmented.  This is usually 

\n--- Creating Quality Flags ---
Created quality flags:
  - 'has_production_country': 486,076 rows (82.4%)
  - 'valid_production_country': 486,076 rows (82.4%)
  - 'multiple_production_countries': 44,432 rows (7.5%)
\n--- Standardized Country Distribution ---


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\4117517964.py:314: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_production_country'] = df['production_countries'].notnull()
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\4117517964.py:315: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['valid_production_country'] = df['main_production_country_clean'] != "Unknown"
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\4117517964.py:316: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calli

\nTop 20 standardized production countries:
  United States                           :  135,917 movies ( 23.0%)
  Unknown                                 :  104,126 movies ( 17.6%)
  France                                  :   35,896 movies (  6.1%)
  United Kingdom                          :   34,265 movies (  5.8%)
  Germany                                 :   33,475 movies (  5.7%)
  Japan                                   :   24,432 movies (  4.1%)
  India                                   :   20,689 movies (  3.5%)
  Canada                                  :   17,941 movies (  3.0%)
  Italy                                   :   17,170 movies (  2.9%)
  Spain                                   :   13,852 movies (  2.3%)
  Brazil                                  :   13,329 movies (  2.3%)
  Soviet Union                            :    9,959 movies (  1.7%)
  Mexico                                  :    9,342 movies (  1.6%)
  China                                   :    8,494 movies

In [28]:
# Clean spoken_languages Column
print("Cleaning spoken_languages column...")
spoken_languages_clean_start = time.time()

# 1. Check current data type before changing
print(f"Current data type of 'spoken_languages' column: {df['spoken_languages'].dtype}")
print(f"\\n--- Basic Statistics ---")
print(f"Total rows: {len(df):,}")
print(f"Non-null count: {df['spoken_languages'].notnull().sum():,}")
print(f"Null count: {df['spoken_languages'].isnull().sum():,}")
print(f"Percentage null: {df['spoken_languages'].isnull().sum()/len(df)*100:.2f}%")

# 2. Show sample values
print(f"\\n--- Sample Values (first 20) ---")
non_null_samples = df[df['spoken_languages'].notnull()]['spoken_languages'].head(20)
for i, languages in enumerate(non_null_samples, 1):
    languages_str = str(languages)[:100] + '...' if len(str(languages)) > 100 else str(languages)
    print(f"{i:2d}. {languages_str}")

# 3. Create a column to mark missing data before cleaning
df['missing_spoken_languages'] = df['spoken_languages'].isnull()
missing_count = df['missing_spoken_languages'].sum()
print(f"\\nCreated 'missing_spoken_languages' column: {missing_count} rows marked as True ({missing_count/len(df)*100:.1f}%)")

# 4. Analyze the format and structure
print(f"\\n--- Data Structure Analysis ---")

# Check if it's a string representation of a list/dictionary
def analyze_languages_format(languages_str):
    if pd.isna(languages_str):
        return 'null'
    
    languages_str = str(languages_str)
    
    # Check if it looks like JSON/list format
    if languages_str.startswith('[') or languages_str.startswith('{'):
        return 'json_like'
    elif ',' in languages_str and any(char.isalpha() for char in languages_str):
        return 'comma_separated'
    elif '|' in languages_str:
        return 'pipe_separated'
    else:
        return 'single_language'

format_counts = df['spoken_languages'].apply(analyze_languages_format).value_counts()
print(f"Format distribution:")
for fmt, count in format_counts.items():
    pct = count / len(df) * 100
    print(f"  {fmt:20s}: {count:8,d} rows ({pct:5.1f}%)")

# 5. Function to parse spoken languages
def parse_spoken_languages(languages_str):
    """
    Parse spoken languages from various string formats.
    Returns a list of language names.
    """
    if pd.isna(languages_str):
        return ["Unknown"]
    
    try:
        languages_str = str(languages_str).strip()
        
        # Handle empty string
        if languages_str == '':
            return ["Unknown"]
        
        # Try to parse as JSON if it looks like JSON
        if (languages_str.startswith('[') and languages_str.endswith(']')) or \
           (languages_str.startswith('{') and languages_str.endswith('}')):
            try:
                import json
                data = json.loads(languages_str)
                
                if isinstance(data, list):
                    # Extract language names from list of dictionaries
                    languages = []
                    for item in data:
                        if isinstance(item, dict):
                            if 'name' in item:
                                languages.append(item['name'])
                            elif 'iso_639_1' in item:
                                languages.append(item['iso_639_1'])
                            elif 'english_name' in item:
                                languages.append(item['english_name'])
                        elif isinstance(item, str):
                            languages.append(item)
                    return languages if languages else ["Unknown"]
                
                elif isinstance(data, dict):
                    # Single dictionary
                    if 'name' in data:
                        return [data['name']]
                    elif 'iso_639_1' in data:
                        return [data['iso_639_1']]
                    elif 'english_name' in data:
                        return [data['english_name']]
                    else:
                        return ["Unknown"]
                
            except json.JSONDecodeError:
                pass
        
        # Try comma-separated
        if ',' in languages_str:
            # Check if it's a list of language names
            languages = [lang.strip() for lang in languages_str.split(',')]
            # Clean each language name
            cleaned_languages = []
            for language in languages:
                # Remove any quotes or brackets
                language = language.strip('[]{}"\' ')
                if language and language.lower() not in ['', 'nan', 'null', 'none']:
                    cleaned_languages.append(language)
            return cleaned_languages if cleaned_languages else ["Unknown"]
        
        # Try pipe-separated
        if '|' in languages_str:
            languages = [lang.strip() for lang in languages_str.split('|')]
            cleaned_languages = []
            for language in languages:
                language = language.strip('[]{}"\' ')
                if language and language.lower() not in ['', 'nan', 'null', 'none']:
                    cleaned_languages.append(language)
            return cleaned_languages if cleaned_languages else ["Unknown"]
        
        # Single language
        language = languages_str.strip('[]{}"\' ')
        if language and language.lower() not in ['', 'nan', 'null', 'none']:
            return [language]
        else:
            return ["Unknown"]
            
    except Exception as e:
        print(f"Error parsing languages string: {languages_str[:50]}... - {e}")
        return ["Unknown"]

# 6. Parse and clean the languages
print(f"\\n--- Parsing Spoken Languages ---")
df['spoken_languages_parsed'] = df['spoken_languages'].apply(parse_spoken_languages)

# Count unique languages
all_languages = []
for languages_list in df['spoken_languages_parsed']:
    all_languages.extend(languages_list)

unique_languages = set(all_languages)
print(f"Total unique language names found: {len(unique_languages)}")

# Show most common languages
from collections import Counter
language_counts = Counter(all_languages)
print(f"\\nTop 20 most common spoken languages:")
top_languages = language_counts.most_common(20)
for language, count in top_languages:
    pct = count / len(df) * 100
    print(f"  {language:40s}: {count:8,d} movies ({pct:5.1f}%)")

# 7. Create useful columns from parsed data
print(f"\\n--- Creating Derived Columns ---")

# Number of spoken languages
df['num_spoken_languages'] = df['spoken_languages_parsed'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Main spoken language (first one)
df['main_spoken_language'] = df['spoken_languages_parsed'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else "Unknown"
)

# Check for multilingual films
df['is_multilingual'] = df['num_spoken_languages'] > 1
multilingual_count = df['is_multilingual'].sum()
print(f"Multilingual films (multiple languages): {multilingual_count:,} ({multilingual_count/len(df)*100:.1f}%)")

# Count by number of languages
print(f"\\nNumber of spoken languages per movie:")
num_languages_dist = df['num_spoken_languages'].value_counts().sort_index()
for num, count in num_languages_dist.head(10).items():
    pct = count / len(df) * 100
    print(f"  {num} language{'s' if num != 1 else ''}: {count:8,d} movies ({pct:5.1f}%)")

if len(num_languages_dist) > 10:
    others = num_languages_dist.iloc[10:].sum()
    print(f"  More than 10 languages: {others:8,d} movies ({others/len(df)*100:.1f}%)")

# 8. Standardize language names
print(f"\\n--- Standardizing Language Names ---")

# Comprehensive language name standardization
language_name_mapping = {
    # English variations
    'English': 'English',
    'Englisch': 'English',
    'Anglais': 'English',
    'Inglés': 'English',
    'Inglese': 'English',
    'inglês': 'English',
    'английский': 'English',
    '英语': 'English',
    '英語': 'English',
    
    # Spanish variations
    'Spanish': 'Spanish',
    'Español': 'Spanish',
    'Spanish; Castilian': 'Spanish',
    'Espagnol': 'Spanish',
    'Spanisch': 'Spanish',
    'Spagnolo': 'Spanish',
    
    # French variations
    'French': 'French',
    'Français': 'French',
    'Französisch': 'French',
    'Francese': 'French',
    'Francés': 'French',
    
    # German variations
    'German': 'German',
    'Deutsch': 'German',
    'Allemand': 'German',
    'Tedesco': 'German',
    'Alemán': 'German',
    
    # Italian variations
    'Italian': 'Italian',
    'Italiano': 'Italian',
    'Italienisch': 'Italian',
    'Italien': 'Italian',
    'Italiana': 'Italian',
    
    # Japanese variations
    'Japanese': 'Japanese',
    '日本語': 'Japanese',
    'Japonés': 'Japanese',
    'Giapponese': 'Japanese',
    'Japanisch': 'Japanese',
    
    # Chinese variations
    'Chinese': 'Chinese',
    '中文': 'Chinese',
    'Chino': 'Chinese',
    'Cinese': 'Chinese',
    'Chinesisch': 'Chinese',
    'Mandarin': 'Chinese',
    'Cantonese': 'Chinese',
    
    # Korean variations
    'Korean': 'Korean',
    '한국어/조선말': 'Korean',
    'Coreano': 'Korean',
    'Coréen': 'Korean',
    'Coreano': 'Korean',
    
    # Russian variations
    'Russian': 'Russian',
    'Pусский': 'Russian',
    'Ruso': 'Russian',
    'Russe': 'Russian',
    'Russo': 'Russian',
    
    # Hindi variations
    'Hindi': 'Hindi',
    'हिन्दी': 'Hindi',
    'Hindú': 'Hindi',
    
    # Portuguese variations
    'Portuguese': 'Portuguese',
    'Português': 'Portuguese',
    'Portugais': 'Portuguese',
    'Portugiesisch': 'Portuguese',
    'Portoghese': 'Portuguese',
    
    # Arabic variations
    'Arabic': 'Arabic',
    'العربية': 'Arabic',
    'Arabe': 'Arabic',
    'Arabisch': 'Arabic',
    'Arabo': 'Arabic',
    
    # Turkish variations
    'Turkish': 'Turkish',
    'Türkçe': 'Turkish',
    'Turco': 'Turkish',
    'Turc': 'Turkish',
    'Turco': 'Turkish',
    
    # Other common languages
    'Dutch': 'Dutch',
    'Nederlands': 'Dutch',
    'Hollandais': 'Dutch',
    'Olandese': 'Dutch',
    
    'Swedish': 'Swedish',
    'Svenska': 'Swedish',
    'Suédois': 'Swedish',
    'Svedese': 'Swedish',
    
    'Danish': 'Danish',
    'Dansk': 'Danish',
    'Danois': 'Danish',
    'Danese': 'Danish',
    
    'Norwegian': 'Norwegian',
    'Norsk': 'Norwegian',
    'Norvégien': 'Norwegian',
    'Norvegese': 'Norwegian',
    
    'Finnish': 'Finnish',
    'Suomi': 'Finnish',
    'Finnois': 'Finnish',
    'Finlandese': 'Finnish',
    
    'Polish': 'Polish',
    'Polski': 'Polish',
    'Polonais': 'Polish',
    'Polacco': 'Polish',
    
    'Czech': 'Czech',
    'Čeština': 'Czech',
    'Tchèque': 'Czech',
    'Ceco': 'Czech',
    
    'Hungarian': 'Hungarian',
    'Magyar': 'Hungarian',
    'Hongrois': 'Hungarian',
    'Ungherese': 'Hungarian',
    
    'Greek': 'Greek',
    'Ελληνικά': 'Greek',
    'Grec': 'Greek',
    'Greco': 'Greek',
    
    # Special cases
    'None': 'No Dialogue',
    'No Dialogue': 'No Dialogue',
    'Silent': 'No Dialogue',
    'Silent film': 'No Dialogue',
    'Mute': 'No Dialogue',
    
    'Multiple languages': 'Multiple Languages',
    'Various': 'Multiple Languages',
    'Multilingual': 'Multiple Languages',
    
    'Unknown': 'Unknown',
    '': 'Unknown',
    'nan': 'Unknown',
}

def standardize_language_name(language_name):
    """Standardize language name using mapping"""
    if pd.isna(language_name):
        return 'Unknown'
    
    language_name = str(language_name).strip()
    
    # Check exact match
    if language_name in language_name_mapping:
        return language_name_mapping[language_name]
    
    # Check case-insensitive match
    language_lower = language_name.lower()
    for key, value in language_name_mapping.items():
        if key.lower() == language_lower:
            return value
    
    # Try to match common patterns
    if 'english' in language_lower:
        return 'English'
    elif 'spanish' in language_lower:
        return 'Spanish'
    elif 'french' in language_lower:
        return 'French'
    elif 'german' in language_lower:
        return 'German'
    elif 'italian' in language_lower:
        return 'Italian'
    elif 'japanese' in language_lower:
        return 'Japanese'
    elif 'chinese' in language_lower:
        return 'Chinese'
    elif 'korean' in language_lower:
        return 'Korean'
    elif 'russian' in language_lower:
        return 'Russian'
    elif 'hindi' in language_lower:
        return 'Hindi'
    elif 'portuguese' in language_lower:
        return 'Portuguese'
    elif 'arabic' in language_lower:
        return 'Arabic'
    elif 'no dialogue' in language_lower or 'silent' in language_lower:
        return 'No Dialogue'
    elif 'multiple' in language_lower or 'various' in language_lower:
        return 'Multiple Languages'
    
    # Return title case for consistency
    return language_name.title()

# Apply standardization
df['spoken_languages_standardized'] = df['spoken_languages_parsed'].apply(
    lambda languages: [standardize_language_name(lang) for lang in languages] if isinstance(languages, list) else ["Unknown"]
)

# Update derived columns with standardized names
df['num_spoken_languages_clean'] = df['spoken_languages_standardized'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

df['main_spoken_language_clean'] = df['spoken_languages_standardized'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else "Unknown"
)

# 9. Create quality flags
print(f"\\n--- Creating Quality Flags ---")

df['has_spoken_languages'] = df['spoken_languages'].notnull()
df['valid_spoken_language'] = df['main_spoken_language_clean'] != "Unknown"
df['is_multilingual_clean'] = df['num_spoken_languages_clean'] > 1
df['has_no_dialogue'] = df['main_spoken_language_clean'] == "No Dialogue"

print(f"Created quality flags:")
print(f"  - 'has_spoken_languages': {df['has_spoken_languages'].sum():,} rows ({df['has_spoken_languages'].sum()/len(df)*100:.1f}%)")
print(f"  - 'valid_spoken_language': {df['valid_spoken_language'].sum():,} rows ({df['valid_spoken_language'].sum()/len(df)*100:.1f}%)")
print(f"  - 'is_multilingual_clean': {df['is_multilingual_clean'].sum():,} rows ({df['is_multilingual_clean'].sum()/len(df)*100:.1f}%)")
print(f"  - 'has_no_dialogue': {df['has_no_dialogue'].sum():,} rows ({df['has_no_dialogue'].sum()/len(df)*100:.1f}%)")

# 10. Analyze standardized language distribution
print(f"\\n--- Standardized Language Distribution ---")

# Get all standardized language names
all_std_languages = []
for languages_list in df['spoken_languages_standardized']:
    all_std_languages.extend(languages_list)

std_language_counts = Counter(all_std_languages)
print(f"\\nTop 20 standardized spoken languages:")
top_std_languages = std_language_counts.most_common(20)
for language, count in top_std_languages:
    pct = count / len(df) * 100
    print(f"  {language:40s}: {count:8,d} movies ({pct:5.1f}%)")

# 11. Replace original column with cleaned data (as string)
print(f"\\n--- Updating Original Column ---")

# Convert cleaned list to readable string
df['spoken_languages'] = df['spoken_languages_standardized'].apply(
    lambda languages: ', '.join(languages) if isinstance(languages, list) else "Unknown"
)

print(f"Updated 'spoken_languages' column with cleaned data")

# 12. Verify the cleaning
print("\\n--- spoken_languages Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count:,} ({missing_count/len(df)*100:.1f}%)")
print(f"2. Created 'missing_spoken_languages' column: {missing_count:,} True values")
print(f"3. Created 'has_spoken_languages' column: {df['has_spoken_languages'].sum():,} True values")
print(f"4. Created 'valid_spoken_language' column: {df['valid_spoken_language'].sum():,} True values")
print(f"5. Created 'is_multilingual_clean' column: {df['is_multilingual_clean'].sum():,} True values")
print(f"6. Created 'has_no_dialogue' column: {df['has_no_dialogue'].sum():,} True values")
print(f"7. Current null count: {df['spoken_languages'].isnull().sum()}")
print(f"8. Current data type: {df['spoken_languages'].dtype}")
print(f"9. Unique standardized languages: {len(set(all_std_languages)):,}")
print(f"10. Multilingual films: {multilingual_count:,} ({multilingual_count/len(df)*100:.1f}%)")
print(f"11. Silent/no dialogue films: {df['has_no_dialogue'].sum():,} ({df['has_no_dialogue'].sum()/len(df)*100:.1f}%)")

# 13. Sample after cleaning
print("\\n--- Sample After Cleaning ---")
print("First 15 rows with spoken languages data:")

sample_cols = [
    'title', 
    'spoken_languages', 
    'num_spoken_languages_clean',
    'main_spoken_language_clean',
    'is_multilingual_clean',
    'has_no_dialogue'
]

sample_df = df[sample_cols].head(15).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
sample_df['languages_truncated'] = sample_df['spoken_languages'].apply(lambda x: x[:30] + '...' if len(x) > 30 else x)

display_cols = [
    'title_truncated', 
    'languages_truncated',
    'num_spoken_languages_clean', 
    'main_spoken_language_clean',
    'is_multilingual_clean',
    'has_no_dialogue'
]

print(sample_df[display_cols].to_string(index=False))

# 14. Additional analysis
print(f"\\n--- Additional Analysis ---")

# Language relationships with other variables
if 'revenue' in df.columns and (df['revenue'] > 0).any():
    # Top languages by average revenue
    print(f"\\nTop 10 languages by average revenue (movies with revenue > 0):")
    revenue_by_language = df[df['revenue'] > 0].groupby('main_spoken_language_clean')['revenue'].mean().sort_values(ascending=False).head(10)
    for language, avg_revenue in revenue_by_language.items():
        print(f"  {language:30s}: ${avg_revenue:,.0f}")

if 'vote_average' in df.columns and (df['vote_count'] > 0).any():
    # Top languages by average rating
    print(f"\\nTop 10 languages by average rating (movies with votes > 0):")
    rated_movies = df[df['vote_count'] > 0]
    rating_by_language = rated_movies.groupby('main_spoken_language_clean')['vote_average'].mean().sort_values(ascending=False).head(10)
    for language, avg_rating in rating_by_language.items():
        print(f"  {language:30s}: {avg_rating:.2f}")

# Temporal trends
if 'release_year' in df.columns:
    print(f"\\n--- Temporal Trends by Decade ---")
    
    # Calculate decade
    df['decade'] = (df['release_year'] // 10) * 10
    
    recent_decades = [1980, 1990, 2000, 2010, 2020]
    for decade in recent_decades:
        decade_movies = df[df['decade'] == decade]
        if len(decade_movies) > 0:
            top_languages_decade = Counter()
            for languages in decade_movies['spoken_languages_standardized']:
                if isinstance(languages, list):
                    top_languages_decade.update(languages)
            
            print(f"\\n{decade}s - Top 5 spoken languages:")
            top_5 = top_languages_decade.most_common(5)
            for language, count in top_5:
                pct = count / len(decade_movies) * 100
                print(f"  {language:30s}: {count:6,d} movies ({pct:5.1f}%)")

# Relationship with production countries
if 'main_production_country_clean' in df.columns:
    print(f"\\n--- Language-Country Relationships ---")
    
    # Most common language for top production countries
    top_countries = ['United States', 'United Kingdom', 'France', 'Germany', 'Japan']
    
    for country in top_countries:
        country_movies = df[df['main_production_country_clean'] == country]
        if len(country_movies) > 0:
            main_language = country_movies['main_spoken_language_clean'].mode()
            if not main_language.empty:
                lang = main_language.iloc[0]
                count = (country_movies['main_spoken_language_clean'] == lang).sum()
                pct = count / len(country_movies) * 100
                print(f"  {country:25s}: {lang} ({pct:.1f}% of movies)")

# Multilingual analysis
print(f"\\n--- Multilingual Film Analysis ---")
if multilingual_count > 0:
    # Most common language combinations
    print(f"\\nMost common bilingual combinations:")
    
    # Get all bilingual movies
    bilingual_movies = df[df['num_spoken_languages_clean'] == 2]
    if len(bilingual_movies) > 0:
        combinations = Counter()
        for languages in bilingual_movies['spoken_languages_standardized']:
            if isinstance(languages, list) and len(languages) == 2:
                combo = tuple(sorted(languages))
                combinations[combo] += 1
        
        top_combos = combinations.most_common(5)
        for combo, count in top_combos:
            pct = count / len(bilingual_movies) * 100
            print(f"  {combo[0]} + {combo[1]}: {count:,} movies ({pct:.1f}%)")

# Clean up temporary columns (optional)
# Uncomment the following lines to remove temporary columns:
# temp_cols = ['spoken_languages_parsed', 'spoken_languages_standardized', 
#              'num_spoken_languages', 'main_spoken_language', 'decade']
# df = df.drop(columns=[col for col in temp_cols if col in df.columns])

spoken_languages_clean_time = time.time() - spoken_languages_clean_start
print(f"\\nspoken_languages column cleaning completed in {spoken_languages_clean_time:.2f} seconds")

Cleaning spoken_languages column...
Current data type of 'spoken_languages' column: object
\n--- Basic Statistics ---
Total rows: 590,202
Non-null count: 480,046
Null count: 110,156
Percentage null: 18.66%
\n--- Sample Values (first 20) ---
 1. suomi
 2. svenska, suomi, English
 3. English
 4. English
 5. Deutsch
 6. English
 7. English
 8. English
 9. English
10. English
11. English
12. Cymraeg, English
13. English, svenska, Deutsch
14. No Language
15. English
16. English
17. English
18. English, 日本語, Français
19. English, Español, العربية, Latin
20. Deutsch, العربية, עִבְרִית, English, Italiano, Türkçe
\nCreated 'missing_spoken_languages' column: 110156 rows marked as True (18.7%)
\n--- Data Structure Analysis ---


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2193510167.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['missing_spoken_languages'] = df['spoken_languages'].isnull()


Format distribution:
  single_language     :  429,984 rows ( 72.9%)
  null                :  110,156 rows ( 18.7%)
  comma_separated     :   50,062 rows (  8.5%)
\n--- Parsing Spoken Languages ---


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2193510167.py:139: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['spoken_languages_parsed'] = df['spoken_languages'].apply(parse_spoken_languages)


Total unique language names found: 91
\nTop 20 most common spoken languages:
  English                                 :  197,195 movies ( 33.4%)
  Unknown                                 :  110,171 movies ( 18.7%)
  Français                                :   39,482 movies (  6.7%)
  Español                                 :   39,420 movies (  6.7%)
  Deutsch                                 :   29,040 movies (  4.9%)
  日本語                                     :   25,466 movies (  4.3%)
  No Language                             :   18,702 movies (  3.2%)
  Pусский                                 :   18,346 movies (  3.1%)
  Italiano                                :   18,108 movies (  3.1%)
  Português                               :   17,033 movies (  2.9%)
  普通话                                     :   15,000 movies (  2.5%)
  한국어/조선말                                 :    9,044 movies (  1.5%)
  العربية                                 :    7,860 movies (  1.3%)
  Český                   

...

Multilingual films (multiple languages): 46,080 (7.8%)
\nNumber of spoken languages per movie:
  1 language:  544,122 movies ( 92.2%)
  2 languages:   34,703 movies (  5.9%)
  3 languages:    8,126 movies (  1.4%)
  4 languages:    2,220 movies (  0.4%)
  5 languages:      675 movies (  0.1%)
  6 languages:      221 movies (  0.0%)
  7 languages:       76 movies (  0.0%)
  8 languages:       24 movies (  0.0%)
  9 languages:       11 movies (  0.0%)
  10 languages:       12 movies (  0.0%)
  More than 10 languages:       12 movies (0.0%)
\n--- Standardizing Language Names ---


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2193510167.py:399: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['spoken_languages_standardized'] = df['spoken_languages_parsed'].apply(
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2193510167.py:404: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['num_spoken_languages_clean'] = df['spoken_languages_standardized'].apply(
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2193510167.py:408: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of ca

\n--- Creating Quality Flags ---
Created quality flags:
  - 'has_spoken_languages': 480,046 rows (81.3%)
  - 'valid_spoken_language': 480,031 rows (81.3%)
  - 'is_multilingual_clean': 46,080 rows (7.8%)
  - 'has_no_dialogue': 0 rows (0.0%)
\n--- Standardized Language Distribution ---


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2193510167.py:415: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_spoken_languages'] = df['spoken_languages'].notnull()
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2193510167.py:416: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['valid_spoken_language'] = df['main_spoken_language_clean'] != "Unknown"
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2193510167.py:417: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.in

\nTop 20 standardized spoken languages:
  English                                 :  197,195 movies ( 33.4%)
  Unknown                                 :  110,171 movies ( 18.7%)
  French                                  :   39,482 movies (  6.7%)
  Spanish                                 :   39,420 movies (  6.7%)
  German                                  :   29,040 movies (  4.9%)
  Japanese                                :   25,466 movies (  4.3%)
  No Language                             :   18,702 movies (  3.2%)
  Russian                                 :   18,346 movies (  3.1%)
  Italian                                 :   18,108 movies (  3.1%)
  Portuguese                              :   17,033 movies (  2.9%)
  普通话                                     :   15,000 movies (  2.5%)
  Korean                                  :    9,044 movies (  1.5%)
  Arabic                                  :    7,860 movies (  1.3%)
  Český                                   :    7,185 movies (  

In [29]:
# Clean cast Column - SIMPLE VERSION
print("Cleaning cast column (simple version)...")
cast_clean_start = time.time()

# 1. Basic statistics
print(f"Current data type: {df['cast'].dtype}")
print(f"Total rows: {len(df):,}")
print(f"Non-null: {df['cast'].notnull().sum():,} ({df['cast'].notnull().sum()/len(df)*100:.1f}%)")
print(f"Null: {df['cast'].isnull().sum():,} ({df['cast'].isnull().sum()/len(df)*100:.1f}%)")

# 2. Mark missing data
df['missing_cast'] = df['cast'].isnull()
missing_count = df['missing_cast'].sum()
print(f"\\nCreated 'missing_cast' flag: {missing_count:,} missing values")

# 3. Fill nulls with empty string
df['cast'] = df['cast'].fillna('')
print(f"Filled nulls with empty strings")

# 4. Convert to string and clean basic formatting
df['cast'] = df['cast'].astype(str)
df['cast'] = df['cast'].str.strip()

# 5. Create simple derived columns
# Has cast data
df['has_cast'] = df['cast'] != ''
print(f"Movies with cast data: {df['has_cast'].sum():,} ({df['has_cast'].sum()/len(df)*100:.1f}%)")

# Count approximate number of actors (by counting commas)
df['cast_count'] = df['cast'].apply(
    lambda x: x.count(',') + 1 if x and x != '' else 0
)
print(f"\\nAverage cast size: {df['cast_count'].mean():.1f} actors")

# 6. Basic quality checks
# Check for movies with unusually large cast (potential data issues)
large_cast_threshold = 50
large_cast = df[df['cast_count'] > large_cast_threshold].shape[0]
print(f"Movies with >{large_cast_threshold} actors (potential data issues): {large_cast}")

# Check for very short cast strings (potential incomplete data)
short_cast = df[(df['has_cast']) & (df['cast'].str.len() < 5)].shape[0]
print(f"Movies with very short cast strings (<5 chars): {short_cast}")

# 7. Create cast data quality flag
df['good_cast_data'] = df['has_cast'] & (df['cast_count'] >= 1) & (df['cast_count'] <= 30)
print(f"\\nMovies with reasonable cast data (1-30 actors): {df['good_cast_data'].sum():,} ({df['good_cast_data'].sum()/len(df)*100:.1f}%)")

# 8. Sample after cleaning
print(f"\\n--- Sample After Cleaning ---")
print("First 10 rows:")
sample_df = df[['title', 'cast', 'cast_count', 'has_cast', 'good_cast_data']].head(10).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
sample_df['cast_truncated'] = sample_df['cast'].apply(lambda x: x[:50] + '...' if len(x) > 50 else x)

print(sample_df[['title_truncated', 'cast_truncated', 'cast_count', 'has_cast', 'good_cast_data']].to_string(index=False))

# 9. Summary
print(f"\\n--- Cast Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count:,} ({missing_count/len(df)*100:.1f}%)")
print(f"2. Created 'missing_cast' flag: {missing_count:,} True values")
print(f"3. Created 'has_cast' flag: {df['has_cast'].sum():,} True values")
print(f"4. Created 'cast_count' column (approximate)")
print(f"5. Created 'good_cast_data' flag: {df['good_cast_data'].sum():,} True values")
print(f"6. Current null count: {(df['cast'] == '').sum():,}")
print(f"7. Average cast size: {df['cast_count'].mean():.1f} actors")
print(f"8. Data type: {df['cast'].dtype}")

cast_clean_time = time.time() - cast_clean_start
print(f"\\nCast column cleaning completed in {cast_clean_time:.2f} seconds")

Cleaning cast column (simple version)...
Current data type: object
Total rows: 590,202
Non-null: 590,202 (100.0%)
Null: 0 (0.0%)
\nCreated 'missing_cast' flag: 0 missing values


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\1800444768.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['missing_cast'] = df['cast'].isnull()


Filled nulls with empty strings


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\1800444768.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_cast'] = df['cast'] != ''


Movies with cast data: 590,202 (100.0%)


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\1800444768.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['cast_count'] = df['cast'].apply(


\nAverage cast size: 10.5 actors
Movies with >50 actors (potential data issues): 8389
Movies with very short cast strings (<5 chars): 397
\nMovies with reasonable cast data (1-30 actors): 558,816 (94.7%)
\n--- Sample After Cleaning ---
First 10 rows:
    title_truncated                                        cast_truncated  cast_count  has_cast  good_cast_data
              Ariel Tarja Keinänen, Marja Packalén, Kari Helaseppä, Ma...          39      True           False
Shadows in Paradise Haije Alanoja, Mari Rantasila, Matti Pellonpää, Ka...          33      True           False
         Four Rooms Lili Taylor, Kimberly Blair, Quinn Hellerman, Paul...          28      True            True
     Judgment Night Doug Wert, Angela Alvarado, Everlast, Sean O'Grady...          31      True           False
   Sunday in August                            Milton Welsh, Rita Lengyel           2      True            True
          Star Wars Doug Beswick, David Prowse, Alex McCrindle, Denis ...    

C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\1800444768.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['good_cast_data'] = df['has_cast'] & (df['cast_count'] >= 1) & (df['cast_count'] <= 30)


In [30]:
# Clean director Column - SIMPLE VERSION
print("Cleaning director column (simple version)...")
director_clean_start = time.time()

# 1. Basic statistics
print(f"Current data type: {df['director'].dtype}")
print(f"Total rows: {len(df):,}")
print(f"Non-null: {df['director'].notnull().sum():,} ({df['director'].notnull().sum()/len(df)*100:.1f}%)")
print(f"Null: {df['director'].isnull().sum():,} ({df['director'].isnull().sum()/len(df)*100:.1f}%)")

# 2. Mark missing data
df['missing_director'] = df['director'].isnull()
missing_count = df['missing_director'].sum()
print(f"\\nCreated 'missing_director' flag: {missing_count:,} missing values")

# 3. Fill nulls with 'Unknown'
df['director'] = df['director'].fillna('Unknown')
print(f"Filled nulls with 'Unknown'")

# 4. Convert to string and clean basic formatting
df['director'] = df['director'].astype(str)
df['director'] = df['director'].str.strip()

# 5. Handle multiple directors (comma-separated)
df['has_multiple_directors'] = df['director'].str.contains(',')
multiple_dirs = df['has_multiple_directors'].sum()
print(f"Movies with multiple directors: {multiple_dirs:,} ({multiple_dirs/len(df)*100:.1f}%)")

# Count number of directors
df['director_count'] = df['director'].apply(
    lambda x: x.count(',') + 1 if x and x != 'Unknown' else 0
)
print(f"Average director count: {df['director_count'].mean():.1f}")

# Main director (first one if multiple)
df['main_director'] = df['director'].apply(
    lambda x: x.split(',')[0].strip() if x and x != 'Unknown' else 'Unknown'
)

# 6. Create quality flags
df['has_director'] = df['director'] != 'Unknown'
print(f"Movies with director data: {df['has_director'].sum():,} ({df['has_director'].sum()/len(df)*100:.1f}%)")

# Good director data flag (has director, reasonable name length)
df['good_director_data'] = df['has_director'] & (df['director'].str.len() >= 3)
good_data = df['good_director_data'].sum()
print(f"Movies with good director data: {good_data:,} ({good_data/len(df)*100:.1f}%)")

# 7. Sample after cleaning
print(f"\\n--- Sample After Cleaning ---")
sample_df = df[['title', 'director', 'director_count', 'has_multiple_directors', 'main_director']].head(8).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
sample_df['director_truncated'] = sample_df['director'].apply(lambda x: x[:40] + '...' if len(x) > 40 else x)

print(sample_df[['title_truncated', 'director_truncated', 'director_count', 'has_multiple_directors', 'main_director']].to_string(index=False))

# 8. Director statistics
print(f"\\n--- Director Statistics ---")
director_counts = df['main_director'].value_counts().head(10)
print(f"Top 10 directors by number of movies:")
for i, (director, count) in enumerate(director_counts.items(), 1):
    if director != 'Unknown':
        pct = count / len(df) * 100
        print(f"  {i:2d}. {director:40s}: {count:6,d} movies ({pct:.2f}%)")

# 9. Summary
print(f"\\n--- Director Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count:,} ({missing_count/len(df)*100:.1f}%)")
print(f"2. Created 'missing_director' flag: {missing_count:,} True values")
print(f"3. Created 'has_director' flag: {df['has_director'].sum():,} True values")
print(f"4. Created 'director_count' column")
print(f"5. Created 'main_director' column (first director)")
print(f"6. Created 'has_multiple_directors' flag: {multiple_dirs:,} True values")
print(f"7. Created 'good_director_data' flag: {good_data:,} True values")
print(f"8. Current null count: {(df['director'] == 'Unknown').sum():,}")
print(f"9. Average director count: {df['director_count'].mean():.1f}")
print(f"10. Data type: {df['director'].dtype}")

director_clean_time = time.time() - director_clean_start
print(f"\\nDirector column cleaning completed in {director_clean_time:.2f} seconds")

Cleaning director column (simple version)...
Current data type: object
Total rows: 590,202
Non-null: 590,202 (100.0%)
Null: 0 (0.0%)
\nCreated 'missing_director' flag: 0 missing values
Filled nulls with 'Unknown'


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2241730863.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['missing_director'] = df['director'].isnull()
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2241730863.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_multiple_directors'] = df['director'].str.contains(',')


Movies with multiple directors: 51,699 (8.8%)


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2241730863.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['director_count'] = df['director'].apply(


Average director count: 1.1


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2241730863.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['main_director'] = df['director'].apply(
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2241730863.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_director'] = df['director'] != 'Unknown'


Movies with director data: 590,200 (100.0%)


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\2241730863.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['good_director_data'] = df['has_director'] & (df['director'].str.len() >= 3)


Movies with good director data: 589,959 (100.0%)
\n--- Sample After Cleaning ---
    title_truncated                          director_truncated  director_count  has_multiple_directors      main_director
              Ariel                              Aki Kaurismäki               1                   False     Aki Kaurismäki
Shadows in Paradise                              Aki Kaurismäki               1                   False     Aki Kaurismäki
         Four Rooms Alexandre Rockwell, Allison Anders, Robe...               4                    True Alexandre Rockwell
     Judgment Night                             Stephen Hopkins               1                   False    Stephen Hopkins
   Sunday in August                                  Marc Meyer               1                   False         Marc Meyer
          Star Wars                                George Lucas               1                   False       George Lucas
       Finding Nemo                              Andrew St

In [31]:
# Clean writers Column - SIMPLE VERSION
print("\\nCleaning writers column (simple version)...")
writers_clean_start = time.time()

# 1. Basic statistics
print(f"Current data type: {df['writers'].dtype}")
print(f"Total rows: {len(df):,}")
print(f"Non-null: {df['writers'].notnull().sum():,} ({df['writers'].notnull().sum()/len(df)*100:.1f}%)")
print(f"Null: {df['writers'].isnull().sum():,} ({df['writers'].isnull().sum()/len(df)*100:.1f}%)")

# 2. Mark missing data
df['missing_writers'] = df['writers'].isnull()
missing_count = df['missing_writers'].sum()
print(f"\\nCreated 'missing_writers' flag: {missing_count:,} missing values")

# 3. Fill nulls with 'Unknown'
df['writers'] = df['writers'].fillna('Unknown')
print(f"Filled nulls with 'Unknown'")

# 4. Convert to string and clean basic formatting
df['writers'] = df['writers'].astype(str)
df['writers'] = df['writers'].str.strip()

# 5. Handle multiple writers (comma-separated)
df['has_multiple_writers'] = df['writers'].str.contains(',')
multiple_writers = df['has_multiple_writers'].sum()
print(f"Movies with multiple writers: {multiple_writers:,} ({multiple_writers/len(df)*100:.1f}%)")

# Count number of writers
df['writers_count'] = df['writers'].apply(
    lambda x: x.count(',') + 1 if x and x != 'Unknown' else 0
)
print(f"Average writer count: {df['writers_count'].mean():.1f}")

# Main writer (first one if multiple)
df['main_writer'] = df['writers'].apply(
    lambda x: x.split(',')[0].strip() if x and x != 'Unknown' else 'Unknown'
)

# 6. Create quality flags
df['has_writers'] = df['writers'] != 'Unknown'
print(f"Movies with writer data: {df['has_writers'].sum():,} ({df['has_writers'].sum()/len(df)*100:.1f}%)")

# Good writer data flag (has writer, reasonable name length)
df['good_writers_data'] = df['has_writers'] & (df['writers'].str.len() >= 3)
good_data = df['good_writers_data'].sum()
print(f"Movies with good writer data: {good_data:,} ({good_data/len(df)*100:.1f}%)")

# 7. Sample after cleaning
print(f"\\n--- Sample After Cleaning ---")
sample_df = df[['title', 'writers', 'writers_count', 'has_multiple_writers', 'main_writer']].head(8).copy()
sample_df['title_truncated'] = sample_df['title'].apply(lambda x: x[:20] + '...' if len(x) > 20 else x)
sample_df['writers_truncated'] = sample_df['writers'].apply(lambda x: x[:40] + '...' if len(x) > 40 else x)

print(sample_df[['title_truncated', 'writers_truncated', 'writers_count', 'has_multiple_writers', 'main_writer']].to_string(index=False))

# 8. Writer statistics
print(f"\\n--- Writer Statistics ---")
# Count unique writers (approximate)
all_writers = []
for writer_str in df[df['writers'] != 'Unknown']['writers']:
    if ',' in writer_str:
        all_writers.extend([w.strip() for w in writer_str.split(',')])
    else:
        all_writers.append(writer_str)

from collections import Counter
writer_counts = Counter(all_writers)
top_writers = writer_counts.most_common(10)

print(f"Top 10 writers by number of credits:")
for i, (writer, count) in enumerate(top_writers, 1):
    if writer != 'Unknown':
        pct = count / len(df) * 100
        print(f"  {i:2d}. {writer:40s}: {count:6,d} credits ({pct:.2f}%)")

# 9. Summary
print(f"\\n--- Writers Column Cleaning Summary ---")
print(f"1. Original null count: {missing_count:,} ({missing_count/len(df)*100:.1f}%)")
print(f"2. Created 'missing_writers' flag: {missing_count:,} True values")
print(f"3. Created 'has_writers' flag: {df['has_writers'].sum():,} True values")
print(f"4. Created 'writers_count' column")
print(f"5. Created 'main_writer' column (first writer)")
print(f"6. Created 'has_multiple_writers' flag: {multiple_writers:,} True values")
print(f"7. Created 'good_writers_data' flag: {good_data:,} True values")
print(f"8. Current null count: {(df['writers'] == 'Unknown').sum():,}")
print(f"9. Average writer count: {df['writers_count'].mean():.1f}")
print(f"10. Data type: {df['writers'].dtype}")

writers_clean_time = time.time() - writers_clean_start
print(f"\\nWriters column cleaning completed in {writers_clean_time:.2f} seconds")


\nCleaning writers column (simple version)...
Current data type: object
Total rows: 590,202
Non-null: 452,708 (76.7%)
Null: 137,494 (23.3%)
\nCreated 'missing_writers' flag: 137,494 missing values
Filled nulls with 'Unknown'


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\1345697890.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['missing_writers'] = df['writers'].isnull()
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\1345697890.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_multiple_writers'] = df['writers'].str.contains(',')


Movies with multiple writers: 198,576 (33.6%)


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\1345697890.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['writers_count'] = df['writers'].apply(


Average writer count: 1.3


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\1345697890.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['main_writer'] = df['writers'].apply(
C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\1345697890.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_writers'] = df['writers'] != 'Unknown'


Movies with writer data: 452,707 (76.7%)


C:\Users\abhir\AppData\Local\Temp\ipykernel_3212\1345697890.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['good_writers_data'] = df['has_writers'] & (df['writers'].str.len() >= 3)


Movies with good writer data: 452,596 (76.7%)
\n--- Sample After Cleaning ---
    title_truncated                           writers_truncated  writers_count  has_multiple_writers    main_writer
              Ariel                              Aki Kaurismäki              1                 False Aki Kaurismäki
Shadows in Paradise                              Aki Kaurismäki              1                 False Aki Kaurismäki
         Four Rooms Allison Anders, Alexandre Rockwell, Quen...              4                  True Allison Anders
     Judgment Night               Lewis Colick, Jere Cunningham              2                  True   Lewis Colick
   Sunday in August                                  Marc Meyer              1                 False     Marc Meyer
          Star Wars                                George Lucas              1                 False   George Lucas
       Finding Nemo Bob Peterson, Ronnie del Carmen, Blake T...              8                  True   Bob Pet

In [32]:
# Drop poster_path column
print("Dropping poster_path column...")

# Check if column exists
if 'poster_path' in df.columns:
    # Drop the column
    df = df.drop(columns=['poster_path'])
    print(f"✅ Dropped poster_path column")
    print(f"New shape: {df.shape[0]:,} rows, {df.shape[1]:,} columns")
else:
    print(f"❌ poster_path column not found in DataFrame")

# Also drop from saved file if needed
print(f"\\nDon't forget to save the DataFrame again without poster_path")

Dropping poster_path column...
✅ Dropped poster_path column
New shape: 590,202 rows, 166 columns
\nDon't forget to save the DataFrame again without poster_path


In [33]:
# Save cleaned DataFrame to CSV file
print("Saving cleaned DataFrame to CSV file...")

# Define the output file name
output_file = "movies_cleaned2.csv"

# Save the entire DataFrame to CSV
df.to_csv(output_file, index=False)

# Check if file was created
if os.path.exists(output_file):
    file_size_mb = os.path.getsize(output_file) / (1024**2)
    print(f" Successfully saved to: {output_file}")
    print(f"File size: {file_size_mb:.2f} MB")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    
    # Show basic info
    print(f"\\nFirst 3 rows:")
    display(df.head(3))
else:
    print(f" Error: Could not save to {output_file}")

print(f"\\nDone!")

Saving cleaned DataFrame to CSV file...
 Successfully saved to: movies_cleaned2.csv
File size: 1033.20 MB
Shape: 590,202 rows × 166 columns
\nFirst 3 rows:


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,original_language,original_title,overview,popularity,genres,production_companies,production_countries,spoken_languages,cast,director,writers,missing_title,missing_vote_average,low_confidence_rating,vote_range,missing_vote_count,low_vote_count,vote_count_range,missing_status,missing_release_date,release_year,release_month,release_day,release_weekday,invalid_release_date,future_release_date,release_decade,missing_revenue,revenue_category,zero_revenue,low_revenue,high_revenue_outlier,missing_budget,budget_category,zero_budget,low_budget,high_budget_outlier,missing_runtime,runtime_category,zero_runtime,short_film,feature_film,extended_film,unusual_runtime,missing_imdb_id,imdb_id_original,imdb_id_clean,valid_imdb_id,has_imdb_id,imdb_id_numeric,missing_original_language,language_category,valid_language_code,rare_language,decade,original_language_name,missing_original_title,original_title_clean,filled_from_title,is_placeholder_title,has_original_title,original_title_length,original_title_length_category,title_has_non_latin,missing_overview,overview_length,overview_length_category,overview_clean,has_overview,short_overview,detailed_overview,template_overview,overview_word_count,missing_popularity,popularity_category,zero_popularity,low_popularity,high_popularity_outlier,missing_genres,genre_format,genres_list,genre_count,genres_clean,genre_drama,genre_comedy,genre_documentary,genre_romance,genre_horror,genre_thriller,genre_action,genre_crime,genre_music,genre_tv_movie,genre_animation,genre_family,genre_adventure,genre_fantasy,genre_mystery,has_genres,multiple_genres,missing_production_companies,production_companies_format,production_companies_list,production_company_count,production_companies_clean,production_company_co,production_company_warner_bros,production_company_universal_pictures,production_company_ltd,production_company_bbc,production_company_inc,production_company_columbia_pictures,production_company_paramount_pictures,production_company_metro-goldwyn-mayer,has_production_companies,multiple_production_companies,major_studio,missing_production_countries,production_countries_parsed,num_production_countries,main_production_country,is_co_production,production_countries_standardized,num_production_countries_clean,main_production_country_clean,has_production_country,valid_production_country,multiple_production_countries,missing_spoken_languages,spoken_languages_parsed,num_spoken_languages,main_spoken_language,is_multilingual,spoken_languages_standardized,num_spoken_languages_clean,main_spoken_language_clean,has_spoken_languages,valid_spoken_language,is_multilingual_clean,has_no_dialogue,missing_cast,has_cast,cast_count,good_cast_data,missing_director,has_multiple_directors,director_count,main_director,has_director,good_director_data,missing_writers,has_multiple_writers,writers_count,main_writer,has_writers,good_writers_data
0,2,Ariel,7.10,367,Released,1988-10-21,0,73,0,tt0094675,fi,Ariel,A Finnish man goes to the city to find a job a...,4.14,"Comedy, Drama, Romance, Crime",Villealfa Filmproductions,Finland,Finnish,"Tarja Keinänen, Marja Packalén, Kari Helaseppä...",Aki Kaurismäki,Aki Kaurismäki,False,False,False,7-8,False,False,101-1K,False,False,1988,10,21,Friday,False,False,1980,False,$0,True,False,False,False,$0,True,False,False,False,61-90 min,False,False,True,False,False,False,tt0094675,tt0094675,True,True,94675.00,False,other,True,False,1980,Finnish,False,Ariel,False,False,True,5,≤10,False,False,117,101-200,A Finnish man goes to the city to find a job a...,True,False,False,False,24,False,Very High,False,False,True,False,comma_separated,"[Comedy, Drama, Romance, Crime]",4,"[Comedy, Drama, Romance, Crime]",True,True,False,True,False,False,False,True,False,False,False,False,False,False,False,True,True,False,single_company,[Villealfa Filmproductions],1,[Villealfa Filmproductions],False,False,False,False,False,False,False,False,False,True,False,F

\nDone!


In [34]:
# Quick check if cleaned dataset is ready for analysis
print("Quick dataset quality check...")

# 1. Check basic shape
print(f"Dataset shape: {df.shape[0]:,} rows, {df.shape[1]:,} columns")

# 2. Check for remaining null values
print(f"\\n--- Null Value Check ---")
null_counts = df.isnull().sum()
columns_with_nulls = null_counts[null_counts > 0]

if len(columns_with_nulls) == 0:
    print(" No null values found")
else:
    print(f"  Found {len(columns_with_nulls)} columns with nulls:")
    for col, count in columns_with_nulls.head(10).items():  # Show top 10
        pct = count / len(df) * 100
        print(f"  {col}: {count:,} nulls ({pct:.1f}%)")

# 3. Check critical columns
print(f"\\n--- Critical Columns Check ---")
critical_columns = ['title', 'release_date', 'genres', 'vote_average', 'revenue', 'budget']
for col in critical_columns:
    if col in df.columns:
        has_data = df[col].notnull().sum()
        pct = has_data / len(df) * 100
        print(f"  {col:20s}: {has_data:,} rows with data ({pct:.1f}%)")
    else:
        print(f"  {col:20s}:  Column not found")

# 4. Check data types
print(f"\\n--- Data Types Check ---")
dtype_summary = df.dtypes.value_counts()
for dtype, count in dtype_summary.items():
    print(f"  {dtype}: {count} columns")

# 5. Check a few samples
print(f"\\n--- Sample Data Check ---")
sample_cols = ['title', 'release_year', 'vote_average', 'revenue', 'genres', 'main_director']
sample_df = df[sample_cols].head(5).copy()
print(sample_df.to_string(index=False))

# 6. Final assessment
print(f"\\n--- FINAL ASSESSMENT ---")
if len(columns_with_nulls) < 5 and all(df[col].notnull().sum() > 0 for col in critical_columns if col in df.columns):
    print(" Dataset looks ready for analysis!")
else:
    print("  Dataset may need more cleaning before analysis")

print(f"\\nDataset saved to: movies_cleaned.csv")
print(f"Ready for analysis!")

Quick dataset quality check...
Dataset shape: 590,202 rows, 166 columns
\n--- Null Value Check ---
  Found 4 columns with nulls:
  production_companies: 183,592 nulls (31.1%)
  imdb_id_original: 139,754 nulls (23.7%)
  imdb_id_numeric: 139,754 nulls (23.7%)
  overview_length_category: 62,635 nulls (10.6%)
\n--- Critical Columns Check ---
  title               : 590,202 rows with data (100.0%)
  release_date        : 590,202 rows with data (100.0%)
  genres              : 590,202 rows with data (100.0%)
  vote_average        : 590,202 rows with data (100.0%)
  revenue             : 590,202 rows with data (100.0%)
  budget              : 590,202 rows with data (100.0%)
\n--- Data Types Check ---
  bool: 96 columns
  object: 36 columns
  int64: 15 columns
  int32: 7 columns
  float64: 3 columns
  datetime64[ns]: 1 columns
  category: 1 columns
  category: 1 columns
  category: 1 columns
  category: 1 columns
  category: 1 columns
  category: 1 columns
  category: 1 columns
  category: 1 c

## Step 4: Quality & Consistency Validation
- Check for duplicate 'id' values
- Validate value ranges (e.g., vote_average between 0–10)
- Cross-check revenue vs. budget for logical consistency

## Step 5: Post-Cleaning Validation
- Show df.info() after cleaning
- Display random sample of 10–20 rows to verify dtypes and cleaning
- Log final dataset shape and memory usage

##  Performance Log
- Record time taken for key steps (loading, cleaning, saving)
- Log hardware specs and Python environment
- Summarize cleaning decisions in a markdown table

## Summary
- Bullet-point summary of what was cleaned
- Note excluded columns and reasons
- Outline next steps for analysis or MapReduce implementation